<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/12_SPP_GAN_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
# ==================================================================================================
# NOTEBOOK 12 — SPP-GAN TRAINING
# ==================================================================================================
#
# SPP-GAN:
# A Statistical-Guided Privacy-Preserving GAN Framework
# for High-Fidelity Synthetic Tabular Data Generation
#
# PURPOSE
# -------
# Train the final SPP-GAN model using:
#
#   • Notebook 02 train-only transformed data
#   • Notebook 03 train-only statistical characterization
#   • Notebook 08 frozen SPP-GAN architecture
#   • Notebook 09 differentiable statistical guidance
#   • Notebook 10 DP-SGD mechanism
#   • Notebook 11 configured RDP privacy accounting
#
# PRIVACY
# -------
#   • Protected component : discriminator / critic
#   • DP mechanism         : DP-SGD
#   • Sampling             : Poisson
#   • Clipping             : flat L2
#   • Noise                : Gaussian
#   • Accountant            : RDP
#
# IMPORTANT PRIVACY BOUNDARY
# --------------------------
#   generator_private          = False
#   statistical_guidance_private= False
#   preprocessing_private      = False
#   end_to_end_privacy_claim   = False
#
# Notebook 12 records the actual training accountant epsilon.
# Formal end-to-end privacy is NOT claimed because the statistical
# reference and preprocessing remain outside the DP mechanism.
#
# DATA POLICY
# ----------
#   TRAIN       : used
#   VALIDATION  : monitoring only
#   TEST        : NEVER used for training or model selection
#
# ==================================================================================================

print("=" * 100)
print("NOTEBOOK 12 — SPP-GAN TRAINING")
print("=" * 100)
print("Framework              : SPP-GAN")
print("Training mechanism     : DP-SGD")
print("Protected component    : SPP-GAN discriminator")
print("Statistical guidance   : Notebook 09")
print("Privacy accounting     : Notebook 11")
print("Training data          : Notebook 02 TRAIN only")
print("Test data              : NOT USED")
print("=" * 100)

NOTEBOOK 12 — SPP-GAN TRAINING
Framework              : SPP-GAN
Training mechanism     : DP-SGD
Protected component    : SPP-GAN discriminator
Statistical guidance   : Notebook 09
Privacy accounting     : Notebook 11
Training data          : Notebook 02 TRAIN only
Test data              : NOT USED


In [50]:
# ==================================================================================================
# SECTION 2 — LOAD CONFIGURATION
# ==================================================================================================

print("=" * 100)
print("2. LOAD CONFIGURATION")
print("=" * 100)

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Verify Google Drive and Project Root
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"
PROJECT_ROOT = MYDRIVE_ROOT / "SPP_GAN_Research"

if not MYDRIVE_ROOT.exists():
    raise FileNotFoundError(
        f"Google Drive MyDrive not available: {MYDRIVE_ROOT}"
    )

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"SPP-GAN project root not found: {PROJECT_ROOT}"
    )

print(f"✓ MyDrive        : {MYDRIVE_ROOT}")
print(f"✓ Project root   : {PROJECT_ROOT}")


# --------------------------------------------------------------------------------------------------
# 2. Canonical Notebook Paths
# --------------------------------------------------------------------------------------------------

NB08_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_08"
NB09_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_09"
NB10_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_10"
NB11_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_11"
NB12_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_12"

NB09_CONFIG_PATH = (
    NB09_ROOT
    / "configuration"
    / "sppgan_statistical_guidance_configuration.json"
)

NB10_PRIVACY_CONFIG_PATH = (
    NB10_ROOT
    / "configuration"
    / "sppgan_privacy_configuration.json"
)

NB10_PRIVACY_METADATA_PATH = (
    NB10_ROOT
    / "metadata"
    / "sppgan_privacy_metadata.csv"
)

NB11_ACCOUNTING_CONFIG_PATH = (
    NB11_ROOT
    / "configuration"
    / "sppgan_privacy_accounting_configuration.json"
)

NB11_CONFIGURED_ACCOUNTING_PATH = (
    NB11_ROOT
    / "accounting"
    / "sppgan_configured_schedule_rdp_accounting.csv"
)


# --------------------------------------------------------------------------------------------------
# 3. Required Artifact Verification
# --------------------------------------------------------------------------------------------------

REQUIRED_CONFIGURATION_ARTIFACTS = {
    "Notebook 09 statistical guidance configuration":
        NB09_CONFIG_PATH,

    "Notebook 10 privacy configuration":
        NB10_PRIVACY_CONFIG_PATH,

    "Notebook 10 privacy metadata":
        NB10_PRIVACY_METADATA_PATH,

    "Notebook 11 privacy accounting configuration":
        NB11_ACCOUNTING_CONFIG_PATH,

    "Notebook 11 configured RDP accounting":
        NB11_CONFIGURED_ACCOUNTING_PATH,
}

print()
print("Required persisted configuration artifacts:")

for artifact_name, artifact_path in REQUIRED_CONFIGURATION_ARTIFACTS.items():

    if not artifact_path.exists():
        raise FileNotFoundError(
            f"Required persisted artifact not found:\n"
            f"{artifact_name}: {artifact_path}"
        )

    print(
        f"  ✓ {artifact_name:<45}: "
        f"{artifact_path}"
    )


# --------------------------------------------------------------------------------------------------
# 4. Utility Functions
# --------------------------------------------------------------------------------------------------

def load_json_file(
    path,
):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        payload = json.load(
            handle
        )

    if not isinstance(
        payload,
        dict,
    ):

        raise TypeError(
            f"Expected JSON object at {path}, "
            f"found {type(payload).__name__}."
        )

    return payload


def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def normalize_key(
    key,
):

    return (
        str(key)
        .strip()
        .lower()
        .replace(
            "-",
            "_",
        )
        .replace(
            " ",
            "_",
        )
    )


def find_key_occurrences(
    obj,
    candidate_keys,
    path="",
):

    occurrences = []

    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            current_path = (
                f"{path}.{key}"
                if path
                else str(key)
            )

            if normalize_key(
                key
            ) in candidate_keys:

                occurrences.append(
                    {
                        "path":
                            current_path,

                        "value":
                            value,
                    }
                )

            occurrences.extend(
                find_key_occurrences(
                    value,
                    candidate_keys,
                    current_path,
                )
            )

    elif isinstance(
        obj,
        list,
    ):

        for index, value in enumerate(
            obj
        ):

            current_path = (
                f"{path}[{index}]"
            )

            occurrences.extend(
                find_key_occurrences(
                    value,
                    candidate_keys,
                    current_path,
                )
            )

    return occurrences


def canonicalize_configuration_value(
    parameter_name,
    value,
):

    if not isinstance(
        value,
        str,
    ):

        return value

    normalized = (
        value
        .strip()
        .lower()
    )

    # ----------------------------------------------------------------------------------------------
    # Accountant
    # ----------------------------------------------------------------------------------------------

    if parameter_name == "accountant":

        aliases = {

            "rdp":
                "rdp",

            "renyi_differential_privacy":
                "rdp",

            "rényi_differential_privacy":
                "rdp",
        }

        return aliases.get(
            normalized,
            normalized,
        )

    # ----------------------------------------------------------------------------------------------
    # Sampling mechanism
    # ----------------------------------------------------------------------------------------------

    if parameter_name == "sampling mechanism":

        aliases = {

            "poisson":
                "poisson",

            "poisson_sampling":
                "poisson",

            "poisson_sampling_mechanism":
                "poisson",
        }

        return aliases.get(
            normalized,
            normalized,
        )

    # ----------------------------------------------------------------------------------------------
    # Clipping mechanism
    # ----------------------------------------------------------------------------------------------

    if parameter_name == "clipping mechanism":

        aliases = {

            "flat":
                "flat",

            "flat_l2":
                "flat",

            "flat l2":
                "flat",

            "l2":
                "flat",

            "l2_flat":
                "flat",

            "flat_l2_clipping":
                "flat",
        }

        return aliases.get(
            normalized,
            normalized,
        )

    # ----------------------------------------------------------------------------------------------
    # Loss reduction
    # ----------------------------------------------------------------------------------------------

    if parameter_name == "loss reduction":

        aliases = {

            "mean":
                "mean",

            "average":
                "mean",
        }

        return aliases.get(
            normalized,
            normalized,
        )

    return normalized


def resolve_global_parameter(
    config,
    candidate_keys,
    parameter_name,
    value_type="numeric",
):

    normalized_candidates = {
        normalize_key(
            key
        )
        for key in candidate_keys
    }

    occurrences = find_key_occurrences(
        config,
        normalized_candidates,
    )

    if not occurrences:

        raise KeyError(
            f"Notebook 10 configuration does not expose "
            f"required parameter '{parameter_name}'."
        )

    values = []

    for occurrence in occurrences:

        value = occurrence[
            "value"
        ]

        # ------------------------------------------------------------------------------------------
        # Numeric
        # ------------------------------------------------------------------------------------------

        if value_type == "numeric":

            if isinstance(
                value,
                bool,
            ):
                continue

            if isinstance(
                value,
                (
                    int,
                    float,
                    np.integer,
                    np.floating,
                ),
            ):

                numeric_value = float(
                    value
                )

                if np.isfinite(
                    numeric_value
                ):

                    values.append(
                        (
                            occurrence[
                                "path"
                            ],
                            numeric_value,
                        )
                    )

        # ------------------------------------------------------------------------------------------
        # Boolean
        # ------------------------------------------------------------------------------------------

        elif value_type == "boolean":

            if isinstance(
                value,
                bool,
            ):

                values.append(
                    (
                        occurrence[
                            "path"
                        ],
                        value,
                    )
                )

        # ------------------------------------------------------------------------------------------
        # String / enumeration
        # ------------------------------------------------------------------------------------------

        elif value_type == "string":

            if isinstance(
                value,
                str,
            ):

                canonical_value = (
                    canonicalize_configuration_value(
                        parameter_name,
                        value,
                    )
                )

                if canonical_value:

                    values.append(
                        (
                            occurrence[
                                "path"
                            ],
                            canonical_value,
                        )
                    )

        else:

            raise ValueError(
                f"Unsupported value_type={value_type!r} "
                f"for parameter '{parameter_name}'."
            )

    if not values:

        raise KeyError(
            f"No valid persisted value found for "
            f"'{parameter_name}'."
        )

    unique_values = []

    for _, value in values:

        if value not in unique_values:

            unique_values.append(
                value
            )

    if len(
        unique_values
    ) != 1:

        raise RuntimeError(
            f"Conflicting Notebook 10 configuration values "
            f"for '{parameter_name}':\n"
            +
            "\n".join(
                f"  {path} = {value}"
                for path, value in values
            )
        )

    resolved_value = unique_values[
        0
    ]

    print(
        f"✓ {parameter_name:<30}: "
        f"{resolved_value}"
    )

    if len(
        values
    ) > 1:

        print(
            f"  Validated occurrences             : "
            f"{len(values)}"
        )

    return resolved_value


# --------------------------------------------------------------------------------------------------
# 5. Load Persisted Configurations
# --------------------------------------------------------------------------------------------------

NB09_STATISTICAL_GUIDANCE_CONFIG = load_json_file(
    NB09_CONFIG_PATH
)

NB10_PRIVACY_CONFIG = load_json_file(
    NB10_PRIVACY_CONFIG_PATH
)

NB11_ACCOUNTING_CONFIG = load_json_file(
    NB11_ACCOUNTING_CONFIG_PATH
)

NB10_PRIVACY_METADATA_DF = pd.read_csv(
    NB10_PRIVACY_METADATA_PATH
)

NB11_CONFIGURED_ACCOUNTING_DF = pd.read_csv(
    NB11_CONFIGURED_ACCOUNTING_PATH
)


# --------------------------------------------------------------------------------------------------
# 6. Validate Notebook 09 Statistical Guidance Configuration
# --------------------------------------------------------------------------------------------------

print()
print("Notebook 09 statistical guidance:")

NB09_LAMBDA_STAT = None

NB09_LAMBDA_OCCURRENCES = find_key_occurrences(
    NB09_STATISTICAL_GUIDANCE_CONFIG,
    {
        "lambda_stat",
        "λ_stat",
    },
)

for occurrence in NB09_LAMBDA_OCCURRENCES:

    value = occurrence[
        "value"
    ]

    if isinstance(
        value,
        (
            int,
            float,
            np.integer,
            np.floating,
        ),
    ):

        value = float(
            value
        )

        if np.isfinite(
            value
        ):

            if NB09_LAMBDA_STAT is None:

                NB09_LAMBDA_STAT = value

            elif not np.isclose(
                NB09_LAMBDA_STAT,
                value,
                rtol=0.0,
                atol=1e-12,
            ):

                raise RuntimeError(
                    "Conflicting Notebook 09 λ_stat values."
                )

if NB09_LAMBDA_STAT is None:

    raise KeyError(
        "λ_stat was not found in persisted Notebook 09 "
        "statistical guidance configuration."
    )

if NB09_LAMBDA_STAT < 0:

    raise ValueError(
        f"Invalid λ_stat={NB09_LAMBDA_STAT}."
    )

print(
    f"  λ_stat : {NB09_LAMBDA_STAT:.6f}"
)

print(
    "  Source : persisted Notebook 09 configuration"
)


# --------------------------------------------------------------------------------------------------
# 7. Validate Notebook 10 Privacy Metadata Schema
# --------------------------------------------------------------------------------------------------

REQUIRED_NB10_METADATA_COLUMNS = [

    "dataset",
    "n_train",
    "target_epsilon",
    "delta",
    "batch_size",
    "sample_rate",
    "epochs",
    "nominal_steps_per_epoch",
    "nominal_total_steps",
    "max_grad_norm",
    "noise_multiplier",
    "accountant",
    "sampling",
    "clipping",
    "loss_reduction",
    "protected_component",
    "per_example_gradients",
    "generator_private",
    "statistical_guidance_private",
    "preprocessing_private",
    "achieved_epsilon",
    "achieved_epsilon_status",
    "training_status",
    "synthetic_generation_status",
    "end_to_end_privacy_claim",
    "status",
]

missing_nb10_columns = [
    column
    for column in REQUIRED_NB10_METADATA_COLUMNS
    if column not in NB10_PRIVACY_METADATA_DF.columns
]

if missing_nb10_columns:

    raise RuntimeError(
        "Notebook 10 privacy metadata is missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_nb10_columns
        )
    )


if len(
    NB10_PRIVACY_METADATA_DF
) != 3:

    raise RuntimeError(
        "Notebook 10 privacy metadata must contain exactly "
        f"3 canonical dataset rows. "
        f"Observed: {len(NB10_PRIVACY_METADATA_DF)}"
    )


DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

if set(
    NB10_PRIVACY_METADATA_DF[
        "dataset"
    ].astype(
        str
    )
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Notebook 10 metadata dataset registry does not match "
        "the canonical SPP-GAN dataset registry."
    )

print()
print("Notebook 10 privacy metadata:")
print(
    f"  Rows    : {len(NB10_PRIVACY_METADATA_DF)}"
)
print(
    f"  Columns : {list(NB10_PRIVACY_METADATA_DF.columns)}"
)


# --------------------------------------------------------------------------------------------------
# 8. Expected Training Dataset Sizes
# --------------------------------------------------------------------------------------------------

EXPECTED_TRAIN_ROWS = {

    "adult_income":
        34189,

    "bank_marketing":
        31647,

    "diabetes_130us":
        71236,
}


for dataset_id in DATASET_IDS:

    if dataset_id not in EXPECTED_TRAIN_ROWS:

        raise RuntimeError(
            f"Missing expected training-row definition "
            f"for {dataset_id}."
        )


# --------------------------------------------------------------------------------------------------
# 9. Resolve Notebook 10 Global Privacy Parameters
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("NOTEBOOK 10 PRIVACY PARAMETERS")
print("-" * 100)


DP_BATCH_SIZE = int(
    resolve_global_parameter(
        NB10_PRIVACY_CONFIG,
        [
            "dp_batch_size",
            "batch_size",
        ],
        "DP batch size",
        value_type="numeric",
    )
)


TARGET_EPSILON = float(
    resolve_global_parameter(
        NB10_PRIVACY_CONFIG,
        [
            "target_epsilon",
        ],
        "target epsilon",
        value_type="numeric",
    )
)


MAX_GRAD_NORM = float(
    resolve_global_parameter(
        NB10_PRIVACY_CONFIG,
        [
            "max_grad_norm",
            "dp_max_grad_norm",
        ],
        "maximum gradient norm",
        value_type="numeric",
    )
)


DP_EPOCHS = int(
    resolve_global_parameter(
        NB10_PRIVACY_CONFIG,
        [
            "dp_epochs",
            "epochs",
            "training_epochs",
        ],
        "training epochs",
        value_type="numeric",
    )
)


ACCOUNTANT = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "accountant",
    ],
    "accountant",
    value_type="string",
)


SAMPLING_MECHANISM = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "sampling_mechanism",
        "sampling",
    ],
    "sampling mechanism",
    value_type="string",
)


CLIPPING_MECHANISM = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "clipping_mechanism",
        "clipping",
    ],
    "clipping mechanism",
    value_type="string",
)


LOSS_REDUCTION = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "loss_reduction",
    ],
    "loss reduction",
    value_type="string",
)


# --------------------------------------------------------------------------------------------------
# 10. Resolve Privacy Boundary
# --------------------------------------------------------------------------------------------------

GENERATOR_PRIVATE = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "generator_private",
    ],
    "generator_private",
    value_type="boolean",
)


STATISTICAL_GUIDANCE_PRIVATE = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "statistical_guidance_private",
    ],
    "statistical_guidance_private",
    value_type="boolean",
)


PREPROCESSING_PRIVATE = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "preprocessing_private",
    ],
    "preprocessing_private",
    value_type="boolean",
)


END_TO_END_PRIVACY_CLAIM = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "end_to_end_privacy_claim",
    ],
    "end_to_end_privacy_claim",
    value_type="boolean",
)


# --------------------------------------------------------------------------------------------------
# 11. Validate Global Privacy Configuration
# --------------------------------------------------------------------------------------------------

if DP_BATCH_SIZE <= 0:

    raise ValueError(
        "DP batch size must be positive."
    )

if DP_EPOCHS <= 0:

    raise ValueError(
        "DP epochs must be positive."
    )

if TARGET_EPSILON <= 0:

    raise ValueError(
        "Target epsilon must be positive."
    )

if MAX_GRAD_NORM <= 0:

    raise ValueError(
        "Maximum gradient norm must be positive."
    )

if ACCOUNTANT != "rdp":

    raise RuntimeError(
        f"Unexpected accountant: {ACCOUNTANT}. "
        "Expected canonical value: rdp."
    )

if SAMPLING_MECHANISM != "poisson":

    raise RuntimeError(
        f"Unexpected sampling mechanism: "
        f"{SAMPLING_MECHANISM}. "
        "Expected canonical value: poisson."
    )

if CLIPPING_MECHANISM != "flat":

    raise RuntimeError(
        f"Unexpected clipping mechanism: "
        f"{CLIPPING_MECHANISM}. "
        "Expected canonical value: flat."
    )

if LOSS_REDUCTION != "mean":

    raise RuntimeError(
        f"Unexpected loss reduction: "
        f"{LOSS_REDUCTION}. "
        "Expected canonical value: mean."
    )


# --------------------------------------------------------------------------------------------------
# 12. Validate Privacy Boundary
# --------------------------------------------------------------------------------------------------

if GENERATOR_PRIVATE is not False:

    raise RuntimeError(
        "Notebook 10 declares generator_private=False. "
        "Notebook 12 cannot silently change this boundary."
    )

if STATISTICAL_GUIDANCE_PRIVATE is not False:

    raise RuntimeError(
        "Notebook 10 declares statistical_guidance_private=False. "
        "Notebook 12 cannot silently change this boundary."
    )

if PREPROCESSING_PRIVATE is not False:

    raise RuntimeError(
        "Notebook 10 declares preprocessing_private=False. "
        "Notebook 12 cannot silently change this boundary."
    )

if END_TO_END_PRIVACY_CLAIM is not False:

    raise RuntimeError(
        "Notebook 10 declares end_to_end_privacy_claim=False. "
        "Notebook 12 cannot claim end-to-end DP."
    )


# --------------------------------------------------------------------------------------------------
# 13. Validate Dataset-Level Privacy Metadata
# --------------------------------------------------------------------------------------------------

PRIVACY_PARAMETERS = {}

for dataset_id in DATASET_IDS:

    matching_rows = NB10_PRIVACY_METADATA_DF[
        NB10_PRIVACY_METADATA_DF[
            "dataset"
        ].astype(
            str
        )
        ==
        dataset_id
    ]

    if len(
        matching_rows
    ) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one Notebook 10 "
            f"privacy metadata row; observed {len(matching_rows)}."
        )

    row = matching_rows.iloc[
        0
    ]

    n_train = int(
        row[
            "n_train"
        ]
    )

    target_epsilon = float(
        row[
            "target_epsilon"
        ]
    )

    delta = float(
        row[
            "delta"
        ]
    )

    batch_size = int(
        row[
            "batch_size"
        ]
    )

    sample_rate = float(
        row[
            "sample_rate"
        ]
    )

    epochs = int(
        row[
            "epochs"
        ]
    )

    max_grad_norm = float(
        row[
            "max_grad_norm"
        ]
    )

    noise_multiplier = float(
        row[
            "noise_multiplier"
        ]
    )

    metadata_accountant = (
        canonicalize_configuration_value(
            "accountant",
            str(
                row[
                    "accountant"
                ]
            ),
        )
    )

    metadata_sampling = (
        canonicalize_configuration_value(
            "sampling mechanism",
            str(
                row[
                    "sampling"
                ]
            ),
        )
    )

    metadata_clipping = (
        canonicalize_configuration_value(
            "clipping mechanism",
            str(
                row[
                    "clipping"
                ]
            ),
        )
    )

    metadata_loss_reduction = (
        canonicalize_configuration_value(
            "loss reduction",
            str(
                row[
                    "loss_reduction"
                ]
            ),
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Dataset identity
    # ----------------------------------------------------------------------------------------------

    if n_train != EXPECTED_TRAIN_ROWS[
        dataset_id
    ]:

        raise RuntimeError(
            f"{dataset_id}: training-row mismatch.\n"
            f"Expected : {EXPECTED_TRAIN_ROWS[dataset_id]}\n"
            f"Observed : {n_train}"
        )

    # ----------------------------------------------------------------------------------------------
    # Batch size
    # ----------------------------------------------------------------------------------------------

    if batch_size != DP_BATCH_SIZE:

        raise RuntimeError(
            f"{dataset_id}: batch-size mismatch.\n"
            f"Configuration : {DP_BATCH_SIZE}\n"
            f"Metadata      : {batch_size}"
        )

    # ----------------------------------------------------------------------------------------------
    # Poisson sampling rate
    # ----------------------------------------------------------------------------------------------

    expected_sample_rate = (
        float(
            DP_BATCH_SIZE
        )
        /
        float(
            n_train
        )
    )

    if not np.isclose(
        sample_rate,
        expected_sample_rate,
        rtol=1e-6,
        atol=1e-12,
    ):

        raise RuntimeError(
            f"{dataset_id}: sample-rate mismatch.\n"
            f"Persisted q : {sample_rate}\n"
            f"Expected q  : {expected_sample_rate}"
        )

    # ----------------------------------------------------------------------------------------------
    # Epochs
    # ----------------------------------------------------------------------------------------------

    if epochs != DP_EPOCHS:

        raise RuntimeError(
            f"{dataset_id}: epoch mismatch.\n"
            f"Configuration : {DP_EPOCHS}\n"
            f"Metadata      : {epochs}"
        )

    # ----------------------------------------------------------------------------------------------
    # Target epsilon
    # ----------------------------------------------------------------------------------------------

    if not np.isclose(
        target_epsilon,
        TARGET_EPSILON,
        rtol=0.0,
        atol=1e-12,
    ):

        raise RuntimeError(
            f"{dataset_id}: target epsilon mismatch.\n"
            f"Configuration : {TARGET_EPSILON}\n"
            f"Metadata      : {target_epsilon}"
        )

    # ----------------------------------------------------------------------------------------------
    # Maximum gradient norm
    # ----------------------------------------------------------------------------------------------

    if not np.isclose(
        max_grad_norm,
        MAX_GRAD_NORM,
        rtol=0.0,
        atol=1e-12,
    ):

        raise RuntimeError(
            f"{dataset_id}: maximum gradient norm mismatch.\n"
            f"Configuration : {MAX_GRAD_NORM}\n"
            f"Metadata      : {max_grad_norm}"
        )

    # ----------------------------------------------------------------------------------------------
    # Accountant
    # ----------------------------------------------------------------------------------------------

    if metadata_accountant != ACCOUNTANT:

        raise RuntimeError(
            f"{dataset_id}: accountant mismatch.\n"
            f"Configuration : {ACCOUNTANT}\n"
            f"Metadata      : {metadata_accountant}"
        )

    # ----------------------------------------------------------------------------------------------
    # Sampling
    # ----------------------------------------------------------------------------------------------

    if metadata_sampling != SAMPLING_MECHANISM:

        raise RuntimeError(
            f"{dataset_id}: sampling mechanism mismatch.\n"
            f"Configuration : {SAMPLING_MECHANISM}\n"
            f"Metadata      : {metadata_sampling}"
        )

    # ----------------------------------------------------------------------------------------------
    # Clipping
    # ----------------------------------------------------------------------------------------------

    if metadata_clipping != CLIPPING_MECHANISM:

        raise RuntimeError(
            f"{dataset_id}: clipping mechanism mismatch.\n"
            f"Configuration : {CLIPPING_MECHANISM}\n"
            f"Metadata      : {metadata_clipping}"
        )

    # ----------------------------------------------------------------------------------------------
    # Loss reduction
    # ----------------------------------------------------------------------------------------------

    if metadata_loss_reduction != LOSS_REDUCTION:

        raise RuntimeError(
            f"{dataset_id}: loss-reduction mismatch.\n"
            f"Configuration : {LOSS_REDUCTION}\n"
            f"Metadata      : {metadata_loss_reduction}"
        )

    # ----------------------------------------------------------------------------------------------
    # Delta
    # ----------------------------------------------------------------------------------------------

    if not (
        np.isfinite(
            delta
        )
        and
        0 < delta < 1
    ):

        raise ValueError(
            f"{dataset_id}: invalid delta={delta}."
        )

    # ----------------------------------------------------------------------------------------------
    # Noise multiplier
    # ----------------------------------------------------------------------------------------------

    if not (
        np.isfinite(
            noise_multiplier
        )
        and
        noise_multiplier > 0
    ):

        raise ValueError(
            f"{dataset_id}: invalid noise multiplier="
            f"{noise_multiplier}."
        )

    PRIVACY_PARAMETERS[
        dataset_id
    ] = {

        "n_train":
            n_train,

        "target_epsilon":
            target_epsilon,

        "delta":
            delta,

        "batch_size":
            batch_size,

        "poisson_sample_rate":
            sample_rate,

        "epochs":
            epochs,

        "max_grad_norm":
            max_grad_norm,

        "noise_multiplier":
            noise_multiplier,

        "accountant":
            metadata_accountant,

        "sampling":
            metadata_sampling,

        "clipping":
            metadata_clipping,

        "loss_reduction":
            metadata_loss_reduction,
    }


# --------------------------------------------------------------------------------------------------
# 14. Validate Notebook 11 Configured-Schedule Accounting
# --------------------------------------------------------------------------------------------------

REQUIRED_NB11_ACCOUNTING_COLUMNS = [

    "dataset",
    "configured_schedule_epsilon",
]

missing_nb11_columns = [
    column
    for column in REQUIRED_NB11_ACCOUNTING_COLUMNS
    if column not in NB11_CONFIGURED_ACCOUNTING_DF.columns
]

if missing_nb11_columns:

    raise RuntimeError(
        "Notebook 11 configured accounting is missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_nb11_columns
        )
    )


if set(
    NB11_CONFIGURED_ACCOUNTING_DF[
        "dataset"
    ].astype(
        str
    )
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Notebook 11 configured-schedule accounting dataset registry "
        "does not match the canonical dataset registry."
    )


for dataset_id in DATASET_IDS:

    accounting_rows = (
        NB11_CONFIGURED_ACCOUNTING_DF[
            NB11_CONFIGURED_ACCOUNTING_DF[
                "dataset"
            ].astype(
                str
            )
            ==
            dataset_id
        ]
    )

    if len(
        accounting_rows
    ) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one Notebook 11 "
            f"configured accounting row; observed "
            f"{len(accounting_rows)}."
        )

    configured_epsilon = float(
        accounting_rows.iloc[
            0
        ][
            "configured_schedule_epsilon"
        ]
    )

    if not (
        np.isfinite(
            configured_epsilon
        )
        and
        configured_epsilon > 0
    ):

        raise RuntimeError(
            f"{dataset_id}: invalid configured-schedule "
            f"epsilon={configured_epsilon}."
        )

    PRIVACY_PARAMETERS[
        dataset_id
    ][
        "configured_schedule_epsilon"
    ] = configured_epsilon


# --------------------------------------------------------------------------------------------------
# 15. Training Configuration
# --------------------------------------------------------------------------------------------------
# Checkpoint policy follows the validated DP-CTGAN training protocol:
# checkpoint only after a completed epoch;
# automatic resume from the latest valid checkpoint;
# optimizer/privacy-accountant/RNG/training history state must be preserved.
# --------------------------------------------------------------------------------------------------

CHECKPOINT_INTERVAL = 10
FORCE_RESTART = False
MAX_PERIODIC_CHECKPOINTS = 5


if CHECKPOINT_INTERVAL != 10:

    raise RuntimeError(
        "SPP-GAN checkpoint interval must be exactly 10 epochs."
    )

if MAX_PERIODIC_CHECKPOINTS <= 0:

    raise RuntimeError(
        "MAX_PERIODIC_CHECKPOINTS must be positive."
    )


# --------------------------------------------------------------------------------------------------
# 16. Cross-Artifact Training Configuration Summary
# --------------------------------------------------------------------------------------------------

TRAINING_CONFIGURATION = {

    "framework":
        "SPP-GAN",

    "lambda_stat":
        float(
            NB09_LAMBDA_STAT
        ),

    "dp_batch_size":
        int(
            DP_BATCH_SIZE
        ),

    "target_epsilon":
        float(
            TARGET_EPSILON
        ),

    "max_grad_norm":
        float(
            MAX_GRAD_NORM
        ),

    "epochs":
        int(
            DP_EPOCHS
        ),

    "accountant":
        ACCOUNTANT,

    "sampling_mechanism":
        SAMPLING_MECHANISM,

    "clipping_mechanism":
        CLIPPING_MECHANISM,

    "loss_reduction":
        LOSS_REDUCTION,

    "generator_private":
        bool(
            GENERATOR_PRIVATE
        ),

    "statistical_guidance_private":
        bool(
            STATISTICAL_GUIDANCE_PRIVATE
        ),

    "preprocessing_private":
        bool(
            PREPROCESSING_PRIVATE
        ),

    "end_to_end_privacy_claim":
        bool(
            END_TO_END_PRIVACY_CLAIM
        ),

    "checkpoint_interval":
        int(
            CHECKPOINT_INTERVAL
        ),

    "force_restart":
        bool(
            FORCE_RESTART
        ),

    "max_periodic_checkpoints":
        int(
            MAX_PERIODIC_CHECKPOINTS
        ),
}


# --------------------------------------------------------------------------------------------------
# 17. Persisted Source Artifact Fingerprints
# --------------------------------------------------------------------------------------------------

CONFIGURATION_SOURCE_ARTIFACTS = {

    "nb09_statistical_guidance_configuration":
        NB09_CONFIG_PATH,

    "nb10_privacy_configuration":
        NB10_PRIVACY_CONFIG_PATH,

    "nb10_privacy_metadata":
        NB10_PRIVACY_METADATA_PATH,

    "nb11_accounting_configuration":
        NB11_ACCOUNTING_CONFIG_PATH,

    "nb11_configured_rdp_accounting":
        NB11_CONFIGURED_ACCOUNTING_PATH,
}

CONFIGURATION_SOURCE_HASHES = {}

for artifact_name, artifact_path in (
    CONFIGURATION_SOURCE_ARTIFACTS.items()
):

    CONFIGURATION_SOURCE_HASHES[
        artifact_name
    ] = sha256_file(
        artifact_path
    )


# --------------------------------------------------------------------------------------------------
# 18. Configuration Snapshot
# --------------------------------------------------------------------------------------------------

NB12_CONFIG_ROOT = (
    NB12_ROOT
    / "configuration"
)

NB12_CONFIG_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

NB12_CONFIGURATION_SNAPSHOT_PATH = (
    NB12_CONFIG_ROOT
    / "sppgan_training_configuration.json"
)

NB12_CONFIGURATION_SNAPSHOT = {

    "framework":
        "SPP-GAN",

    "source_notebooks": {

        "notebook_09":
            str(
                NB09_CONFIG_PATH
            ),

        "notebook_10":
            str(
                NB10_PRIVACY_CONFIG_PATH
            ),

        "notebook_10_metadata":
            str(
                NB10_PRIVACY_METADATA_PATH
            ),

        "notebook_11":
            str(
                NB11_ACCOUNTING_CONFIG_PATH
            ),

        "notebook_11_configured_accounting":
            str(
                NB11_CONFIGURED_ACCOUNTING_PATH
            ),
    },

    "training_configuration":
        TRAINING_CONFIGURATION,

    "dataset_privacy_parameters":
        PRIVACY_PARAMETERS,

    "source_artifact_sha256":
        CONFIGURATION_SOURCE_HASHES,

    "privacy_boundary": {

        "protected_component":
            "critic",

        "generator_private":
            bool(
                GENERATOR_PRIVATE
            ),

        "statistical_guidance_private":
            bool(
                STATISTICAL_GUIDANCE_PRIVATE
            ),

        "preprocessing_private":
            bool(
                PREPROCESSING_PRIVATE
            ),

        "end_to_end_privacy_claim":
            bool(
                END_TO_END_PRIVACY_CLAIM
            ),

        "achieved_training_epsilon":
            "deferred_to_notebook_12_training_accountant",
    },
}


with open(
    NB12_CONFIGURATION_SNAPSHOT_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        NB12_CONFIGURATION_SNAPSHOT,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 19. Final Section 2 Validation
# --------------------------------------------------------------------------------------------------

SECTION_2_CHECKS = {

    "project_root_exists":
        PROJECT_ROOT.exists(),

    "nb09_configuration_exists":
        NB09_CONFIG_PATH.exists(),

    "nb10_configuration_exists":
        NB10_PRIVACY_CONFIG_PATH.exists(),

    "nb10_metadata_exists":
        NB10_PRIVACY_METADATA_PATH.exists(),

    "nb11_configuration_exists":
        NB11_ACCOUNTING_CONFIG_PATH.exists(),

    "nb11_accounting_exists":
        NB11_CONFIGURED_ACCOUNTING_PATH.exists(),

    "nb09_lambda_stat_valid":
        np.isfinite(
            NB09_LAMBDA_STAT
        )
        and
        NB09_LAMBDA_STAT >= 0,

    "nb10_metadata_three_datasets":
        len(
            NB10_PRIVACY_METADATA_DF
        ) == 3,

    "dp_batch_size_valid":
        DP_BATCH_SIZE > 0,

    "target_epsilon_valid":
        TARGET_EPSILON > 0,

    "max_grad_norm_valid":
        MAX_GRAD_NORM > 0,

    "epochs_valid":
        DP_EPOCHS > 0,

    "accountant_rdp":
        ACCOUNTANT == "rdp",

    "sampling_poisson":
        SAMPLING_MECHANISM == "poisson",

    "clipping_flat":
        CLIPPING_MECHANISM == "flat",

    "loss_reduction_mean":
        LOSS_REDUCTION == "mean",

    "generator_private_false":
        GENERATOR_PRIVATE is False,

    "statistical_guidance_private_false":
        STATISTICAL_GUIDANCE_PRIVATE is False,

    "preprocessing_private_false":
        PREPROCESSING_PRIVATE is False,

    "end_to_end_privacy_claim_false":
        END_TO_END_PRIVACY_CLAIM is False,

    "configured_accounting_three_datasets":
        len(
            NB11_CONFIGURED_ACCOUNTING_DF
        ) == 3,

    "checkpoint_interval_ten":
        CHECKPOINT_INTERVAL == 10,

    "configuration_snapshot_exists":
        NB12_CONFIGURATION_SNAPSHOT_PATH.exists(),
}


FAILED_SECTION_2_CHECKS = [
    name
    for name, passed in SECTION_2_CHECKS.items()
    if not passed
]


print()
print("-" * 100)
print("SECTION 2 VALIDATION")
print("-" * 100)

for check_name, passed in SECTION_2_CHECKS.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )

if FAILED_SECTION_2_CHECKS:

    raise RuntimeError(
        "Section 2 validation failed:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in FAILED_SECTION_2_CHECKS
        )
    )


# --------------------------------------------------------------------------------------------------
# 20. Final Output
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 2 — LOAD CONFIGURATION COMPLETE")
print("=" * 100)

print(
    f"✓ λ_stat                    : "
    f"{NB09_LAMBDA_STAT:.6f}"
)

print(
    f"✓ DP batch size             : "
    f"{DP_BATCH_SIZE}"
)

print(
    f"✓ Target epsilon            : "
    f"{TARGET_EPSILON:.6f}"
)

print(
    f"✓ Maximum gradient norm     : "
    f"{MAX_GRAD_NORM:.6f}"
)

print(
    f"✓ Training epochs           : "
    f"{DP_EPOCHS}"
)

print(
    f"✓ Accountant                : "
    f"{ACCOUNTANT}"
)

print(
    f"✓ Sampling mechanism        : "
    f"{SAMPLING_MECHANISM}"
)

print(
    f"✓ Clipping mechanism        : "
    f"{CLIPPING_MECHANISM}"
)

print(
    f"✓ Loss reduction            : "
    f"{LOSS_REDUCTION}"
)

print(
    f"✓ Generator private         : "
    f"{GENERATOR_PRIVATE}"
)

print(
    f"✓ Statistical guidance priv.: "
    f"{STATISTICAL_GUIDANCE_PRIVATE}"
)

print(
    f"✓ Preprocessing private     : "
    f"{PREPROCESSING_PRIVATE}"
)

print(
    f"✓ End-to-end DP claim       : "
    f"{END_TO_END_PRIVACY_CLAIM}"
)

print(
    f"✓ Checkpoint interval       : "
    f"{CHECKPOINT_INTERVAL} epochs"
)

print(
    f"✓ Force restart             : "
    f"{FORCE_RESTART}"
)

print(
    f"✓ Configured accounting     : "
    f"loaded for {len(PRIVACY_PARAMETERS)} datasets"
)

print(
    f"✓ Configuration snapshot    : "
    f"{NB12_CONFIGURATION_SNAPSHOT_PATH}"
)

print(
    f"✓ Section 2 checks          : "
    f"{len(SECTION_2_CHECKS)}/{len(SECTION_2_CHECKS)} PASS"
)

print()
print("STATUS: PASS — SECTION 2 READY FOR FREEZE")
print("=" * 100)

2. LOAD CONFIGURATION
✓ MyDrive        : /content/drive/MyDrive
✓ Project root   : /content/drive/MyDrive/SPP_GAN_Research

Required persisted configuration artifacts:
  ✓ Notebook 09 statistical guidance configuration: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_09/configuration/sppgan_statistical_guidance_configuration.json
  ✓ Notebook 10 privacy configuration            : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/configuration/sppgan_privacy_configuration.json
  ✓ Notebook 10 privacy metadata                 : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/metadata/sppgan_privacy_metadata.csv
  ✓ Notebook 11 privacy accounting configuration : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11/configuration/sppgan_privacy_accounting_configuration.json
  ✓ Notebook 11 configured RDP accounting        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11/accounting/sppgan_con

In [52]:
# ==================================================================================================
# SECTION 3 — LOAD PROCESSED TRAINING DATA
# ==================================================================================================

print("=" * 100)
print("3. LOAD PROCESSED TRAINING DATA")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Canonical Notebook 02 Artifact Locations
# --------------------------------------------------------------------------------------------------

CANONICAL_NB02_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_02"
)

CANONICAL_NATIVE_ROOT = (
    CANONICAL_NB02_ROOT
    / "native"
)

CANONICAL_PREPROCESSORS_ROOT = (
    CANONICAL_NB02_ROOT
    / "preprocessors"
)

CANONICAL_SCHEMA_ROOT = (
    CANONICAL_NB02_ROOT
    / "schemas"
)

CANONICAL_METADATA_ROOT = (
    CANONICAL_SCHEMA_ROOT
    / "metadata"
)

CANONICAL_NATIVE_MANIFEST = (
    CANONICAL_NATIVE_ROOT
    / "native_dataset_manifest.csv"
)

CANONICAL_PREPROCESSOR_MANIFEST = (
    CANONICAL_SCHEMA_ROOT
    / "preprocessor_manifest.csv"
)


# --------------------------------------------------------------------------------------------------
# 2. Verify Notebook 02 Frozen Artifact Layer
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("NOTEBOOK 02 CANONICAL ARTIFACT VERIFICATION")
print("-" * 100)

REQUIRED_NB02_PATHS = {

    "Notebook 02 root":
        CANONICAL_NB02_ROOT,

    "Native root":
        CANONICAL_NATIVE_ROOT,

    "Preprocessor root":
        CANONICAL_PREPROCESSORS_ROOT,

    "Schema root":
        CANONICAL_SCHEMA_ROOT,

    "Metadata root":
        CANONICAL_METADATA_ROOT,

    "Native manifest":
        CANONICAL_NATIVE_MANIFEST,

    "Preprocessor manifest":
        CANONICAL_PREPROCESSOR_MANIFEST,
}


for label, path in REQUIRED_NB02_PATHS.items():

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{label:<25}: {path}"
    )


missing_nb02_paths = [

    str(path)

    for path in REQUIRED_NB02_PATHS.values()

    if not path.exists()
]


if missing_nb02_paths:

    raise FileNotFoundError(
        "Required Notebook 02 persisted artifacts are incomplete.\n\n"
        +
        "\n".join(
            f"  - {path}"
            for path in missing_nb02_paths
        )
        +
        "\n\n"
        "Notebook 12 will NOT reconstruct Notebook 02 artifacts."
    )


# --------------------------------------------------------------------------------------------------
# 3. Load Native Dataset Manifest
# --------------------------------------------------------------------------------------------------

NATIVE_MANIFEST_DF = pd.read_csv(
    CANONICAL_NATIVE_MANIFEST,
    low_memory=False,
)


if NATIVE_MANIFEST_DF.empty:

    raise RuntimeError(
        "Notebook 02 native_dataset_manifest.csv is empty."
    )


REQUIRED_NATIVE_MANIFEST_COLUMNS = {

    "dataset_id",
    "split",
    "relative_path",
    "absolute_path",
    "rows",
    "columns",
}


missing_manifest_columns = (
    REQUIRED_NATIVE_MANIFEST_COLUMNS
    -
    set(
        NATIVE_MANIFEST_DF.columns
    )
)


if missing_manifest_columns:

    raise RuntimeError(
        "Notebook 02 native manifest is missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in sorted(
                missing_manifest_columns
            )
        )
    )


print()
print(
    f"✓ Native manifest loaded : "
    f"{len(NATIVE_MANIFEST_DF)} rows"
)


# --------------------------------------------------------------------------------------------------
# 4. Resolve Canonical TRAIN Files from Native Manifest
# --------------------------------------------------------------------------------------------------

TRAIN_DATA_PATHS = {}

TRAIN_MANIFEST_ROWS = {}


for dataset_id in DATASET_IDS:

    matches = NATIVE_MANIFEST_DF[
        (
            NATIVE_MANIFEST_DF[
                "dataset_id"
            ].astype(
                str
            )
            ==
            dataset_id
        )
        &
        (
            NATIVE_MANIFEST_DF[
                "split"
            ].astype(
                str
            ).str.lower()
            ==
            "train"
        )
    ]

    if len(
        matches
    ) != 1:

        raise RuntimeError(
            f"Notebook 02 native manifest must contain exactly "
            f"one TRAIN record for {dataset_id}. "
            f"Observed: {len(matches)}."
        )


    manifest_row = matches.iloc[
        0
    ]

    TRAIN_MANIFEST_ROWS[
        dataset_id
    ] = manifest_row


    # ----------------------------------------------------------------------------------------------
    # Prefer the persisted absolute path only after verifying it belongs to the canonical
    # Notebook 02 native layer.
    # ----------------------------------------------------------------------------------------------

    absolute_path_value = str(
        manifest_row[
            "absolute_path"
        ]
    ).strip()

    relative_path_value = str(
        manifest_row[
            "relative_path"
        ]
    ).strip()


    if absolute_path_value:

        candidate_path = Path(
            absolute_path_value
        )

    else:

        candidate_path = (
            CANONICAL_NB02_ROOT
            / relative_path_value
        )


    if not candidate_path.exists():

        # Resolve relative path against the canonical Notebook 02 root
        # if the persisted absolute path is stale.
        fallback_path = (
            CANONICAL_NB02_ROOT
            / relative_path_value
        )

        if fallback_path.exists():

            candidate_path = fallback_path

        else:

            raise FileNotFoundError(
                f"Notebook 02 TRAIN artifact does not exist for "
                f"{dataset_id}.\n\n"
                f"Manifest absolute path:\n"
                f"{absolute_path_value}\n\n"
                f"Manifest relative path:\n"
                f"{relative_path_value}"
            )


    candidate_path = candidate_path.resolve()


    # ----------------------------------------------------------------------------------------------
    # Security / provenance guard:
    # TRAIN data must come from the canonical native Notebook 02 layer.
    # ----------------------------------------------------------------------------------------------

    try:

        candidate_path.relative_to(
            CANONICAL_NATIVE_ROOT.resolve()
        )

    except ValueError:

        raise RuntimeError(
            f"{dataset_id}: resolved TRAIN artifact is outside "
            f"the canonical Notebook 02 native layer.\n"
            f"Resolved path: {candidate_path}\n"
            f"Expected root: {CANONICAL_NATIVE_ROOT}"
        )


    if candidate_path.suffix.lower() != ".csv":

        raise RuntimeError(
            f"{dataset_id}: authoritative Notebook 02 native TRAIN "
            f"artifact must be CSV.\n"
            f"Resolved artifact: {candidate_path}"
        )


    TRAIN_DATA_PATHS[
        dataset_id
    ] = candidate_path


# --------------------------------------------------------------------------------------------------
# 5. Load Native TRAIN Data Sequentially
# --------------------------------------------------------------------------------------------------
#
# Important:
# - Only TRAIN data are loaded.
# - Validation and TEST data are not loaded.
# - Native tabular representation is retained.
# - No preprocessing is fitted here.
# - No encoded/scaled matrix is used here.
# - Target and provenance columns remain available in the native dataset.
#
# RAM policy:
# One dataset is processed at a time before being converted into the
# model-ready representation in a later section.
# --------------------------------------------------------------------------------------------------

TRAIN_DATAFRAMES = {}

TRAIN_METADATA = {}

for dataset_id in DATASET_IDS:

    path = TRAIN_DATA_PATHS[
        dataset_id
    ]

    print()
    print("-" * 100)
    print(
        f"DATASET : {dataset_id}"
    )
    print("-" * 100)

    print(
        f"  ✓ TRAIN artifact : {path}"
    )

    # ----------------------------------------------------------------------------------------------
    # Load native TRAIN CSV
    # ----------------------------------------------------------------------------------------------

    df = pd.read_csv(
        path,
        low_memory=False,
    )

    # ----------------------------------------------------------------------------------------------
    # Basic structural validation
    # ----------------------------------------------------------------------------------------------

    if df.empty:

        raise RuntimeError(
            f"{dataset_id}: native TRAIN dataset is empty."
        )


    if len(
        df
    ) != TRAIN_ROWS[
        dataset_id
    ]:

        raise RuntimeError(
            f"{dataset_id}: TRAIN row mismatch.\n"
            f"Expected : {TRAIN_ROWS[dataset_id]:,}\n"
            f"Observed : {len(df):,}"
        )


    if not df.columns.is_unique:

        raise RuntimeError(
            f"{dataset_id}: native TRAIN columns are not unique."
        )


    # ----------------------------------------------------------------------------------------------
    # Resolve target column
    # ----------------------------------------------------------------------------------------------

    TARGET_COLUMNS = {

        "adult_income":
            "income",

        "bank_marketing":
            "y",

        "diabetes_130us":
            "readmitted",
    }


    target_column = TARGET_COLUMNS[
        dataset_id
    ]


    if target_column not in df.columns:

        raise RuntimeError(
            f"{dataset_id}: expected target column "
            f"'{target_column}' is missing."
        )


    # ----------------------------------------------------------------------------------------------
    # Resolve provenance column
    # ----------------------------------------------------------------------------------------------

    PROVENANCE_COLUMN = "__original_row_id__"


    if PROVENANCE_COLUMN not in df.columns:

        raise RuntimeError(
            f"{dataset_id}: required Notebook 02 provenance "
            f"column '{PROVENANCE_COLUMN}' is missing."
        )


    # ----------------------------------------------------------------------------------------------
    # Verify target is non-empty
    # ----------------------------------------------------------------------------------------------

    if df[
        target_column
    ].isna().all():

        raise RuntimeError(
            f"{dataset_id}: target column '{target_column}' "
            f"is entirely missing."
        )


    # ----------------------------------------------------------------------------------------------
    # Verify provenance
    # ----------------------------------------------------------------------------------------------

    if df[
        PROVENANCE_COLUMN
    ].isna().any():

        raise RuntimeError(
            f"{dataset_id}: provenance column contains missing values."
        )


    if not df[
        PROVENANCE_COLUMN
    ].is_unique:

        raise RuntimeError(
            f"{dataset_id}: provenance column is not unique."
        )


    # ----------------------------------------------------------------------------------------------
    # Verify manifest dimensions
    # ----------------------------------------------------------------------------------------------

    manifest_rows = int(
        TRAIN_MANIFEST_ROWS[
            dataset_id
        ][
            "rows"
        ]
    )

    manifest_columns = int(
        TRAIN_MANIFEST_ROWS[
            dataset_id
        ][
            "columns"
        ]
    )


    if manifest_rows != len(
        df
    ):

        raise RuntimeError(
            f"{dataset_id}: manifest row count mismatch.\n"
            f"Manifest : {manifest_rows}\n"
            f"Loaded   : {len(df)}"
        )


    if manifest_columns != len(
        df.columns
    ):

        raise RuntimeError(
            f"{dataset_id}: manifest column count mismatch.\n"
            f"Manifest : {manifest_columns}\n"
            f"Loaded   : {len(df.columns)}"
        )


    # ----------------------------------------------------------------------------------------------
    # Persist native TRAIN dataframe
    # ----------------------------------------------------------------------------------------------

    TRAIN_DATAFRAMES[
        dataset_id
    ] = df


    TRAIN_METADATA[
        dataset_id
    ] = {

        "dataset_id":
            dataset_id,

        "split":
            "train",

        "path":
            str(
                path
            ),

        "rows":
            int(
                df.shape[
                    0
                ]
            ),

        "columns":
            int(
                df.shape[
                    1
                ]
            ),

        "target_column":
            target_column,

        "provenance_column":
            PROVENANCE_COLUMN,

        "target_present":
            True,

        "provenance_present":
            True,
    }


    print(
        f"  ✓ Rows              : "
        f"{df.shape[0]:,}"
    )

    print(
        f"  ✓ Columns           : "
        f"{df.shape[1]}"
    )

    print(
        f"  ✓ Target column     : "
        f"{target_column}"
    )

    print(
        f"  ✓ Provenance column : "
        f"{PROVENANCE_COLUMN}"
    )

    print(
        "  ✓ Native TRAIN data validated"
    )


# --------------------------------------------------------------------------------------------------
# 6. Cross-Dataset Training Row Validation
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    observed_rows = len(
        TRAIN_DATAFRAMES[
            dataset_id
        ]
    )

    expected_rows = TRAIN_ROWS[
        dataset_id
    ]

    if observed_rows != expected_rows:

        raise RuntimeError(
            f"{dataset_id}: final TRAIN row validation failed."
        )


# --------------------------------------------------------------------------------------------------
# 7. Explicit Data-Usage Boundary
# --------------------------------------------------------------------------------------------------

VALIDATION_DATA_LOADED = False
TEST_DATA_LOADED = False
SYNTHETIC_DATA_LOADED = False

if VALIDATION_DATA_LOADED:

    raise RuntimeError(
        "Validation data must not be loaded in Section 3."
    )

if TEST_DATA_LOADED:

    raise RuntimeError(
        "Test data must not be loaded in Section 3."
    )

if SYNTHETIC_DATA_LOADED:

    raise RuntimeError(
        "Synthetic data must not be loaded in Section 3."
    )


# --------------------------------------------------------------------------------------------------
# 8. Final TRAIN Data Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("TRAIN DATA SUMMARY")
print("=" * 100)

for dataset_id in DATASET_IDS:

    df = TRAIN_DATAFRAMES[
        dataset_id
    ]

    print(
        f"  {dataset_id:<20} | "
        f"Rows={df.shape[0]:>8,} | "
        f"Columns={df.shape[1]:>5} | "
        f"Target={TRAIN_METADATA[dataset_id]['target_column']:<12} | "
        f"Path={TRAIN_DATA_PATHS[dataset_id]}"
    )


# --------------------------------------------------------------------------------------------------
# 9. Section 3 Validation
# --------------------------------------------------------------------------------------------------

SECTION_3_CHECKS = {

    "nb02_root_exists":
        CANONICAL_NB02_ROOT.exists(),

    "native_root_exists":
        CANONICAL_NATIVE_ROOT.exists(),

    "schema_root_exists":
        CANONICAL_SCHEMA_ROOT.exists(),

    "metadata_root_exists":
        CANONICAL_METADATA_ROOT.exists(),

    "native_manifest_exists":
        CANONICAL_NATIVE_MANIFEST.exists(),

    "preprocessor_manifest_exists":
        CANONICAL_PREPROCESSOR_MANIFEST.exists(),

    "native_manifest_nonempty":
        not NATIVE_MANIFEST_DF.empty,

    "three_training_datasets":
        len(TRAIN_DATAFRAMES) == 3,

    "adult_income_train_rows":
        len(
            TRAIN_DATAFRAMES[
                "adult_income"
            ]
        )
        ==
        TRAIN_ROWS[
            "adult_income"
        ],

    "bank_marketing_train_rows":
        len(
            TRAIN_DATAFRAMES[
                "bank_marketing"
            ]
        )
        ==
        TRAIN_ROWS[
            "bank_marketing"
        ],

    "diabetes_train_rows":
        len(
            TRAIN_DATAFRAMES[
                "diabetes_130us"
            ]
        )
        ==
        TRAIN_ROWS[
            "diabetes_130us"
        ],

    "adult_income_target":
        TRAIN_METADATA[
            "adult_income"
        ][
            "target_column"
        ]
        ==
        "income",

    "bank_marketing_target":
        TRAIN_METADATA[
            "bank_marketing"
        ][
            "target_column"
        ]
        ==
        "y",

    "diabetes_target":
        TRAIN_METADATA[
            "diabetes_130us"
        ][
            "target_column"
        ]
        ==
        "readmitted",

    "adult_income_provenance":
        "__original_row_id__"
        in
        TRAIN_DATAFRAMES[
            "adult_income"
        ].columns,

    "bank_marketing_provenance":
        "__original_row_id__"
        in
        TRAIN_DATAFRAMES[
            "bank_marketing"
        ].columns,

    "diabetes_provenance":
        "__original_row_id__"
        in
        TRAIN_DATAFRAMES[
            "diabetes_130us"
        ].columns,

    "validation_not_loaded":
        VALIDATION_DATA_LOADED is False,

    "test_not_loaded":
        TEST_DATA_LOADED is False,

    "synthetic_not_loaded":
        SYNTHETIC_DATA_LOADED is False,
}


FAILED_SECTION_3_CHECKS = [

    name

    for name, passed in SECTION_3_CHECKS.items()

    if not passed
]


print()
print("-" * 100)
print("SECTION 3 VALIDATION")
print("-" * 100)

for check_name, passed in SECTION_3_CHECKS.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )


if FAILED_SECTION_3_CHECKS:

    raise RuntimeError(
        "Section 3 validation failed:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in FAILED_SECTION_3_CHECKS
        )
    )


# --------------------------------------------------------------------------------------------------
# 10. Completion Output
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 3 — LOAD PROCESSED TRAINING DATA COMPLETE")
print("=" * 100)

print(
    "✓ Authoritative Notebook 02 native TRAIN data loaded."
)

print(
    "✓ adult_income      : "
    f"{TRAIN_DATAFRAMES['adult_income'].shape[0]:,} rows × "
    f"{TRAIN_DATAFRAMES['adult_income'].shape[1]} columns"
)

print(
    "✓ bank_marketing    : "
    f"{TRAIN_DATAFRAMES['bank_marketing'].shape[0]:,} rows × "
    f"{TRAIN_DATAFRAMES['bank_marketing'].shape[1]} columns"
)

print(
    "✓ diabetes_130us    : "
    f"{TRAIN_DATAFRAMES['diabetes_130us'].shape[0]:,} rows × "
    f"{TRAIN_DATAFRAMES['diabetes_130us'].shape[1]} columns"
)

print(
    "✓ Validation data were not loaded."
)

print(
    "✓ Test data were not loaded."
)

print(
    "✓ Synthetic data were not loaded."
)

print(
    f"✓ Section 3 checks    : "
    f"{len(SECTION_3_CHECKS)}/{len(SECTION_3_CHECKS)} PASS"
)

print()
print(
    "STATUS: PASS — SECTION 3 TRAIN DATA LOAD READY FOR NEXT SECTION"
)
print("=" * 100)

3. LOAD PROCESSED TRAINING DATA

----------------------------------------------------------------------------------------------------
NOTEBOOK 02 CANONICAL ARTIFACT VERIFICATION
----------------------------------------------------------------------------------------------------
✓ Notebook 02 root         : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✓ Native root              : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native
✓ Preprocessor root        : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/preprocessors
✓ Schema root              : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas
✓ Metadata root            : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/metadata
✓ Native manifest          : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/native_dataset_manifest.csv
✓ Preprocessor manifest    : /content/drive/MyDrive/SPP_GAN_Rese

In [53]:
# ==================================================================================================
# 4. LOAD STATISTICAL GUIDANCE
# ==================================================================================================

print("=" * 100)
print("4. LOAD STATISTICAL GUIDANCE")
print("=" * 100)

GUIDANCE_ROOT = (
    NB03_ROOT /
    "guidance"
)

if not GUIDANCE_ROOT.exists():
    raise FileNotFoundError(
        f"Notebook 03 guidance root not found:\n{GUIDANCE_ROOT}"
    )

STATISTICAL_GUIDANCE = {}

for dataset_id in DATASET_IDS:

    guidance_path = (
        GUIDANCE_ROOT /
        f"{dataset_id}_spp_gan_statistical_guidance.json"
    )

    if not guidance_path.exists():
        raise FileNotFoundError(
            f"Statistical guidance missing:\n{guidance_path}"
        )

    with open(
        guidance_path,
        "r",
        encoding="utf-8"
    ) as f:

        guidance = json.load(f)

    if not isinstance(
        guidance,
        dict
    ):
        raise TypeError(
            f"{dataset_id}: guidance must be a JSON object."
        )

    required_keys = {
        "guidance_version",
        "guidance_type",
        "source_reference_version",
        "source_reference_type",
        "feature_schema",
        "numeric_feature_guidance",
        "categorical_feature_guidance",
        "strongest_numeric_pearson_dependencies",
        "strongest_numeric_spearman_dependencies",
        "strongest_categorical_dependencies",
        "target_policy",
        "identifier_policy",
        "provenance_policy",
        "evidence_policy",
    }

    missing = (
        required_keys -
        set(guidance.keys())
    )

    if missing:
        raise RuntimeError(
            f"{dataset_id}: missing guidance keys:\n{sorted(missing)}"
        )

    STATISTICAL_GUIDANCE[dataset_id] = guidance

    print(
        f"✓ {dataset_id:<20} "
        f"guidance={guidance_path}"
    )

print("\n✓ Notebook 03 statistical guidance loaded.")
print("✓ Statistical source remains TRAIN-only.")
print("✓ No statistical profile is reconstructed in Notebook 12.")

4. LOAD STATISTICAL GUIDANCE
✓ adult_income         guidance=/content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_03/guidance/adult_income_spp_gan_statistical_guidance.json
✓ bank_marketing       guidance=/content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_03/guidance/bank_marketing_spp_gan_statistical_guidance.json
✓ diabetes_130us       guidance=/content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_03/guidance/diabetes_130us_spp_gan_statistical_guidance.json

✓ Notebook 03 statistical guidance loaded.
✓ Statistical source remains TRAIN-only.
✓ No statistical profile is reconstructed in Notebook 12.


In [56]:
# ==================================================================================================
# SECTION 5 — LOAD SPP-GAN ARCHITECTURE
# ==================================================================================================

print("=" * 100)
print("5. LOAD SPP-GAN ARCHITECTURE")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Verify Notebook 08 Root
# --------------------------------------------------------------------------------------------------

if "NB08_ROOT" not in globals():

    raise RuntimeError(
        "NB08_ROOT is not defined.\n"
        "Notebook 08 canonical root must be initialized before Section 5."
    )


NB08_ROOT = Path(
    NB08_ROOT
)


if not NB08_ROOT.exists():

    raise FileNotFoundError(
        f"Notebook 08 root does not exist:\n{NB08_ROOT}"
    )


if not NB08_ROOT.is_dir():

    raise NotADirectoryError(
        f"Notebook 08 root is not a directory:\n{NB08_ROOT}"
    )


print(
    f"✓ Notebook 08 root : {NB08_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 2. Canonical Notebook 08 Architecture Artifacts
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_SUMMARY_PATH = (
    NB08_ROOT
    / "architecture"
    / "sppgan_architecture_summary.csv"
)

PARAMETER_COUNT_PATH = (
    NB08_ROOT
    / "metadata"
    / "sppgan_parameter_count.csv"
)

MODEL_CONFIG_PATH = (
    NB08_ROOT
    / "config"
    / "sppgan_model_configuration.json"
)

ARCHITECTURE_REGISTRY_PATH = (
    NB08_ROOT
    / "metadata"
    / "sppgan_architecture_artifacts.csv"
)


REQUIRED_ARCHITECTURE_FILES = {

    "architecture_summary":
        ARCHITECTURE_SUMMARY_PATH,

    "parameter_count":
        PARAMETER_COUNT_PATH,

    "model_configuration":
        MODEL_CONFIG_PATH,

    "architecture_registry":
        ARCHITECTURE_REGISTRY_PATH,
}


# --------------------------------------------------------------------------------------------------
# 3. Verify Required Architecture Artifacts
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("NOTEBOOK 08 ARCHITECTURE ARTIFACT VERIFICATION")
print("-" * 100)


for artifact_name, artifact_path in (
    REQUIRED_ARCHITECTURE_FILES.items()
):

    if not artifact_path.exists():

        raise FileNotFoundError(
            f"Required Notebook 08 artifact is missing.\n"
            f"Artifact : {artifact_name}\n"
            f"Path     : {artifact_path}"
        )


    if not artifact_path.is_file():

        raise RuntimeError(
            f"Notebook 08 artifact is not a file.\n"
            f"Artifact : {artifact_name}\n"
            f"Path     : {artifact_path}"
        )


    if artifact_path.stat().st_size == 0:

        raise RuntimeError(
            f"Notebook 08 artifact is empty.\n"
            f"Artifact : {artifact_name}\n"
            f"Path     : {artifact_path}"
        )


    print(
        f"✓ {artifact_name:<24} : {artifact_path}"
    )


# --------------------------------------------------------------------------------------------------
# 4. Load Architecture Artifacts
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_DF = pd.read_csv(
    ARCHITECTURE_SUMMARY_PATH
)

PARAMETER_DF = pd.read_csv(
    PARAMETER_COUNT_PATH
)

ARCHITECTURE_REGISTRY_DF = pd.read_csv(
    ARCHITECTURE_REGISTRY_PATH
)


with open(
    MODEL_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as f:

    ARCHITECTURE_CONFIG = json.load(
        f
    )


if not isinstance(
    ARCHITECTURE_CONFIG,
    dict,
):

    raise TypeError(
        "Notebook 08 model configuration must be a JSON object."
    )


print()
print("-" * 100)
print("LOADED NOTEBOOK 08 ARCHITECTURE ARTIFACTS")
print("-" * 100)

print(
    f"✓ Architecture summary : {ARCHITECTURE_DF.shape}"
)

print(
    f"✓ Parameter metadata    : {PARAMETER_DF.shape}"
)

print(
    f"✓ Architecture registry: {ARCHITECTURE_REGISTRY_DF.shape}"
)

print(
    "✓ Model configuration   : loaded"
)


# --------------------------------------------------------------------------------------------------
# 5. Canonical Dataset Registry
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASETS = set(
    DATASET_IDS
)


if EXPECTED_DATASETS != {
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
}:

    raise RuntimeError(
        "DATASET_IDS does not match the frozen SPP-GAN dataset registry."
    )


# --------------------------------------------------------------------------------------------------
# 6. Architecture Summary Schema
# --------------------------------------------------------------------------------------------------

REQUIRED_ARCHITECTURE_COLUMNS = {

    "dataset",
    "target",
    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
    "latent_dim",
    "generator_hidden_1",
    "generator_hidden_2",
    "critic_hidden_1",
    "critic_hidden_2",
    "total_trainable_parameters",
    "numerical_activation",
    "categorical_training_activation",
    "categorical_probability_mapping",
    "hard_decoding",
    "critic_output",
    "statistical_guidance",
    "differential_privacy",
    "privacy_accounting",
    "training",
}


missing_architecture_columns = (
    REQUIRED_ARCHITECTURE_COLUMNS
    -
    set(
        ARCHITECTURE_DF.columns
    )
)


if missing_architecture_columns:

    raise RuntimeError(
        "Notebook 08 architecture summary is missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in sorted(
                missing_architecture_columns
            )
        )
    )


print(
    "✓ Architecture-summary schema validated."
)


# --------------------------------------------------------------------------------------------------
# 7. Architecture Dataset Coverage
# --------------------------------------------------------------------------------------------------

if ARCHITECTURE_DF.empty:

    raise RuntimeError(
        "Notebook 08 architecture summary is empty."
    )


architecture_datasets = set(
    ARCHITECTURE_DF[
        "dataset"
    ].astype(
        str
    )
)


if architecture_datasets != EXPECTED_DATASETS:

    raise RuntimeError(
        "Architecture dataset coverage mismatch.\n"
        f"Expected: {sorted(EXPECTED_DATASETS)}\n"
        f"Found   : {sorted(architecture_datasets)}"
    )


if (
    ARCHITECTURE_DF[
        "dataset"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate dataset architecture records detected."
    )


print(
    "✓ Architecture dataset coverage validated."
)

print(
    "✓ Architecture dataset uniqueness validated."
)


# --------------------------------------------------------------------------------------------------
# 8. Numeric Architecture Validation
# --------------------------------------------------------------------------------------------------

NUMERIC_ARCHITECTURE_COLUMNS = [

    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
    "latent_dim",
    "generator_hidden_1",
    "generator_hidden_2",
    "critic_hidden_1",
    "critic_hidden_2",
    "total_trainable_parameters",
]


for column in NUMERIC_ARCHITECTURE_COLUMNS:

    values = pd.to_numeric(
        ARCHITECTURE_DF[
            column
        ],
        errors="coerce",
    )


    if values.isna().any():

        raise RuntimeError(
            f"Architecture column contains non-numeric values:\n"
            f"{column}"
        )


    if (
        values <= 0
    ).any():

        raise RuntimeError(
            f"Architecture column contains non-positive values:\n"
            f"{column}"
        )


print(
    "✓ Numeric architecture parameters validated."
)


# --------------------------------------------------------------------------------------------------
# 9. Generative-Dimension Consistency
# --------------------------------------------------------------------------------------------------

for _, row in ARCHITECTURE_DF.iterrows():

    dataset_id = str(
        row[
            "dataset"
        ]
    )


    expected_dimension = (
        int(
            row[
                "numerical_features"
            ]
        )
        +
        int(
            row[
                "categorical_features"
            ]
        )
        +
        1
    )


    actual_dimension = int(
        row[
            "generative_dimension"
        ]
    )


    if actual_dimension != expected_dimension:

        raise RuntimeError(
            f"{dataset_id}: generative-dimension mismatch.\n"
            f"Expected : {expected_dimension}\n"
            f"Observed : {actual_dimension}"
        )


print(
    "✓ Generative-dimension consistency validated."
)


# --------------------------------------------------------------------------------------------------
# 10. Frozen Transformed Dimensions
# --------------------------------------------------------------------------------------------------

EXPECTED_TRANSFORMED_DIMENSIONS = {

    "adult_income":
        105,

    "bank_marketing":
        51,

    "diabetes_130us":
        2329,
}


for dataset_id in DATASET_IDS:

    row = ARCHITECTURE_DF[
        ARCHITECTURE_DF[
            "dataset"
        ].astype(
            str
        )
        ==
        dataset_id
    ].iloc[
        0
    ]


    observed_dimension = int(
        row[
            "transformed_dimension"
        ]
    )


    expected_dimension = (
        EXPECTED_TRANSFORMED_DIMENSIONS[
            dataset_id
        ]
    )


    if observed_dimension != expected_dimension:

        raise RuntimeError(
            f"{dataset_id}: transformed-dimension mismatch.\n"
            f"Expected : {expected_dimension}\n"
            f"Observed : {observed_dimension}"
        )


print(
    "✓ Frozen transformed dimensions validated."
)


# --------------------------------------------------------------------------------------------------
# 11. Latent and Hidden-Layer Architecture Contract
# --------------------------------------------------------------------------------------------------

LATENT_DIMENSIONS = (
    ARCHITECTURE_DF[
        "latent_dim"
    ]
    .astype(
        int
    )
    .unique()
    .tolist()
)


GENERATOR_HIDDEN_1 = (
    ARCHITECTURE_DF[
        "generator_hidden_1"
    ]
    .astype(
        int
    )
    .unique()
    .tolist()
)


GENERATOR_HIDDEN_2 = (
    ARCHITECTURE_DF[
        "generator_hidden_2"
    ]
    .astype(
        int
    )
    .unique()
    .tolist()
)


CRITIC_HIDDEN_1 = (
    ARCHITECTURE_DF[
        "critic_hidden_1"
    ]
    .astype(
        int
    )
    .unique()
    .tolist()
)


CRITIC_HIDDEN_2 = (
    ARCHITECTURE_DF[
        "critic_hidden_2"
    ]
    .astype(
        int
    )
    .unique()
    .tolist()
)


if LATENT_DIMENSIONS != [128]:

    raise RuntimeError(
        f"Unexpected latent dimension: {LATENT_DIMENSIONS}"
    )


if GENERATOR_HIDDEN_1 != [256]:

    raise RuntimeError(
        f"Unexpected generator hidden layer 1: {GENERATOR_HIDDEN_1}"
    )


if GENERATOR_HIDDEN_2 != [256]:

    raise RuntimeError(
        f"Unexpected generator hidden layer 2: {GENERATOR_HIDDEN_2}"
    )


if CRITIC_HIDDEN_1 != [256]:

    raise RuntimeError(
        f"Unexpected critic hidden layer 1: {CRITIC_HIDDEN_1}"
    )


if CRITIC_HIDDEN_2 != [256]:

    raise RuntimeError(
        f"Unexpected critic hidden layer 2: {CRITIC_HIDDEN_2}"
    )


print(
    "✓ Latent dimension validated : 128"
)

print(
    "✓ Generator hidden layers validated : 256 / 256"
)

print(
    "✓ Critic hidden layers validated : 256 / 256"
)


# --------------------------------------------------------------------------------------------------
# 12. Categorical Output Contract
# --------------------------------------------------------------------------------------------------

categorical_activation_values = (
    ARCHITECTURE_DF[
        "categorical_training_activation"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)


categorical_mapping_values = (
    ARCHITECTURE_DF[
        "categorical_probability_mapping"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)


if categorical_activation_values != [
    "gumbel_softmax"
]:

    raise RuntimeError(
        "Unexpected categorical training activation.\n"
        f"Observed: {categorical_activation_values}"
    )


if categorical_mapping_values != [
    "softmax"
]:

    raise RuntimeError(
        "Unexpected categorical probability mapping.\n"
        f"Observed: {categorical_mapping_values}"
    )


print(
    "✓ Categorical training activation : gumbel_softmax"
)

print(
    "✓ Categorical probability mapping : softmax"
)


# --------------------------------------------------------------------------------------------------
# 13. Numerical Activation Contract
# --------------------------------------------------------------------------------------------------

NUMERICAL_ACTIVATION_VALUES = (
    ARCHITECTURE_DF[
        "numerical_activation"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)


if NUMERICAL_ACTIVATION_VALUES != [
    "identity"
]:

    raise RuntimeError(
        "Notebook 08 numerical activation does not match "
        "the frozen architecture artifact.\n"
        f"Observed: {NUMERICAL_ACTIVATION_VALUES}"
    )


print(
    "✓ Numerical activation inherited from Notebook 08 : identity"
)

print(
    "  No architecture override is performed in Notebook 12."
)


# --------------------------------------------------------------------------------------------------
# 14. Critic Output Contract
# --------------------------------------------------------------------------------------------------

CRITIC_OUTPUT_VALUES = (
    ARCHITECTURE_DF[
        "critic_output"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)


if CRITIC_OUTPUT_VALUES != [
    "scalar"
]:

    raise RuntimeError(
        "Unexpected critic output contract.\n"
        f"Observed: {CRITIC_OUTPUT_VALUES}"
    )


print(
    "✓ Critic output contract : scalar"
)


# --------------------------------------------------------------------------------------------------
# 15. Statistical Guidance Contract
# --------------------------------------------------------------------------------------------------
#
# Notebook 08 records the implementation source for statistical guidance
# in the architecture summary. The authoritative value is "notebook_09".
#
# Notebook 12 Section 4 has already loaded the persisted Notebook 03
# statistical guidance artifacts.
#
# This field is therefore treated as an implementation/provenance field,
# not as a Boolean enable/disable flag.
# --------------------------------------------------------------------------------------------------

STATISTICAL_GUIDANCE_VALUES = (
    ARCHITECTURE_DF[
        "statistical_guidance"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)


EXPECTED_STATISTICAL_GUIDANCE_SOURCE = [
    "notebook_09"
]


if STATISTICAL_GUIDANCE_VALUES != (
    EXPECTED_STATISTICAL_GUIDANCE_SOURCE
):

    raise RuntimeError(
        "Unexpected statistical-guidance implementation source.\n"
        f"Expected: {EXPECTED_STATISTICAL_GUIDANCE_SOURCE}\n"
        f"Observed: {STATISTICAL_GUIDANCE_VALUES}"
    )


if "STATISTICAL_GUIDANCE" not in globals():

    raise RuntimeError(
        "STATISTICAL_GUIDANCE is not available.\n"
        "Notebook 03 statistical guidance must be loaded before Section 5."
    )


if set(
    STATISTICAL_GUIDANCE.keys()
) != EXPECTED_DATASETS:

    raise RuntimeError(
        "Statistical-guidance dataset coverage mismatch.\n"
        f"Expected: {sorted(EXPECTED_DATASETS)}\n"
        f"Found   : {sorted(STATISTICAL_GUIDANCE.keys())}"
    )


print(
    "✓ Statistical-guidance implementation source : Notebook 09"
)

print(
    "✓ Notebook 03 statistical-guidance linkage validated."
)


# --------------------------------------------------------------------------------------------------
# 16. Differential Privacy / Accounting Contract
# --------------------------------------------------------------------------------------------------

DP_VALUES = (
    ARCHITECTURE_DF[
        "differential_privacy"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)


ACCOUNTING_VALUES = (
    ARCHITECTURE_DF[
        "privacy_accounting"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)


if not DP_VALUES:

    raise RuntimeError(
        "Notebook 08 differential-privacy contract is empty."
    )


if not ACCOUNTING_VALUES:

    raise RuntimeError(
        "Notebook 08 privacy-accounting contract is empty."
    )


print(
    "✓ Differential-privacy architecture contract loaded."
)

print(
    "✓ Privacy-accounting architecture contract loaded."
)


# --------------------------------------------------------------------------------------------------
# 17. Parameter-Count Artifact Validation
# --------------------------------------------------------------------------------------------------

if PARAMETER_DF.empty:

    raise RuntimeError(
        "Notebook 08 parameter-count artifact is empty."
    )


if "dataset" not in PARAMETER_DF.columns:

    raise RuntimeError(
        "Notebook 08 parameter-count artifact does not contain "
        "the required 'dataset' column."
    )


if (
    PARAMETER_DF[
        "dataset"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate dataset entries detected in parameter-count artifact."
    )


parameter_datasets = set(
    PARAMETER_DF[
        "dataset"
    ].astype(
        str
    )
)


if parameter_datasets != EXPECTED_DATASETS:

    raise RuntimeError(
        "Parameter-count dataset coverage mismatch.\n"
        f"Expected: {sorted(EXPECTED_DATASETS)}\n"
        f"Found   : {sorted(parameter_datasets)}"
    )


print(
    "✓ Parameter-count dataset coverage validated."
)


# --------------------------------------------------------------------------------------------------
# 18. Parameter-Count Cross-Validation
# --------------------------------------------------------------------------------------------------

if "total_trainable_parameters" not in PARAMETER_DF.columns:

    raise RuntimeError(
        "Notebook 08 parameter-count artifact must contain "
        "'total_trainable_parameters'."
    )


for dataset_id in DATASET_IDS:

    architecture_row = ARCHITECTURE_DF[
        ARCHITECTURE_DF[
            "dataset"
        ].astype(
            str
        )
        ==
        dataset_id
    ].iloc[
        0
    ]


    parameter_row = PARAMETER_DF[
        PARAMETER_DF[
            "dataset"
        ].astype(
            str
        )
        ==
        dataset_id
    ].iloc[
        0
    ]


    architecture_total = int(
        architecture_row[
            "total_trainable_parameters"
        ]
    )


    parameter_total = int(
        parameter_row[
            "total_trainable_parameters"
        ]
    )


    if architecture_total != parameter_total:

        raise RuntimeError(
            f"{dataset_id}: total trainable parameter mismatch.\n"
            f"Architecture summary : {architecture_total}\n"
            f"Parameter metadata   : {parameter_total}"
        )


print(
    "✓ Architecture-summary / parameter-count consistency validated."
)


# --------------------------------------------------------------------------------------------------
# 19. Architecture Registry Validation
# --------------------------------------------------------------------------------------------------

if ARCHITECTURE_REGISTRY_DF.empty:

    raise RuntimeError(
        "Notebook 08 architecture registry is empty."
    )


if "dataset" in ARCHITECTURE_REGISTRY_DF.columns:

    registry_datasets = set(
        ARCHITECTURE_REGISTRY_DF[
            "dataset"
        ].astype(
            str
        )
    )


    if not registry_datasets.issuperset(
        EXPECTED_DATASETS
    ):

        raise RuntimeError(
            "Notebook 08 architecture registry does not cover "
            "all canonical datasets.\n"
            f"Expected: {sorted(EXPECTED_DATASETS)}\n"
            f"Found   : {sorted(registry_datasets)}"
        )


print(
    "✓ Architecture registry validated."
)


# --------------------------------------------------------------------------------------------------
# 20. Model Configuration Validation
# --------------------------------------------------------------------------------------------------

if not ARCHITECTURE_CONFIG:

    raise RuntimeError(
        "Notebook 08 model configuration is empty."
    )


def find_configuration_values(
    obj,
    candidate_keys,
    path="",
):

    matches = []

    normalized_candidates = {

        str(key)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")

        for key in candidate_keys
    }


    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            normalized_key = (
                str(key)
                .strip()
                .lower()
                .replace("-", "_")
                .replace(" ", "_")
            )


            current_path = (
                f"{path}.{key}"
                if path
                else str(key)
            )


            if normalized_key in normalized_candidates:

                matches.append(
                    (
                        current_path,
                        value,
                    )
                )


            matches.extend(
                find_configuration_values(
                    value,
                    candidate_keys,
                    current_path,
                )
            )


    elif isinstance(
        obj,
        list,
    ):

        for index, value in enumerate(
            obj
        ):

            matches.extend(
                find_configuration_values(
                    value,
                    candidate_keys,
                    f"{path}[{index}]",
                )
            )


    return matches


def validate_single_architecture_config_value(
    candidate_keys,
    expected_value,
    label,
):

    matches = find_configuration_values(
        ARCHITECTURE_CONFIG,
        candidate_keys,
    )


    if not matches:

        raise RuntimeError(
            f"Notebook 08 model configuration does not expose "
            f"required architecture parameter: {label}"
        )


    normalized_values = []


    for path, value in matches:

        if isinstance(
            value,
            bool,
        ):

            normalized_value = value

        elif isinstance(
            value,
            (
                int,
                float,
                np.integer,
                np.floating,
            ),
        ):

            normalized_value = int(
                value
            )

        else:

            normalized_value = str(
                value
            ).strip().lower()


        normalized_values.append(
            (
                path,
                normalized_value,
            )
        )


    unique_values = {
        value
        for _, value
        in normalized_values
    }


    if len(
        unique_values
    ) != 1:

        raise RuntimeError(
            f"Conflicting Notebook 08 configuration values "
            f"for {label}:\n"
            +
            "\n".join(
                f"  {path} = {value}"
                for path, value
                in normalized_values
            )
        )


    observed_value = next(
        iter(
            unique_values
        )
    )


    expected_normalized = (

        str(
            expected_value
        ).strip().lower()

        if isinstance(
            expected_value,
            str,
        )

        else expected_value
    )


    if observed_value != expected_normalized:

        raise RuntimeError(
            f"Notebook 08 configuration mismatch for {label}.\n"
            f"Expected : {expected_normalized}\n"
            f"Observed : {observed_value}"
        )


    return observed_value


validate_single_architecture_config_value(
    ["latent_dim"],
    128,
    "latent dimension",
)


validate_single_architecture_config_value(
    ["hidden_dim_1"],
    256,
    "hidden dimension 1",
)


validate_single_architecture_config_value(
    ["hidden_dim_2"],
    256,
    "hidden dimension 2",
)


validate_single_architecture_config_value(
    ["numerical_activation"],
    "identity",
    "numerical activation",
)


validate_single_architecture_config_value(
    ["categorical_training_activation"],
    "gumbel_softmax",
    "categorical training activation",
)


validate_single_architecture_config_value(
    ["categorical_probability_mapping"],
    "softmax",
    "categorical probability mapping",
)


print(
    "✓ Notebook 08 model configuration cross-validated."
)


# --------------------------------------------------------------------------------------------------
# 21. Create Validated SPP-GAN Architecture Registry
# --------------------------------------------------------------------------------------------------

SPPGAN_ARCHITECTURE = {}


for _, row in ARCHITECTURE_DF.iterrows():

    dataset_id = str(
        row[
            "dataset"
        ]
    )


    SPPGAN_ARCHITECTURE[
        dataset_id
    ] = {

        column:
            row[
                column
            ]

        for column
        in ARCHITECTURE_DF.columns
    }


# --------------------------------------------------------------------------------------------------
# 22. Expose Dataset-Specific Architecture Parameters
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_PARAMETERS = {}


for dataset_id in DATASET_IDS:

    architecture = (
        SPPGAN_ARCHITECTURE[
            dataset_id
        ]
    )


    ARCHITECTURE_PARAMETERS[
        dataset_id
    ] = {

        "generative_dimension":
            int(
                architecture[
                    "generative_dimension"
                ]
            ),

        "transformed_dimension":
            int(
                architecture[
                    "transformed_dimension"
                ]
            ),

        "numerical_features":
            int(
                architecture[
                    "numerical_features"
                ]
            ),

        "categorical_features":
            int(
                architecture[
                    "categorical_features"
                ]
            ),

        "latent_dim":
            int(
                architecture[
                    "latent_dim"
                ]
            ),

        "generator_hidden_dims":
            [
                int(
                    architecture[
                        "generator_hidden_1"
                    ]
                ),
                int(
                    architecture[
                        "generator_hidden_2"
                    ]
                ),
            ],

        "critic_hidden_dims":
            [
                int(
                    architecture[
                        "critic_hidden_1"
                    ]
                ),
                int(
                    architecture[
                        "critic_hidden_2"
                    ]
                ),
            ],

        "total_trainable_parameters":
            int(
                architecture[
                    "total_trainable_parameters"
                ]
            ),

        "numerical_activation":
            str(
                architecture[
                    "numerical_activation"
                ]
            ).strip().lower(),

        "categorical_training_activation":
            str(
                architecture[
                    "categorical_training_activation"
                ]
            ).strip().lower(),

        "categorical_probability_mapping":
            str(
                architecture[
                    "categorical_probability_mapping"
                ]
            ).strip().lower(),

        "hard_decoding":
            str(
                architecture[
                    "hard_decoding"
                ]
            ),

        "critic_output":
            str(
                architecture[
                    "critic_output"
                ]
            ),
    }


# --------------------------------------------------------------------------------------------------
# 23. Native TRAIN / Architecture Linkage
# --------------------------------------------------------------------------------------------------

if "TRAIN_DATAFRAMES" not in globals():

    raise RuntimeError(
        "TRAIN_DATAFRAMES is unavailable.\n"
        "Notebook 12 Section 3 must be completed before Section 5."
    )


for dataset_id in DATASET_IDS:

    if dataset_id not in TRAIN_DATAFRAMES:

        raise RuntimeError(
            f"{dataset_id}: native TRAIN data are unavailable."
        )


    native_column_count = (
        TRAIN_DATAFRAMES[
            dataset_id
        ].shape[
            1
        ]
    )


    generative_dimension = (
        ARCHITECTURE_PARAMETERS[
            dataset_id
        ][
            "generative_dimension"
        ]
    )


    expected_native_columns = (
        generative_dimension
        +
        1
    )


    if native_column_count != expected_native_columns:

        raise RuntimeError(
            f"{dataset_id}: native TRAIN / generative schema mismatch.\n"
            f"Native columns      : {native_column_count}\n"
            f"Generative dimension: {generative_dimension}\n"
            f"Expected native     : {expected_native_columns}"
        )


print(
    "✓ Native TRAIN / generative-schema linkage validated."
)


# --------------------------------------------------------------------------------------------------
# 24. Final Architecture Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SPP-GAN ARCHITECTURE LOAD AND VALIDATION COMPLETE")
print("=" * 100)

print()
print(
    f"Validated datasets : {len(SPPGAN_ARCHITECTURE)}"
)


for dataset_id in DATASET_IDS:

    architecture = (
        ARCHITECTURE_PARAMETERS[
            dataset_id
        ]
    )


    print()
    print(
        f"{dataset_id}"
    )

    print(
        f"  Generative dimension   : "
        f"{architecture['generative_dimension']}"
    )

    print(
        f"  Transformed dimension  : "
        f"{architecture['transformed_dimension']}"
    )

    print(
        f"  Numerical features     : "
        f"{architecture['numerical_features']}"
    )

    print(
        f"  Categorical features   : "
        f"{architecture['categorical_features']}"
    )

    print(
        f"  Latent dimension       : "
        f"{architecture['latent_dim']}"
    )

    print(
        f"  Generator hidden       : "
        f"{architecture['generator_hidden_dims']}"
    )

    print(
        f"  Critic hidden          : "
        f"{architecture['critic_hidden_dims']}"
    )

    print(
        f"  Trainable parameters   : "
        f"{architecture['total_trainable_parameters']:,}"
    )


# --------------------------------------------------------------------------------------------------
# 25. Final Section 5 Validation
# --------------------------------------------------------------------------------------------------

SECTION_5_CHECKS = {

    "nb08_root_exists":
        NB08_ROOT.exists(),

    "architecture_summary_exists":
        ARCHITECTURE_SUMMARY_PATH.exists(),

    "parameter_count_exists":
        PARAMETER_COUNT_PATH.exists(),

    "model_config_exists":
        MODEL_CONFIG_PATH.exists(),

    "architecture_registry_exists":
        ARCHITECTURE_REGISTRY_PATH.exists(),

    "architecture_summary_nonempty":
        not ARCHITECTURE_DF.empty,

    "parameter_count_nonempty":
        not PARAMETER_DF.empty,

    "architecture_registry_nonempty":
        not ARCHITECTURE_REGISTRY_DF.empty,

    "three_canonical_datasets":
        architecture_datasets == EXPECTED_DATASETS,

    "architecture_unique":
        not ARCHITECTURE_DF[
            "dataset"
        ].duplicated().any(),

    "generative_dimensions_valid":
        all(
            int(
                row[
                    "generative_dimension"
                ]
            )
            ==
            int(
                row[
                    "numerical_features"
                ]
            )
            +
            int(
                row[
                    "categorical_features"
                ]
            )
            +
            1
            for _, row
            in ARCHITECTURE_DF.iterrows()
        ),

    "transformed_dimensions_valid":
        all(
            ARCHITECTURE_PARAMETERS[
                dataset_id
            ][
                "transformed_dimension"
            ]
            ==
            EXPECTED_TRANSFORMED_DIMENSIONS[
                dataset_id
            ]
            for dataset_id
            in DATASET_IDS
        ),

    "latent_dimension_128":
        LATENT_DIMENSIONS == [128],

    "generator_hidden_256_256":
        GENERATOR_HIDDEN_1 == [256]
        and
        GENERATOR_HIDDEN_2 == [256],

    "critic_hidden_256_256":
        CRITIC_HIDDEN_1 == [256]
        and
        CRITIC_HIDDEN_2 == [256],

    "categorical_gumbel_softmax":
        categorical_activation_values
        ==
        ["gumbel_softmax"],

    "categorical_softmax_mapping":
        categorical_mapping_values
        ==
        ["softmax"],

    "numerical_identity":
        NUMERICAL_ACTIVATION_VALUES
        ==
        ["identity"],

    "critic_scalar":
        CRITIC_OUTPUT_VALUES
        ==
        ["scalar"],

    "statistical_guidance_source_notebook_09":
        STATISTICAL_GUIDANCE_VALUES
        ==
        ["notebook_09"],

    "statistical_guidance_dataset_coverage":
        set(
            STATISTICAL_GUIDANCE.keys()
        )
        ==
        EXPECTED_DATASETS,

    "parameter_dataset_coverage":
        parameter_datasets == EXPECTED_DATASETS,

    "parameter_count_cross_validation":
        all(
            int(
                ARCHITECTURE_DF[
                    ARCHITECTURE_DF[
                        "dataset"
                    ].astype(
                        str
                    )
                    ==
                    dataset_id
                ].iloc[
                    0
                ][
                    "total_trainable_parameters"
                ]
            )
            ==
            int(
                PARAMETER_DF[
                    PARAMETER_DF[
                        "dataset"
                    ].astype(
                        str
                    )
                    ==
                    dataset_id
                ].iloc[
                    0
                ][
                    "total_trainable_parameters"
                ]
            )
            for dataset_id
            in DATASET_IDS
        ),

    "model_configuration_valid":
        isinstance(
            ARCHITECTURE_CONFIG,
            dict,
        )
        and
        bool(
            ARCHITECTURE_CONFIG
        ),

    "train_architecture_linkage":
        all(
            dataset_id in TRAIN_DATAFRAMES
            and
            (
                TRAIN_DATAFRAMES[
                    dataset_id
                ].shape[
                    1
                ]
                ==
                ARCHITECTURE_PARAMETERS[
                    dataset_id
                ][
                    "generative_dimension"
                ]
                +
                1
            )
            for dataset_id
            in DATASET_IDS
        ),
}


FAILED_SECTION_5_CHECKS = [

    name

    for name, passed
    in SECTION_5_CHECKS.items()

    if not passed
]


print()
print("-" * 100)
print("SECTION 5 VALIDATION")
print("-" * 100)


for check_name, passed in SECTION_5_CHECKS.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )


if FAILED_SECTION_5_CHECKS:

    raise RuntimeError(
        "Section 5 validation failed:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in FAILED_SECTION_5_CHECKS
        )
    )


# --------------------------------------------------------------------------------------------------
# 26. Completion
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 5 — LOAD SPP-GAN ARCHITECTURE COMPLETE")
print("=" * 100)

print(
    "✓ Notebook 08 architecture summary loaded."
)

print(
    "✓ Notebook 08 parameter-count metadata loaded."
)

print(
    "✓ Notebook 08 model configuration loaded."
)

print(
    "✓ Notebook 08 architecture registry loaded."
)

print(
    "✓ Three canonical datasets validated."
)

print(
    "✓ Generative dimensions validated."
)

print(
    "✓ Transformed dimensions validated."
)

print(
    "✓ Generator and critic architecture validated."
)

print(
    "✓ Categorical Gumbel-Softmax contract validated."
)

print(
    "✓ Numerical activation inherited without override."
)

print(
    "✓ Statistical-guidance source Notebook 09 validated."
)

print(
    "✓ Notebook 03 statistical-guidance linkage validated."
)

print(
    "✓ Parameter-count cross-validation passed."
)

print(
    "✓ Native TRAIN / generative-schema linkage passed."
)

print(
    f"✓ Section 5 checks : "
    f"{len(SECTION_5_CHECKS)}/{len(SECTION_5_CHECKS)} PASS"
)

print()
print(
    "STATUS: PASS — SECTION 5 READY FOR FREEZE"
)

print("=" * 100)

5. LOAD SPP-GAN ARCHITECTURE
✓ Notebook 08 root : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08

----------------------------------------------------------------------------------------------------
NOTEBOOK 08 ARCHITECTURE ARTIFACT VERIFICATION
----------------------------------------------------------------------------------------------------
✓ architecture_summary     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/architecture/sppgan_architecture_summary.csv
✓ parameter_count          : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/metadata/sppgan_parameter_count.csv
✓ model_configuration      : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/config/sppgan_model_configuration.json
✓ architecture_registry    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/metadata/sppgan_architecture_artifacts.csv

----------------------------------------------------------------------

In [59]:
!pip install -q opacus==1.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 1.9 MB/s eta 0:00:00


In [60]:
# ==================================================================================================
# SECTION 6 — LOAD DP MECHANISM
# ==================================================================================================

print("=" * 100)
print("6. LOAD DP MECHANISM")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Verify Notebook 10 Root
# --------------------------------------------------------------------------------------------------

if "NB10_ROOT" not in globals():

    raise RuntimeError(
        "NB10_ROOT is not defined.\n"
        "Notebook 10 canonical root must be initialized before Section 6."
    )


NB10_ROOT = Path(
    NB10_ROOT
)


if not NB10_ROOT.exists():

    raise FileNotFoundError(
        f"Notebook 10 root does not exist:\n{NB10_ROOT}"
    )


if not NB10_ROOT.is_dir():

    raise NotADirectoryError(
        f"Notebook 10 root is not a directory:\n{NB10_ROOT}"
    )


print(
    f"✓ Notebook 10 root : {NB10_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 2. Canonical DP Module
# --------------------------------------------------------------------------------------------------

DP_MODULE_PATH = (
    NB10_ROOT
    / "models"
    / "sppgan_differential_privacy.py"
)


if not DP_MODULE_PATH.exists():

    raise FileNotFoundError(
        "Notebook 10 DP module not found:\n"
        f"{DP_MODULE_PATH}"
    )


if not DP_MODULE_PATH.is_file():

    raise RuntimeError(
        "Notebook 10 DP module is not a file:\n"
        f"{DP_MODULE_PATH}"
    )


if DP_MODULE_PATH.stat().st_size == 0:

    raise RuntimeError(
        "Notebook 10 DP module is empty:\n"
        f"{DP_MODULE_PATH}"
    )


print(
    f"✓ DP module path    : {DP_MODULE_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 3. Verify Required Runtime Dependency
# --------------------------------------------------------------------------------------------------

EXPECTED_OPACUS_VERSION = "1.6.0"


try:

    import opacus

except ModuleNotFoundError as exc:

    raise RuntimeError(
        "Opacus is not installed in the current runtime.\n\n"
        "Notebook 12 Section 6 requires the validated Notebook 10 "
        "DP environment.\n\n"
        f"Required Opacus version : {EXPECTED_OPACUS_VERSION}\n"
        "Observed                 : NOT INSTALLED\n\n"
        "Install the pinned dependency in the Notebook 00 "
        "environment/setup stage, restart the runtime if required, "
        "and rerun Notebook 12 from Section 1."
    ) from exc


INSTALLED_OPACUS_VERSION = getattr(
    opacus,
    "__version__",
    None,
)


if INSTALLED_OPACUS_VERSION is None:

    raise RuntimeError(
        "Unable to determine the installed Opacus version."
    )


if INSTALLED_OPACUS_VERSION != EXPECTED_OPACUS_VERSION:

    raise RuntimeError(
        "Opacus version mismatch.\n"
        f"Required validated version : {EXPECTED_OPACUS_VERSION}\n"
        f"Installed version           : {INSTALLED_OPACUS_VERSION}\n\n"
        "Do not continue with Notebook 12 until the runtime "
        "matches the validated Notebook 10 environment."
    )


print(
    f"✓ Opacus version   : {INSTALLED_OPACUS_VERSION}"
)


# --------------------------------------------------------------------------------------------------
# 4. Verify Opacus Core Components
# --------------------------------------------------------------------------------------------------

try:

    from opacus import PrivacyEngine

except ImportError as exc:

    raise RuntimeError(
        "Opacus PrivacyEngine could not be imported."
    ) from exc


try:

    from opacus.grad_sample import GradSampleModule

except ImportError as exc:

    raise RuntimeError(
        "Opacus GradSampleModule could not be imported."
    ) from exc


print(
    "✓ PrivacyEngine available."
)

print(
    "✓ GradSampleModule available."
)


# --------------------------------------------------------------------------------------------------
# 5. Dynamically Load Notebook 10 DP Module
# --------------------------------------------------------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "sppgan_differential_privacy_nb12",
    DP_MODULE_PATH,
)


if spec is None:

    raise RuntimeError(
        "Unable to create import specification for the Notebook 10 DP module."
    )


if spec.loader is None:

    raise RuntimeError(
        "Notebook 10 DP module import loader is unavailable."
    )


DP_MODULE = importlib.util.module_from_spec(
    spec
)


try:

    spec.loader.exec_module(
        DP_MODULE
    )

except Exception as exc:

    raise RuntimeError(
        "Notebook 10 DP module failed during import.\n"
        f"Module : {DP_MODULE_PATH}\n"
        f"Error  : {type(exc).__name__}: {exc}"
    ) from exc


print(
    "✓ Notebook 10 DP module imported successfully."
)


# --------------------------------------------------------------------------------------------------
# 6. Required DP Function Contract
# --------------------------------------------------------------------------------------------------

REQUIRED_DP_FUNCTIONS = [

    "wrap_critic_for_per_sample_gradients",

    "make_private_discriminator",

    "dp_discriminator_loss",

    "dp_discriminator_step",
]


missing_dp_functions = [

    name

    for name
    in REQUIRED_DP_FUNCTIONS

    if not hasattr(
        DP_MODULE,
        name,
    )
]


if missing_dp_functions:

    raise RuntimeError(
        "Notebook 10 DP module is incomplete.\n"
        "Missing required functions:\n"
        +
        "\n".join(
            f"  - {name}"
            for name
            in missing_dp_functions
        )
    )


print(
    "✓ Required DP function contract validated."
)


for function_name in REQUIRED_DP_FUNCTIONS:

    function_object = getattr(
        DP_MODULE,
        function_name,
    )


    if not callable(
        function_object
    ):

        raise RuntimeError(
            "Required DP component is not callable.\n"
            f"Function : {function_name}"
        )


print(
    "✓ All required DP functions are callable."
)


# --------------------------------------------------------------------------------------------------
# 7. Required DP Class / Component Contract
# --------------------------------------------------------------------------------------------------

if not hasattr(
    DP_MODULE,
    "SPPGANCritic",
):

    raise RuntimeError(
        "Notebook 10 DP module does not expose the required "
        "SPPGANCritic class."
    )


if not callable(
    getattr(
        DP_MODULE,
        "SPPGANCritic",
    )
):

    raise RuntimeError(
        "Notebook 10 SPPGANCritic is not callable."
    )


print(
    "✓ SPPGANCritic class available."
)


# --------------------------------------------------------------------------------------------------
# 8. Validate PrivacyEngine Compatibility
# --------------------------------------------------------------------------------------------------

try:

    privacy_engine_class = PrivacyEngine

    if privacy_engine_class is None:

        raise RuntimeError(
            "PrivacyEngine resolved to None."
        )

except Exception as exc:

    raise RuntimeError(
        "Opacus PrivacyEngine compatibility validation failed."
    ) from exc


print(
    "✓ Opacus PrivacyEngine compatibility validated."
)


# --------------------------------------------------------------------------------------------------
# 9. Validate Frozen Notebook 10 Privacy Contract
# --------------------------------------------------------------------------------------------------

EXPECTED_DP_CONTRACT = {

    "accountant":
        "rdp",

    "sampling":
        "poisson",

    "clipping":
        "flat",

    "loss_reduction":
        "mean",

    "protected_component":
        "discriminator",

    "generator_private":
        False,

    "statistical_guidance_private":
        False,

    "preprocessing_private":
        False,

    "end_to_end_privacy_claim":
        False,
}


if "ACCOUNTANT" in globals():

    if str(
        ACCOUNTANT
    ).strip().lower() != "rdp":

        raise RuntimeError(
            "Notebook 10 accountant mismatch.\n"
            f"Expected: rdp\n"
            f"Observed : {ACCOUNTANT}"
        )


if "SAMPLING_MECHANISM" in globals():

    if str(
        SAMPLING_MECHANISM
    ).strip().lower() != "poisson":

        raise RuntimeError(
            "Notebook 10 sampling mechanism mismatch.\n"
            f"Expected: poisson\n"
            f"Observed : {SAMPLING_MECHANISM}"
        )


if "CLIPPING_MECHANISM" in globals():

    if str(
        CLIPPING_MECHANISM
    ).strip().lower() != "flat":

        raise RuntimeError(
            "Notebook 10 clipping mechanism mismatch.\n"
            f"Expected: flat\n"
            f"Observed : {CLIPPING_MECHANISM}"
        )


if "LOSS_REDUCTION" in globals():

    if str(
        LOSS_REDUCTION
    ).strip().lower() != "mean":

        raise RuntimeError(
            "Notebook 10 loss reduction mismatch.\n"
            f"Expected: mean\n"
            f"Observed : {LOSS_REDUCTION}"
        )


print(
    "✓ Notebook 10 RDP / Poisson / flat-L2 / mean contract validated."
)


# --------------------------------------------------------------------------------------------------
# 10. DP Module Function Identity Validation
# --------------------------------------------------------------------------------------------------

DP_FUNCTION_REGISTRY = {

    function_name:
        getattr(
            DP_MODULE,
            function_name,
        )

    for function_name
    in REQUIRED_DP_FUNCTIONS
}


DP_CLASS_REGISTRY = {

    "SPPGANCritic":
        getattr(
            DP_MODULE,
            "SPPGANCritic",
        ),
}


# --------------------------------------------------------------------------------------------------
# 11. Final Section 6 Checks
# --------------------------------------------------------------------------------------------------

SECTION_6_CHECKS = {

    "nb10_root_exists":
        NB10_ROOT.exists(),

    "nb10_root_is_directory":
        NB10_ROOT.is_dir(),

    "dp_module_exists":
        DP_MODULE_PATH.exists(),

    "dp_module_is_file":
        DP_MODULE_PATH.is_file(),

    "dp_module_nonempty":
        DP_MODULE_PATH.stat().st_size > 0,

    "opacus_installed":
        True,

    "opacus_version_validated":
        INSTALLED_OPACUS_VERSION
        ==
        EXPECTED_OPACUS_VERSION,

    "privacy_engine_available":
        PrivacyEngine is not None,

    "grad_sample_module_available":
        GradSampleModule is not None,

    "required_dp_functions_present":
        not missing_dp_functions,

    "required_dp_functions_callable":
        all(
            callable(
                function
            )
            for function
            in DP_FUNCTION_REGISTRY.values()
        ),

    "sppgan_critic_available":
        callable(
            DP_CLASS_REGISTRY[
                "SPPGANCritic"
            ]
        ),

    "dp_module_imported":
        DP_MODULE is not None,

    "rdp_contract":
        (
            "ACCOUNTANT" not in globals()
            or
            str(
                ACCOUNTANT
            ).strip().lower()
            ==
            "rdp"
        ),

    "poisson_contract":
        (
            "SAMPLING_MECHANISM" not in globals()
            or
            str(
                SAMPLING_MECHANISM
            ).strip().lower()
            ==
            "poisson"
        ),

    "flat_clipping_contract":
        (
            "CLIPPING_MECHANISM" not in globals()
            or
            str(
                CLIPPING_MECHANISM
            ).strip().lower()
            ==
            "flat"
        ),

    "mean_loss_reduction_contract":
        (
            "LOSS_REDUCTION" not in globals()
            or
            str(
                LOSS_REDUCTION
            ).strip().lower()
            ==
            "mean"
        ),
}


FAILED_SECTION_6_CHECKS = [

    name

    for name, passed
    in SECTION_6_CHECKS.items()

    if not passed
]


print()
print("-" * 100)
print("SECTION 6 VALIDATION")
print("-" * 100)


for check_name, passed in SECTION_6_CHECKS.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )


if FAILED_SECTION_6_CHECKS:

    raise RuntimeError(
        "Section 6 validation failed:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in FAILED_SECTION_6_CHECKS
        )
    )


# --------------------------------------------------------------------------------------------------
# 12. Completion
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 6 — LOAD DP MECHANISM COMPLETE")
print("=" * 100)

print(
    "✓ Notebook 10 DP module loaded."
)

print(
    f"✓ Opacus version validated : "
    f"{INSTALLED_OPACUS_VERSION}"
)

print(
    "✓ PrivacyEngine available."
)

print(
    "✓ GradSampleModule available."
)

print(
    "✓ SPPGANCritic available."
)

print(
    "✓ Per-example gradient wrapper available."
)

print(
    "✓ DP discriminator construction available."
)

print(
    "✓ DP discriminator loss available."
)

print(
    "✓ DP discriminator step available."
)

print(
    "✓ RDP / Poisson / flat-L2 / mean contract validated."
)

print(
    f"✓ Section 6 checks : "
    f"{len(SECTION_6_CHECKS)}/{len(SECTION_6_CHECKS)} PASS"
)

print()
print(
    "STATUS: PASS — SECTION 6 READY FOR FREEZE"
)

print("=" * 100)

6. LOAD DP MECHANISM
✓ Notebook 10 root : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10
✓ DP module path    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/models/sppgan_differential_privacy.py
✓ Opacus version   : 1.6.0
✓ PrivacyEngine available.
✓ GradSampleModule available.
✓ Notebook 10 DP module imported successfully.
✓ Required DP function contract validated.
✓ All required DP functions are callable.
✓ SPPGANCritic class available.
✓ Opacus PrivacyEngine compatibility validated.
✓ Notebook 10 RDP / Poisson / flat-L2 / mean contract validated.

----------------------------------------------------------------------------------------------------
SECTION 6 VALIDATION
----------------------------------------------------------------------------------------------------
✓ nb10_root_exists
✓ nb10_root_is_directory
✓ dp_module_exists
✓ dp_module_is_file
✓ dp_module_nonempty
✓ opacus_installed
✓ opacus_version_validated
✓ privacy_engine_availa

In [64]:
# ==================================================================================================
# 7. VALIDATE ALL INPUTS
# ==================================================================================================

print("=" * 100)
print("7. VALIDATE ALL INPUTS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Required Imports
# --------------------------------------------------------------------------------------------------

import json
import hashlib
import joblib
import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 2. Canonical Dataset Registry
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]


if list(DATASET_IDS) != EXPECTED_DATASET_IDS:

    raise RuntimeError(
        "Canonical dataset registry mismatch.\n"
        f"Expected : {EXPECTED_DATASET_IDS}\n"
        f"Observed : {list(DATASET_IDS)}"
    )


print(
    "✓ Dataset registry validated."
)


# --------------------------------------------------------------------------------------------------
# 3. Canonical Dataset Contracts
# --------------------------------------------------------------------------------------------------

EXPECTED_TRAIN_ROWS = {
    "adult_income": 34189,
    "bank_marketing": 31647,
    "diabetes_130us": 71236,
}


EXPECTED_NATIVE_COLUMNS = {
    "adult_income": 16,
    "bank_marketing": 18,
    "diabetes_130us": 49,
}


EXPECTED_TRANSFORMED_DIMS = {
    "adult_income": 105,
    "bank_marketing": 51,
    "diabetes_130us": 2329,
}


EXPECTED_GENERATIVE_DIMS = {
    "adult_income": 15,
    "bank_marketing": 17,
    "diabetes_130us": 48,
}


EXPECTED_NUMERICAL_FEATURES = {
    "adult_income": 6,
    "bank_marketing": 7,
    "diabetes_130us": 11,
}


EXPECTED_CATEGORICAL_FEATURES = {
    "adult_income": 8,
    "bank_marketing": 9,
    "diabetes_130us": 36,
}


EXPECTED_TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}


EXPECTED_PROVENANCE_COLUMNS = {
    "adult_income": "__original_row_id__",
    "bank_marketing": "__original_row_id__",
    "diabetes_130us": "__original_row_id__",
}


# --------------------------------------------------------------------------------------------------
# 4. Verify Notebook 02 Canonical Artifact Layer
# --------------------------------------------------------------------------------------------------

if "PROJECT_ROOT" not in globals():

    raise RuntimeError(
        "PROJECT_ROOT is not available.\n"
        "Run Notebook 12 Section 2 before Section 7."
    )


NB02_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_02"
)


NB02_NATIVE_ROOT = (
    NB02_ROOT
    / "native"
)


NB02_SCHEMA_ROOT = (
    NB02_ROOT
    / "schemas"
)


NB02_METADATA_ROOT = (
    NB02_SCHEMA_ROOT
    / "metadata"
)


NB02_NATIVE_MANIFEST = (
    NB02_NATIVE_ROOT
    / "native_dataset_manifest.csv"
)


NB02_PREPROCESSOR_MANIFEST = (
    NB02_SCHEMA_ROOT
    / "preprocessor_manifest.csv"
)


REQUIRED_NB02_PATHS = {

    "Notebook 02 root":
        NB02_ROOT,

    "Native root":
        NB02_NATIVE_ROOT,

    "Schema root":
        NB02_SCHEMA_ROOT,

    "Metadata root":
        NB02_METADATA_ROOT,

    "Native manifest":
        NB02_NATIVE_MANIFEST,

    "Preprocessor manifest":
        NB02_PREPROCESSOR_MANIFEST,
}


for label, path in REQUIRED_NB02_PATHS.items():

    if not path.exists():

        raise FileNotFoundError(
            f"Required Notebook 02 artifact is missing.\n"
            f"{label} : {path}"
        )


print(
    f"✓ Notebook 02 root     : {NB02_ROOT}"
)

print(
    f"✓ Native manifest      : {NB02_NATIVE_MANIFEST}"
)

print(
    f"✓ Preprocessor manifest: {NB02_PREPROCESSOR_MANIFEST}"
)


# --------------------------------------------------------------------------------------------------
# 5. Load Native Dataset Manifest
# --------------------------------------------------------------------------------------------------

NATIVE_MANIFEST_DF = pd.read_csv(
    NB02_NATIVE_MANIFEST,
    low_memory=False,
)


if NATIVE_MANIFEST_DF.empty:

    raise RuntimeError(
        "Notebook 02 native_dataset_manifest.csv is empty."
    )


REQUIRED_MANIFEST_COLUMNS = [
    "dataset_id",
    "split",
    "absolute_path",
    "rows",
    "columns",
    "generative_columns",
    "target_column",
    "provenance_column",
    "identifier_columns",
    "provenance_present",
    "target_present",
    "identifiers_excluded",
    "file_exists",
    "reload_validation",
    "status",
]


missing_manifest_columns = [
    column
    for column in REQUIRED_MANIFEST_COLUMNS
    if column not in NATIVE_MANIFEST_DF.columns
]


if missing_manifest_columns:

    raise RuntimeError(
        "Notebook 02 native manifest is missing required columns:\n"
        f"{missing_manifest_columns}"
    )


print(
    f"✓ Native manifest loaded : "
    f"{len(NATIVE_MANIFEST_DF)} rows"
)

print(
    "✓ Native manifest schema validated."
)


# --------------------------------------------------------------------------------------------------
# 6. Resolve Exactly One Validated TRAIN Record Per Dataset
# --------------------------------------------------------------------------------------------------

TRAIN_MANIFEST_RECORDS = {}


for dataset_id in DATASET_IDS:

    matches = NATIVE_MANIFEST_DF[
        (
            NATIVE_MANIFEST_DF["dataset_id"]
            .astype(str)
            .str.strip()
            ==
            dataset_id
        )
        &
        (
            NATIVE_MANIFEST_DF["split"]
            .astype(str)
            .str.strip()
            .str.lower()
            ==
            "train"
        )
    ]


    if len(matches) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one Notebook 02 "
            f"TRAIN manifest record, found {len(matches)}."
        )


    record = matches.iloc[0].to_dict()


    if str(
        record["status"]
    ).strip().upper() != "PASS":

        raise RuntimeError(
            f"{dataset_id}: Notebook 02 TRAIN manifest status "
            "is not PASS."
        )


    if not bool(
        record["file_exists"]
    ):

        raise FileNotFoundError(
            f"{dataset_id}: Notebook 02 TRAIN file is marked unavailable."
        )


    if not bool(
        record["reload_validation"]
    ):

        raise RuntimeError(
            f"{dataset_id}: Notebook 02 TRAIN reload validation "
            "did not PASS."
        )


    TRAIN_MANIFEST_RECORDS[
        dataset_id
    ] = record


print(
    "✓ Exactly one validated TRAIN manifest record resolved for each dataset."
)


# --------------------------------------------------------------------------------------------------
# 7. Load Authoritative Native TRAIN Data
# --------------------------------------------------------------------------------------------------

TRAINING_DATA = {}


TRAINING_SOURCE_PATHS = {}


TRAINING_SOURCE_HASHES = {}


for dataset_id in DATASET_IDS:

    record = TRAIN_MANIFEST_RECORDS[
        dataset_id
    ]


    train_path = Path(
        str(
            record["absolute_path"]
        )
    )


    if not train_path.exists():

        raise FileNotFoundError(
            f"{dataset_id}: authoritative Notebook 02 TRAIN "
            f"file does not exist:\n{train_path}"
        )


    if not train_path.is_file():

        raise RuntimeError(
            f"{dataset_id}: TRAIN path is not a file:\n{train_path}"
        )


    train_df = pd.read_csv(
        train_path,
        low_memory=False,
    )


    expected_rows = EXPECTED_TRAIN_ROWS[
        dataset_id
    ]


    expected_columns = EXPECTED_NATIVE_COLUMNS[
        dataset_id
    ]


    if len(train_df) != expected_rows:

        raise RuntimeError(
            f"{dataset_id}: TRAIN row-count mismatch.\n"
            f"Expected : {expected_rows:,}\n"
            f"Observed : {len(train_df):,}"
        )


    if train_df.shape[1] != expected_columns:

        raise RuntimeError(
            f"{dataset_id}: native TRAIN column-count mismatch.\n"
            f"Expected : {expected_columns}\n"
            f"Observed : {train_df.shape[1]}"
        )


    if train_df.empty:

        raise RuntimeError(
            f"{dataset_id}: native TRAIN dataframe is empty."
        )


    if train_df.columns.duplicated().any():

        duplicated_columns = (
            train_df.columns[
                train_df.columns.duplicated()
            ]
            .tolist()
        )

        raise RuntimeError(
            f"{dataset_id}: duplicate TRAIN columns detected:\n"
            f"{duplicated_columns}"
        )


    target_column = EXPECTED_TARGET_COLUMNS[
        dataset_id
    ]


    provenance_column = EXPECTED_PROVENANCE_COLUMNS[
        dataset_id
    ]


    if target_column not in train_df.columns:

        raise RuntimeError(
            f"{dataset_id}: target column '{target_column}' "
            "is missing from TRAIN data."
        )


    if provenance_column not in train_df.columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column "
            f"'{provenance_column}' is missing from TRAIN data."
        )


    # ----------------------------------------------------------------------------------------------
    # Manifest target / provenance validation
    # ----------------------------------------------------------------------------------------------

    manifest_target = str(
        record["target_column"]
    ).strip()


    if manifest_target != target_column:

        raise RuntimeError(
            f"{dataset_id}: manifest target mismatch.\n"
            f"Expected : {target_column}\n"
            f"Observed : {manifest_target}"
        )


    manifest_provenance = str(
        record["provenance_column"]
    ).strip()


    if manifest_provenance != provenance_column:

        raise RuntimeError(
            f"{dataset_id}: manifest provenance mismatch.\n"
            f"Expected : {provenance_column}\n"
            f"Observed : {manifest_provenance}"
        )


    # ----------------------------------------------------------------------------------------------
    # Generative schema
    #
    # Notebook 02 authority:
    #
    # native columns = provenance + generative columns
    #
    # generative columns = preprocessing columns + target
    # ----------------------------------------------------------------------------------------------

    native_without_provenance = [
        column
        for column in train_df.columns
        if column != provenance_column
    ]


    if len(native_without_provenance) != EXPECTED_GENERATIVE_DIMS[
        dataset_id
    ]:

        raise RuntimeError(
            f"{dataset_id}: native generative dimension mismatch.\n"
            f"Expected : "
            f"{EXPECTED_GENERATIVE_DIMS[dataset_id]}\n"
            f"Observed : {len(native_without_provenance)}"
        )


    if target_column not in native_without_provenance:

        raise RuntimeError(
            f"{dataset_id}: target is absent from generative schema."
        )


    if native_without_provenance[-1] != target_column:

        raise RuntimeError(
            f"{dataset_id}: target is not the final generative column.\n"
            f"Expected final column : {target_column}\n"
            f"Observed final column : {native_without_provenance[-1]}"
        )


    # ----------------------------------------------------------------------------------------------
    # Manifest row/column validation
    # ----------------------------------------------------------------------------------------------

    if int(
        record["rows"]
    ) != len(train_df):

        raise RuntimeError(
            f"{dataset_id}: manifest TRAIN row count does not "
            "match loaded data."
        )


    if int(
        record["columns"]
    ) != train_df.shape[1]:

        raise RuntimeError(
            f"{dataset_id}: manifest native column count does not "
            "match loaded data."
        )


    # ----------------------------------------------------------------------------------------------
    # SHA-256 integrity
    # ----------------------------------------------------------------------------------------------

    sha256 = hashlib.sha256()


    with open(
        train_path,
        "rb",
    ) as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):

            sha256.update(
                chunk
            )


    actual_sha256 = sha256.hexdigest()


    manifest_hash = str(
        record.get(
            "sha256",
            ""
        )
    ).strip()


    if manifest_hash:

        if actual_sha256.lower() != manifest_hash.lower():

            raise RuntimeError(
                f"{dataset_id}: TRAIN SHA-256 mismatch.\n"
                f"Manifest: {manifest_hash}\n"
                f"Actual  : {actual_sha256}"
            )


    TRAINING_DATA[
        dataset_id
    ] = train_df


    TRAINING_SOURCE_PATHS[
        dataset_id
    ] = str(
        train_path
    )


    TRAINING_SOURCE_HASHES[
        dataset_id
    ] = actual_sha256


    print(
        f"✓ {dataset_id:<18} "
        f"TRAIN loaded : "
        f"{train_df.shape}"
    )


print(
    "✓ Authoritative Notebook 02 native TRAIN data loaded."
)

print(
    "✓ TRAIN-only data policy preserved."
)


# --------------------------------------------------------------------------------------------------
# 8. Load Notebook 02 Preprocessing Metadata
# --------------------------------------------------------------------------------------------------

PREPROCESSING_METADATA = {}


REQUIRED_PREPROCESSING_METADATA_FIELDS = [

    "dataset_id",
    "training_rows",
    "input_columns",
    "numeric_columns",
    "categorical_columns",
    "target_column",
    "target_retained_in_training_dataset",
    "target_excluded_from_preprocessor_input",
    "identifier_columns",
    "provenance_column",
    "provenance_excluded_from_modeling",
    "identifiers_excluded_from_modeling",
    "generative_columns",
    "fit_dataset",
    "fit_scope",
    "fit_policy",
    "preprocessor_artifact",
    "schema_artifact",
    "preprocessing_feature_count",
    "generative_column_count",
    "target_retained_in_generative_schema",
    "target_excluded_from_transformed_features",
    "raw_target_manually_appended",
    "identifiers_excluded",
    "provenance_excluded_from_model_input",
    "transformed_feature_count",
    "transformed_feature_names",
]


for dataset_id in DATASET_IDS:

    metadata_path = (
        NB02_METADATA_ROOT
        / f"{dataset_id}_preprocessing_metadata.json"
    )


    if not metadata_path.exists():

        raise FileNotFoundError(
            f"{dataset_id}: Notebook 02 preprocessing metadata "
            f"not found:\n{metadata_path}"
        )


    with open(
        metadata_path,
        "r",
        encoding="utf-8",
    ) as handle:

        metadata = json.load(
            handle
        )


    if not isinstance(
        metadata,
        dict,
    ):

        raise RuntimeError(
            f"{dataset_id}: preprocessing metadata is not a JSON object."
        )


    missing_fields = [
        field
        for field in REQUIRED_PREPROCESSING_METADATA_FIELDS
        if field not in metadata
    ]


    if missing_fields:

        raise RuntimeError(
            f"{dataset_id}: preprocessing metadata is missing:\n"
            f"{missing_fields}"
        )


    if metadata["dataset_id"] != dataset_id:

        raise RuntimeError(
            f"{dataset_id}: metadata dataset identity mismatch."
        )


    if int(
        metadata["training_rows"]
    ) != EXPECTED_TRAIN_ROWS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: metadata training-row mismatch."
        )


    if metadata["target_column"] != EXPECTED_TARGET_COLUMNS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: metadata target mismatch."
        )


    if metadata["provenance_column"] != EXPECTED_PROVENANCE_COLUMNS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: metadata provenance mismatch."
        )


    if int(
        metadata["generative_column_count"]
    ) != EXPECTED_GENERATIVE_DIMS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: metadata generative dimension mismatch."
        )


    if int(
        metadata["transformed_feature_count"]
    ) != EXPECTED_TRANSFORMED_DIMS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: metadata transformed dimension mismatch."
        )


    expected_preprocessing_count = (
        EXPECTED_NUMERICAL_FEATURES[dataset_id]
        +
        EXPECTED_CATEGORICAL_FEATURES[dataset_id]
    )


    if int(
        metadata["preprocessing_feature_count"]
    ) != expected_preprocessing_count:

        raise RuntimeError(
            f"{dataset_id}: preprocessing feature count mismatch.\n"
            f"Expected : {expected_preprocessing_count}\n"
            f"Observed : {metadata['preprocessing_feature_count']}"
        )


    PREPROCESSING_METADATA[
        dataset_id
    ] = metadata


    print(
        f"✓ {dataset_id:<18} "
        f"Notebook 02 preprocessing metadata validated."
    )


print(
    "✓ Notebook 02 preprocessing metadata validated."
)


# --------------------------------------------------------------------------------------------------
# 9. Resolve Authoritative Preprocessing Columns
#
# IMPORTANT:
#
# The preprocessing input is:
#
#     numeric columns + categorical columns
#
# The target is NOT part of the preprocessor input.
#
# The generative schema is:
#
#     preprocessing columns + target
#
# Therefore we explicitly construct the preprocessing-column contract from
# Notebook 02 numeric/categorical metadata instead of comparing it directly
# with the full generative schema.
# --------------------------------------------------------------------------------------------------

PREPROCESSING_COLUMNS = {}


GENERATIVE_COLUMNS = {}


for dataset_id in DATASET_IDS:

    train_df = TRAINING_DATA[
        dataset_id
    ]

    metadata = PREPROCESSING_METADATA[
        dataset_id
    ]


    numeric_columns = list(
        metadata["numeric_columns"]
    )


    categorical_columns = list(
        metadata["categorical_columns"]
    )


    target_column = metadata[
        "target_column"
    ]


    provenance_column = metadata[
        "provenance_column"
    ]


    preprocessing_columns = (
        numeric_columns
        +
        categorical_columns
    )


    generative_columns = list(
        metadata["generative_columns"]
    )


    # ----------------------------------------------------------------------------------------------
    # Numeric/categorical partition
    # ----------------------------------------------------------------------------------------------

    if set(numeric_columns).intersection(
        categorical_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: numeric and categorical columns overlap."
        )


    if len(preprocessing_columns) != (
        EXPECTED_NUMERICAL_FEATURES[dataset_id]
        +
        EXPECTED_CATEGORICAL_FEATURES[dataset_id]
    ):

        raise RuntimeError(
            f"{dataset_id}: preprocessing-column count mismatch."
        )


    if set(preprocessing_columns) != (
        set(generative_columns)
        -
        {target_column}
    ):

        raise RuntimeError(
            f"{dataset_id}: preprocessing columns do not exactly "
            "match generative columns excluding target."
        )


    # ----------------------------------------------------------------------------------------------
    # Ordered generative schema
    # ----------------------------------------------------------------------------------------------

    expected_generative_columns = (
        preprocessing_columns
        +
        [target_column]
    )


    if generative_columns != expected_generative_columns:

        raise RuntimeError(
            f"{dataset_id}: generative column order is inconsistent "
            "with preprocessing columns + target.\n"
            f"Expected: {expected_generative_columns}\n"
            f"Observed: {generative_columns}"
        )


    # ----------------------------------------------------------------------------------------------
    # Native schema
    # ----------------------------------------------------------------------------------------------

    expected_native_columns = (
        [provenance_column]
        +
        generative_columns
    )


    if list(train_df.columns) != expected_native_columns:

        raise RuntimeError(
            f"{dataset_id}: native column order/schema mismatch.\n"
            f"Expected: {expected_native_columns}\n"
            f"Observed: {list(train_df.columns)}"
        )


    # ----------------------------------------------------------------------------------------------
    # Target policy
    # ----------------------------------------------------------------------------------------------

    if target_column in preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' incorrectly "
            "appears in preprocessing input."
        )


    if target_column not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' is not retained "
            "in generative schema."
        )


    # ----------------------------------------------------------------------------------------------
    # Provenance policy
    # ----------------------------------------------------------------------------------------------

    if provenance_column in preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column '{provenance_column}' "
            "incorrectly appears in preprocessing input."
        )


    if provenance_column in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column '{provenance_column}' "
            "incorrectly appears in generative schema."
        )


    # ----------------------------------------------------------------------------------------------
    # Identifier policy
    # ----------------------------------------------------------------------------------------------

    identifier_columns = list(
        metadata["identifier_columns"]
    )


    identifier_overlap = (
        set(identifier_columns)
        .intersection(
            generative_columns
        )
    )


    if identifier_overlap:

        raise RuntimeError(
            f"{dataset_id}: excluded identifier columns appear "
            f"in generative schema:\n{sorted(identifier_overlap)}"
        )


    PREPROCESSING_COLUMNS[
        dataset_id
    ] = preprocessing_columns


    GENERATIVE_COLUMNS[
        dataset_id
    ] = generative_columns


    print(
        f"✓ {dataset_id:<18} "
        f"preprocessing={len(preprocessing_columns):>2} | "
        f"generative={len(generative_columns):>2} | "
        f"target={target_column}"
    )


print(
    "✓ Notebook 02 preprocessing/generative schema contract validated."
)


# --------------------------------------------------------------------------------------------------
# 10. Load Frozen Notebook 02 Preprocessors
# --------------------------------------------------------------------------------------------------

FITTED_PREPROCESSORS = {}


for dataset_id in DATASET_IDS:

    metadata = PREPROCESSING_METADATA[
        dataset_id
    ]


    preprocessor_path = Path(
        metadata["preprocessor_artifact"]
    )


    if not preprocessor_path.exists():

        raise FileNotFoundError(
            f"{dataset_id}: fitted Notebook 02 preprocessor "
            f"not found:\n{preprocessor_path}"
        )


    preprocessor = joblib.load(
        preprocessor_path
    )


    if preprocessor is None:

        raise RuntimeError(
            f"{dataset_id}: loaded preprocessor is None."
        )


    if not hasattr(
        preprocessor,
        "transform",
    ):

        raise RuntimeError(
            f"{dataset_id}: fitted preprocessor does not expose transform()."
        )


    FITTED_PREPROCESSORS[
        dataset_id
    ] = preprocessor


    print(
        f"✓ {dataset_id:<18} "
        f"fitted preprocessor loaded."
    )


print(
    "✓ Frozen Notebook 02 preprocessors loaded."
)


# --------------------------------------------------------------------------------------------------
# 11. Transform TRAIN Using Frozen Notebook 02 Preprocessors
# --------------------------------------------------------------------------------------------------

TRAIN_ARRAYS = {}


TRANSFORMED_TRAIN_SHAPES = {}


for dataset_id in DATASET_IDS:

    train_df = TRAINING_DATA[
        dataset_id
    ]

    preprocessing_columns = PREPROCESSING_COLUMNS[
        dataset_id
    ]

    preprocessor = FITTED_PREPROCESSORS[
        dataset_id
    ]


    X_train_native = train_df[
        preprocessing_columns
    ].copy()


    transformed = preprocessor.transform(
        X_train_native
    )


    if hasattr(
        transformed,
        "toarray",
    ):

        transformed = transformed.toarray()


    transformed = np.asarray(
        transformed,
        dtype=np.float32,
    )


    if transformed.ndim != 2:

        raise RuntimeError(
            f"{dataset_id}: transformed TRAIN data must be 2-dimensional."
        )


    expected_shape = (
        EXPECTED_TRAIN_ROWS[dataset_id],
        EXPECTED_TRANSFORMED_DIMS[dataset_id],
    )


    if transformed.shape != expected_shape:

        raise RuntimeError(
            f"{dataset_id}: transformed TRAIN shape mismatch.\n"
            f"Expected : {expected_shape}\n"
            f"Observed : {transformed.shape}"
        )


    if not np.isfinite(
        transformed
    ).all():

        raise RuntimeError(
            f"{dataset_id}: transformed TRAIN contains "
            "NaN or Inf values."
        )


    TRAIN_ARRAYS[
        dataset_id
    ] = transformed


    TRANSFORMED_TRAIN_SHAPES[
        dataset_id
    ] = tuple(
        transformed.shape
    )


    print(
        f"✓ {dataset_id:<18} "
        f"transformed TRAIN : "
        f"{transformed.shape}"
    )


print(
    "✓ TRAIN_ARRAYS constructed from frozen Notebook 02 preprocessors."
)


# --------------------------------------------------------------------------------------------------
# 12. Validate Statistical Guidance
# --------------------------------------------------------------------------------------------------

if "STATISTICAL_GUIDANCE" not in globals():

    raise RuntimeError(
        "STATISTICAL_GUIDANCE is not available.\n"
        "Run Notebook 12 Section 4 before Section 7."
    )


missing_guidance = [
    dataset_id
    for dataset_id in DATASET_IDS
    if dataset_id not in STATISTICAL_GUIDANCE
]


if missing_guidance:

    raise RuntimeError(
        "Statistical guidance is missing for:\n"
        f"{missing_guidance}"
    )


print(
    "✓ Statistical guidance validated."
)


# --------------------------------------------------------------------------------------------------
# 13. Validate Notebook 08 Architecture Dimensions
# --------------------------------------------------------------------------------------------------

if "SPPGAN_ARCHITECTURE" not in globals():

    raise RuntimeError(
        "SPPGAN_ARCHITECTURE is not available.\n"
        "Run Notebook 12 Section 5 before Section 7."
    )


for dataset_id in DATASET_IDS:

    architecture = SPPGAN_ARCHITECTURE[
        dataset_id
    ]


    if int(
        architecture["transformed_dimension"]
    ) != EXPECTED_TRANSFORMED_DIMS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: Notebook 08 transformed dimension mismatch."
        )


    if int(
        architecture["generative_dimension"]
    ) != EXPECTED_GENERATIVE_DIMS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: Notebook 08 generative dimension mismatch."
        )


    if int(
        architecture["numerical_features"]
    ) != EXPECTED_NUMERICAL_FEATURES[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: Notebook 08 numerical feature count mismatch."
        )


    if int(
        architecture["categorical_features"]
    ) != EXPECTED_CATEGORICAL_FEATURES[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: Notebook 08 categorical feature count mismatch."
        )


    if int(
        architecture["latent_dim"]
    ) != 128:

        raise RuntimeError(
            f"{dataset_id}: latent dimension mismatch."
        )


print(
    "✓ Notebook 08 architecture dimensions validated."
)


# --------------------------------------------------------------------------------------------------
# 14. Validate DP Parameters
# --------------------------------------------------------------------------------------------------

if TARGET_EPSILON <= 0:

    raise ValueError(
        "Target epsilon must be positive."
    )


if MAX_GRAD_NORM <= 0:

    raise ValueError(
        "Maximum gradient norm must be positive."
    )


if DP_BATCH_SIZE <= 0:

    raise ValueError(
        "DP batch size must be positive."
    )


if DP_EPOCHS <= 0:

    raise ValueError(
        "DP epochs must be positive."
    )


if ACCOUNTANT.lower() != "rdp":

    raise ValueError(
        "Notebook 12 requires RDP accounting."
    )


if SAMPLING_MECHANISM.lower() != "poisson":

    raise ValueError(
        "Notebook 12 requires Poisson sampling."
    )


if CLIPPING_MECHANISM.lower() != "flat":

    raise ValueError(
        "Notebook 12 requires flat L2 clipping."
    )


if LOSS_REDUCTION.lower() != "mean":

    raise ValueError(
        "Notebook 12 requires mean loss reduction."
    )


print(
    "✓ DP parameters validated."
)


# --------------------------------------------------------------------------------------------------
# 15. Validate Privacy Boundary
# --------------------------------------------------------------------------------------------------

if GENERATOR_PRIVATE is not False:

    raise RuntimeError(
        "Generator privacy boundary mismatch."
    )


if STATISTICAL_GUIDANCE_PRIVATE is not False:

    raise RuntimeError(
        "Statistical-guidance privacy boundary mismatch."
    )


if PREPROCESSING_PRIVATE is not False:

    raise RuntimeError(
        "Preprocessing privacy boundary mismatch."
    )


if END_TO_END_PRIVACY_CLAIM is not False:

    raise RuntimeError(
        "End-to-end privacy claim must remain False."
    )


print(
    "✓ Privacy boundary validated."
)


# --------------------------------------------------------------------------------------------------
# 16. Final Validation Registry
# --------------------------------------------------------------------------------------------------

SECTION_7_CHECKS = {

    "dataset_registry":
        list(DATASET_IDS)
        ==
        EXPECTED_DATASET_IDS,

    "native_manifest_loaded":
        not NATIVE_MANIFEST_DF.empty,

    "train_manifest_records":
        len(TRAIN_MANIFEST_RECORDS)
        ==
        len(DATASET_IDS),

    "training_data_loaded":
        len(TRAINING_DATA)
        ==
        len(DATASET_IDS),

    "training_rows":
        all(
            len(TRAINING_DATA[dataset_id])
            ==
            EXPECTED_TRAIN_ROWS[dataset_id]
            for dataset_id in DATASET_IDS
        ),

    "native_training_columns":
        all(
            TRAINING_DATA[dataset_id].shape[1]
            ==
            EXPECTED_NATIVE_COLUMNS[dataset_id]
            for dataset_id in DATASET_IDS
        ),

    "preprocessing_metadata":
        len(PREPROCESSING_METADATA)
        ==
        len(DATASET_IDS),

    "preprocessing_schema":
        len(PREPROCESSING_COLUMNS)
        ==
        len(DATASET_IDS),

    "generative_schema":
        len(GENERATIVE_COLUMNS)
        ==
        len(DATASET_IDS),

    "fitted_preprocessors":
        len(FITTED_PREPROCESSORS)
        ==
        len(DATASET_IDS),

    "transformed_arrays":
        len(TRAIN_ARRAYS)
        ==
        len(DATASET_IDS),

    "transformed_dimensions":
        all(
            TRAIN_ARRAYS[dataset_id].shape[1]
            ==
            EXPECTED_TRANSFORMED_DIMS[dataset_id]
            for dataset_id in DATASET_IDS
        ),

    "transformed_row_counts":
        all(
            TRAIN_ARRAYS[dataset_id].shape[0]
            ==
            EXPECTED_TRAIN_ROWS[dataset_id]
            for dataset_id in DATASET_IDS
        ),

    "transformed_finite":
        all(
            np.isfinite(
                TRAIN_ARRAYS[dataset_id]
            ).all()
            for dataset_id in DATASET_IDS
        ),

    "statistical_guidance":
        all(
            dataset_id in STATISTICAL_GUIDANCE
            for dataset_id in DATASET_IDS
        ),

    "architecture_dimensions":
        all(
            int(
                SPPGAN_ARCHITECTURE[
                    dataset_id
                ]["transformed_dimension"]
            )
            ==
            EXPECTED_TRANSFORMED_DIMS[dataset_id]
            for dataset_id in DATASET_IDS
        ),

    "rdp_accountant":
        ACCOUNTANT.lower()
        ==
        "rdp",

    "poisson_sampling":
        SAMPLING_MECHANISM.lower()
        ==
        "poisson",

    "flat_clipping":
        CLIPPING_MECHANISM.lower()
        ==
        "flat",

    "mean_loss_reduction":
        LOSS_REDUCTION.lower()
        ==
        "mean",

    "generator_not_private":
        GENERATOR_PRIVATE is False,

    "statistical_guidance_not_private":
        STATISTICAL_GUIDANCE_PRIVATE is False,

    "preprocessing_not_private":
        PREPROCESSING_PRIVATE is False,

    "end_to_end_privacy_not_claimed":
        END_TO_END_PRIVACY_CLAIM is False,
}


FAILED_SECTION_7_CHECKS = [
    name
    for name, passed
    in SECTION_7_CHECKS.items()
    if not passed
]


# --------------------------------------------------------------------------------------------------
# 17. Validation Output
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("SECTION 7 VALIDATION")
print("-" * 100)


for check_name, passed in SECTION_7_CHECKS.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )


if FAILED_SECTION_7_CHECKS:

    raise RuntimeError(
        "Section 7 validation failed:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in FAILED_SECTION_7_CHECKS
        )
    )


# --------------------------------------------------------------------------------------------------
# 18. Completion
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 7 — VALIDATE ALL INPUTS COMPLETE")
print("=" * 100)

print(
    "✓ Authoritative Notebook 02 TRAIN manifest validated."
)

print(
    "✓ Native TRAIN datasets loaded from persisted artifacts."
)

print(
    "✓ TRAIN row counts and schemas validated."
)

print(
    "✓ TRAIN SHA-256 integrity validated."
)

print(
    "✓ Notebook 02 preprocessing metadata validated."
)

print(
    "✓ Preprocessing columns separated correctly from retained target."
)

print(
    "✓ Native generative schema validated."
)

print(
    "✓ Frozen Notebook 02 preprocessors loaded."
)

print(
    "✓ TRAIN_ARRAYS constructed using frozen Notebook 02 preprocessors."
)

print(
    "✓ Transformed dimensions cross-validated against Notebook 08."
)

print(
    "✓ Statistical guidance validated."
)

print(
    "✓ DP parameters validated."
)

print(
    "✓ Privacy boundary validated."
)

print(
    f"✓ Section 7 checks : "
    f"{len(SECTION_7_CHECKS)}/{len(SECTION_7_CHECKS)} PASS"
)

print()
print(
    "STATUS: PASS — SECTION 7 READY FOR FREEZE"
)

print("=" * 100)

7. VALIDATE ALL INPUTS
✓ Dataset registry validated.
✓ Notebook 02 root     : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✓ Native manifest      : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/native_dataset_manifest.csv
✓ Preprocessor manifest: /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/preprocessor_manifest.csv
✓ Native manifest loaded : 9 rows
✓ Native manifest schema validated.
✓ Exactly one validated TRAIN manifest record resolved for each dataset.
✓ adult_income       TRAIN loaded : (34189, 16)
✓ bank_marketing     TRAIN loaded : (31647, 18)
✓ diabetes_130us     TRAIN loaded : (71236, 49)
✓ Authoritative Notebook 02 native TRAIN data loaded.
✓ TRAIN-only data policy preserved.
✓ adult_income       Notebook 02 preprocessing metadata validated.
✓ bank_marketing     Notebook 02 preprocessing metadata validated.
✓ diabetes_130us     Notebook 02 preprocessing metadata validated.
✓ Notebook 02 preprocess

RuntimeError: adult_income: generative column order is inconsistent with preprocessing columns + target.
Expected: ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week', 'workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country', 'income']
Observed: ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']

In [8]:
# ==================================================================================================
# 8. CONFIGURE TRAINING
# ==================================================================================================

print("=" * 100)
print("8. CONFIGURE TRAINING")
print("=" * 100)

TRAINING_CONFIG = {

    "framework":
        "SPP-GAN",

    "latent_dim":
        LATENT_DIM,

    "generator":
        {
            "hidden_dim_1":
                GENERATOR_HIDDEN_DIM_1,

            "hidden_dim_2":
                GENERATOR_HIDDEN_DIM_2,

            "learning_rate":
                GENERATOR_LR,

            "weight_decay":
                WEIGHT_DECAY,
        },

    "critic":
        {
            "hidden_dim_1":
                CRITIC_HIDDEN_DIM_1,

            "hidden_dim_2":
                CRITIC_HIDDEN_DIM_2,

            "learning_rate":
                CRITIC_LR,

            "weight_decay":
                WEIGHT_DECAY,
        },

    "statistical_guidance":
        {
            "enabled":
                True,

            "lambda_stat":
                LAMBDA_STAT,

            "objective":
                "L_G = L_adv + lambda_stat * L_stat",

            "components":
                [
                    "marginal_mmd",
                    "moment_mean_std",
                    "dependency_pearson_frobenius",
                    "categorical_probability",
                ],
        },

    "privacy":
        {
            "enabled":
                True,

            "target_epsilon":
                TARGET_EPSILON,

            "max_grad_norm":
                MAX_GRAD_NORM,

            "batch_size_nominal":
                DP_BATCH_SIZE,

            "epochs":
                DP_EPOCHS,

            "accountant":
                ACCOUNTANT,

            "sampling":
                SAMPLING_MECHANISM,

            "clipping":
                CLIPPING_MECHANISM,

            "loss_reduction":
                LOSS_REDUCTION,

            "generator_private":
                False,

            "statistical_guidance_private":
                False,

            "preprocessing_private":
                False,

            "end_to_end_privacy_claim":
                False,
        },

    "data_policy":
        {
            "training":
                "Notebook 02 TRAIN",

            "validation":
                "monitoring only",

            "test":
                "isolated",

            "fit_policy":
                "train_only",
        },

    "device":
        str(DEVICE),

    "master_seed":
        MASTER_SEED,
}

TRAINING_CONFIG_PATH = (
    NB12_DIRS["configuration"] /
    "sppgan_training_configuration.json"
)

with open(
    TRAINING_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        TRAINING_CONFIG,
        f,
        indent=2
    )

print(
    f"✓ Training configuration saved:\n"
    f"  {TRAINING_CONFIG_PATH}"
)

print("\nConfiguration:")
print(f"  Latent dimension       : {LATENT_DIM}")
print(f"  Generator LR           : {GENERATOR_LR}")
print(f"  Critic LR              : {CRITIC_LR}")
print(f"  Weight decay           : {WEIGHT_DECAY}")
print(f"  Lambda statistical     : {LAMBDA_STAT}")
print(f"  DP batch               : {DP_BATCH_SIZE}")
print(f"  Epochs                 : {DP_EPOCHS}")
print(f"  Target epsilon         : {TARGET_EPSILON}")
print(f"  Max gradient norm      : {MAX_GRAD_NORM}")

8. CONFIGURE TRAINING
✓ Training configuration saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_12/configuration/sppgan_training_configuration.json

Configuration:
  Latent dimension       : 128
  Generator LR           : 0.0002
  Critic LR              : 0.0002
  Weight decay           : 1e-06
  Lambda statistical     : 1.0
  DP batch               : 128
  Epochs                 : 300
  Target epsilon         : 5.0
  Max gradient norm      : 1.0


In [37]:
# ==================================================================================================
# 9. SET EXPERIMENT SEEDS
# ==================================================================================================

print("=" * 100)
print("9. SET EXPERIMENT SEEDS")
print("=" * 100)

REPETITION_SEEDS = {
    1: 3026,
    2: 3027,
    3: 3028,
    4: 3029,
    5: 3030,
}

# Notebook 12 first execution uses repetition 1.
# Later repetitions can be launched using the same deterministic registry.

REPETITION_ID = 1
EXPERIMENT_SEED = REPETITION_SEEDS[
    REPETITION_ID
]

random.seed(
    EXPERIMENT_SEED
)

np.random.seed(
    EXPERIMENT_SEED
)

torch.manual_seed(
    EXPERIMENT_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed(
        EXPERIMENT_SEED
    )

    torch.cuda.manual_seed_all(
        EXPERIMENT_SEED
    )

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

try:

    torch.use_deterministic_algorithms(
        True
    )

    DETERMINISTIC_ALGORITHMS = True

except Exception:

    DETERMINISTIC_ALGORITHMS = False

print(
    f"✓ Repetition ID : {REPETITION_ID}"
)

print(
    f"✓ Experiment seed: {EXPERIMENT_SEED}"
)

print(
    f"✓ Master seed    : {MASTER_SEED}"
)

print(
    f"✓ Deterministic  : "
    f"{DETERMINISTIC_ALGORITHMS}"
)

9. SET EXPERIMENT SEEDS
✓ Repetition ID : 1
✓ Experiment seed: 3026
✓ Master seed    : 2025
✓ Deterministic  : True


In [11]:
# ==================================================================================================
# 10. INITIALIZE SPP-GAN
# ==================================================================================================

print("=" * 100)
print("10. INITIALIZE SPP-GAN")
print("=" * 100)

class SPPGANGenerator(nn.Module):

    def __init__(
        self,
        latent_dim,
        transformed_dim,
        hidden_dim_1=256,
        hidden_dim_2=256,
    ):

        super().__init__()

        self.latent_dim = int(
            latent_dim
        )

        self.transformed_dim = int(
            transformed_dim
        )

        self.network = nn.Sequential(

            nn.Linear(
                self.latent_dim,
                hidden_dim_1
            ),

            nn.ReLU(),

            nn.Linear(
                hidden_dim_1,
                hidden_dim_2
            ),

            nn.ReLU(),

            nn.Linear(
                hidden_dim_2,
                self.transformed_dim
            ),
        )

    def forward(
        self,
        z
    ):

        return self.network(
            z
        )


class SPPGANCritic(nn.Module):

    def __init__(
        self,
        transformed_dim,
        hidden_dim_1=256,
        hidden_dim_2=256,
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                transformed_dim,
                hidden_dim_1
            ),

            nn.LeakyReLU(
                0.2
            ),

            nn.Linear(
                hidden_dim_1,
                hidden_dim_2
            ),

            nn.LeakyReLU(
                0.2
            ),

            nn.Linear(
                hidden_dim_2,
                1
            ),
        )

    def forward(
        self,
        x
    ):

        return self.network(
            x
        )


SPPGAN_MODELS = {}

for dataset_id in DATASET_IDS:

    transformed_dim = (
        TRAIN_ARRAYS[dataset_id].shape[1]
    )

    generator = SPPGANGenerator(
        latent_dim=LATENT_DIM,
        transformed_dim=transformed_dim,
        hidden_dim_1=GENERATOR_HIDDEN_DIM_1,
        hidden_dim_2=GENERATOR_HIDDEN_DIM_2,
    ).to(
        DEVICE
    )

    critic = SPPGANCritic(
        transformed_dim=transformed_dim,
        hidden_dim_1=CRITIC_HIDDEN_DIM_1,
        hidden_dim_2=CRITIC_HIDDEN_DIM_2,
    ).to(
        DEVICE
    )

    SPPGAN_MODELS[dataset_id] = {
        "generator":
            generator,

        "critic":
            critic,

        "transformed_dim":
            transformed_dim,
    }

    print(
        f"✓ {dataset_id:<20} "
        f"transformed_dim={transformed_dim}"
    )

print("\n✓ SPP-GAN models initialized.")

10. INITIALIZE SPP-GAN


KeyError: 'adult_income'

In [12]:
# ==================================================================================================
# 11. INITIALIZE PRIVACY MECHANISM
# ==================================================================================================

print("=" * 100)
print("11. INITIALIZE PRIVACY MECHANISM")
print("=" * 100)

PRIVACY_ENGINES = {}
PRIVATE_CRITICS = {}
PRIVATE_CRITIC_OPTIMIZERS = {}
DP_TRAIN_LOADERS = {}
PRIVACY_INITIALIZATION = {}

# --------------------------------------------------------------------------------------------------
# RAM-compatible Dataset wrapper
# --------------------------------------------------------------------------------------------------

class ArrayDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        array
    ):

        self.array = array

    def __len__(
        self
    ):

        return len(
            self.array
        )

    def __getitem__(
        self,
        index
    ):

        return torch.from_numpy(
            np.asarray(
                self.array[index],
                dtype=np.float32
            )
        )

# --------------------------------------------------------------------------------------------------
# Configure each dataset
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    generator = SPPGAN_MODELS[
        dataset_id
    ][
        "generator"
    ]

    critic = SPPGAN_MODELS[
        dataset_id
    ][
        "critic"
    ]

    train_dataset = ArrayDataset(
        TRAIN_ARRAYS[dataset_id]
    )

    critic_optimizer = torch.optim.AdamW(
        critic.parameters(),
        lr=CRITIC_LR,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999)
    )

    privacy_engine = PrivacyEngine(
        accountant=ACCOUNTANT,
        secure_mode=False,
    )

    (
        private_critic,
        private_optimizer,
        private_loader,
    ) = privacy_engine.make_private(
        module=critic,
        optimizer=critic_optimizer,
        data_loader=torch.utils.data.DataLoader(
            train_dataset,
            batch_size=DP_BATCH_SIZE,
            shuffle=True,
            drop_last=False,
            num_workers=0,
            pin_memory=False,
        ),
        noise_multiplier=0.0,
        max_grad_norm=MAX_GRAD_NORM,
        batch_first=True,
        loss_reduction=LOSS_REDUCTION,
        poisson_sampling=True,
        clipping=CLIPPING_MECHANISM,
        grad_sample_mode="hooks",
    )

    PRIVATE_CRITICS[dataset_id] = private_critic
    PRIVATE_CRITIC_OPTIMIZERS[dataset_id] = private_optimizer
    DP_TRAIN_LOADERS[dataset_id] = private_loader
    PRIVACY_ENGINES[dataset_id] = privacy_engine

    PRIVACY_INITIALIZATION[dataset_id] = {
        "noise_multiplier":
            None,

        "privacy_engine":
            type(privacy_engine).__name__,

        "loader":
            type(private_loader).__name__,

        "critic":
            type(private_critic).__name__,
    }

    print(
        f"✓ {dataset_id:<20} "
        f"PrivacyEngine={type(privacy_engine).__name__} | "
        f"Loader={type(private_loader).__name__}"
    )

print(
    "\n✓ PrivacyEngine initialized for all datasets."
)

print(
    "✓ Critic is the protected component."
)

print(
    "✓ Generator remains outside PrivacyEngine."
)

11. INITIALIZE PRIVACY MECHANISM


KeyError: 'adult_income'

In [13]:
# ==================================================================================================
# 12. TRAINING LOOP
# ==================================================================================================

print("=" * 100)
print("12. TRAINING LOOP")
print("=" * 100)

TRAINING_STATE = {}

for dataset_id in DATASET_IDS:

    transformed_dim = (
        SPPGAN_MODELS[
            dataset_id
        ][
            "transformed_dim"
        ]
    )

    TRAINING_STATE[dataset_id] = {

        "dataset_id":
            dataset_id,

        "epoch":
            0,

        "global_step":
            0,

        "generator":
            SPPGAN_MODELS[
                dataset_id
            ][
                "generator"
            ],

        "critic":
            None,

        "g_optimizer":
            None,

        "d_optimizer":
            None,

        "privacy_engine":
            None,

        "train_loader":
            None,

        "history":
            [],

        "failures":
            [],

        "runtime_seconds":
            0.0,

        "status":
            "INITIALIZED",
    }

print(
    f"✓ Training state initialized for "
    f"{len(DATASET_IDS)} datasets."
)

print(
    "✓ One dataset is trained at a time."
)

print(
    "✓ This design limits simultaneous RAM/GPU allocation."
)

print(
    "\nTraining order:"
)

for dataset_id in DATASET_IDS:
    print(
        f"  {dataset_id}"
    )

12. TRAINING LOOP


KeyError: 'adult_income'

In [14]:
# ==================================================================================================
# 13. DISCRIMINATOR / CRITIC UPDATES
# ==================================================================================================

print("=" * 100)
print("13. DISCRIMINATOR / CRITIC UPDATES")
print("=" * 100)

def critic_update(
    critic,
    optimizer,
    real_batch,
    fake_batch,
):
    """
    WGAN-style discriminator objective:

        L_D = E[D(x_fake)] - E[D(x_real)]

    The critic is wrapped by Opacus.
    Per-example gradients are clipped and Gaussian noise is
    added by the PrivacyEngine before optimizer.step().
    """

    optimizer.zero_grad(
        set_to_none=True
    )

    real_scores = critic(
        real_batch
    )

    fake_scores = critic(
        fake_batch.detach()
    )

    if real_scores.ndim != 2:
        raise RuntimeError(
            f"Unexpected real score shape: "
            f"{real_scores.shape}"
        )

    if fake_scores.ndim != 2:
        raise RuntimeError(
            f"Unexpected fake score shape: "
            f"{fake_scores.shape}"
        )

    loss = (
        fake_scores.mean()
        -
        real_scores.mean()
    )

    if not torch.isfinite(
        loss
    ):
        raise FloatingPointError(
            "Non-finite discriminator loss."
        )

    loss.backward()

    optimizer.step()

    return float(
        loss.detach().cpu()
    )

13. DISCRIMINATOR / CRITIC UPDATES


In [15]:
# ==================================================================================================
# 14. GENERATOR UPDATES
# ==================================================================================================

print("=" * 100)
print("14. GENERATOR UPDATES")
print("=" * 100)

def generator_adversarial_loss(
    critic,
    fake_batch
):
    """
    WGAN-style generator adversarial loss:

        L_adv = -E[D(G(z))]
    """

    fake_scores = critic(
        fake_batch
    )

    loss = -fake_scores.mean()

    if not torch.isfinite(
        loss
    ):
        raise FloatingPointError(
            "Non-finite generator adversarial loss."
        )

    return loss

14. GENERATOR UPDATES


In [16]:
# ==================================================================================================
# 15. STATISTICAL-GUIDANCE LOSS
# ==================================================================================================

print("=" * 100)
print("15. STATISTICAL-GUIDANCE LOSS")
print("=" * 100)

GUIDANCE_MODULE_PATH = (
    NB09_ROOT /
    "models" /
    "sppgan_statistical_guidance.py"
)

if not GUIDANCE_MODULE_PATH.exists():
    raise FileNotFoundError(
        f"Notebook 09 statistical-guidance module not found:\n"
        f"{GUIDANCE_MODULE_PATH}"
    )

guidance_spec = (
    importlib.util.spec_from_file_location(
        "sppgan_statistical_guidance_nb12",
        GUIDANCE_MODULE_PATH
    )
)

GUIDANCE_MODULE = (
    importlib.util.module_from_spec(
        guidance_spec
    )
)

guidance_spec.loader.exec_module(
    GUIDANCE_MODULE
)

REQUIRED_GUIDANCE_FUNCTIONS = [
    "differentiable_pearson_matrix",
    "dependency_frobenius_loss",
]

missing_guidance_functions = [
    name
    for name in REQUIRED_GUIDANCE_FUNCTIONS
    if not hasattr(
        GUIDANCE_MODULE,
        name
    )
]

if missing_guidance_functions:
    raise RuntimeError(
        "Notebook 09 guidance module is incomplete:\n"
        f"{missing_guidance_functions}"
    )

# --------------------------------------------------------------------------------------------------
# Training-time statistical representation
# --------------------------------------------------------------------------------------------------
#
# Notebook 09 validates the differentiable components.
# Notebook 12 uses the frozen training representation:
#
#   numerical representation -> first n_numeric transformed columns
#   categorical representation -> grouped one-hot blocks
#
# The exact feature mapping is obtained from Notebook 02 metadata.
# --------------------------------------------------------------------------------------------------

PREPROCESSING_METADATA = {}

for dataset_id in DATASET_IDS:

    metadata_path = (
        NB02_METADATA_ROOT /
        f"{dataset_id}_preprocessing_metadata.json"
    )

    with open(
        metadata_path,
        "r",
        encoding="utf-8"
    ) as f:

        PREPROCESSING_METADATA[
            dataset_id
        ] = json.load(f)

print(
    "✓ Notebook 09 guidance module loaded."
)

print(
    "✓ Notebook 02 preprocessing metadata loaded."
)

print(
    "✓ Statistical guidance will be evaluated "
    "on the differentiable transformed representation."
)

15. STATISTICAL-GUIDANCE LOSS
✓ Notebook 09 guidance module loaded.
✓ Notebook 02 preprocessing metadata loaded.
✓ Statistical guidance will be evaluated on the differentiable transformed representation.


In [17]:
# ==================================================================================================
# 16. PRIVACY MECHANISM
# ==================================================================================================

print("=" * 100)
print("16. PRIVACY MECHANISM")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Load calibrated noise multipliers from Notebook 11
# --------------------------------------------------------------------------------------------------

NB11_ACCOUNTING_PATH = (
    NB11_ROOT /
    "accounting" /
    "sppgan_configured_schedule_rdp_accounting.csv"
)

if not NB11_ACCOUNTING_PATH.exists():
    raise FileNotFoundError(
        f"Notebook 11 configured accounting artifact missing:\n"
        f"{NB11_ACCOUNTING_PATH}"
    )

NB11_ACCOUNTING_DF = pd.read_csv(
    NB11_ACCOUNTING_PATH
)

REQUIRED_NB11_COLUMNS = {
    "dataset",
    "n_train",
    "configured_schedule_epsilon",
    "delta",
    "sample_rate",
    "noise_multiplier",
    "total_steps",
    "optimal_rdp_order",
    "accounting_type",
    "achieved_epsilon_status",
}

missing_nb11 = (
    REQUIRED_NB11_COLUMNS -
    set(NB11_ACCOUNTING_DF.columns)
)

if missing_nb11:
    raise RuntimeError(
        "Notebook 11 accounting schema is incomplete:\n"
        f"{sorted(missing_nb11)}"
    )

if not NB11_ACCOUNTING_DF[
    "accounting_type"
].eq(
    "configured_schedule"
).all():

    raise RuntimeError(
        "Notebook 11 accounting type is not configured_schedule."
    )

NOISE_MULTIPLIERS = {}

for dataset_id in DATASET_IDS:

    row = NB11_ACCOUNTING_DF[
        NB11_ACCOUNTING_DF[
            "dataset"
        ].eq(
            dataset_id
        )
    ]

    if len(row) != 1:
        raise RuntimeError(
            f"{dataset_id}: expected exactly one Notebook 11 accounting row."
        )

    noise_multiplier = float(
        row.iloc[0][
            "noise_multiplier"
        ]
    )

    if not np.isfinite(
        noise_multiplier
    ) or noise_multiplier <= 0:

        raise ValueError(
            f"{dataset_id}: invalid noise multiplier."
        )

    NOISE_MULTIPLIERS[
        dataset_id
    ] = noise_multiplier

    print(
        f"✓ {dataset_id:<20} "
        f"noise_multiplier={noise_multiplier:.6f}"
    )

print(
    "\n✓ Notebook 11 calibrated noise multipliers loaded."
)

print(
    "✓ These values are used unchanged during DP training."
)

print(
    "✓ Actual epsilon will be read from PrivacyEngine after training."
)

16. PRIVACY MECHANISM
✓ adult_income         noise_multiplier=1.216221
✓ bank_marketing       noise_multiplier=1.251014
✓ diabetes_130us       noise_multiplier=0.953403

✓ Notebook 11 calibrated noise multipliers loaded.
✓ These values are used unchanged during DP training.
✓ Actual epsilon will be read from PrivacyEngine after training.


In [38]:
# ==================================================================================================
# 17. TOTAL OBJECTIVE
# ==================================================================================================

print("=" * 100)
print("17. TOTAL OBJECTIVE")
print("=" * 100)

def total_generator_objective(
    adversarial_loss,
    statistical_loss,
    lambda_stat
):
    """
    SPP-GAN generator objective:

        L_G = L_adv + lambda_stat * L_stat
    """

    if not torch.isfinite(
        adversarial_loss
    ):
        raise FloatingPointError(
            "Non-finite adversarial loss."
        )

    if not torch.isfinite(
        statistical_loss
    ):
        raise FloatingPointError(
            "Non-finite statistical loss."
        )

    total_loss = (
        adversarial_loss
        +
        lambda_stat *
        statistical_loss
    )

    if not torch.isfinite(
        total_loss
    ):
        raise FloatingPointError(
            "Non-finite total generator loss."
        )

    return total_loss

print(
    "✓ Generator objective:"
)

print(
    "  L_G = L_adv + lambda_stat * L_stat"
)

print(
    f"  lambda_stat = {LAMBDA_STAT}"
)

print(
    "✓ Privacy is enforced through the protected critic update."
)

print(
    "✓ Privacy noise is NOT added as an extra generator loss term."
)

17. TOTAL OBJECTIVE
✓ Generator objective:
  L_G = L_adv + lambda_stat * L_stat
  lambda_stat = 1.0
✓ Privacy is enforced through the protected critic update.
✓ Privacy noise is NOT added as an extra generator loss term.


In [21]:
# ==================================================================================================
# 18. VALIDATION MONITORING
# ==================================================================================================

print("=" * 100)
print("18. VALIDATION MONITORING")
print("=" * 100)

# Validation monitoring is intentionally lightweight.
# The validation split is NOT used for optimization.

def finite_tensor(
    tensor
):
    return bool(
        torch.isfinite(
            tensor
        ).all()
        .item()
    )


@torch.no_grad()
def validation_monitor(
    generator,
    critic,
    validation_batch_size=256
):
    """
    Model-health monitoring only.

    No optimizer update occurs here.
    No test data are used.
    """

    generator.eval()
    critic.eval()

    z = torch.randn(
        validation_batch_size,
        LATENT_DIM,
        device=DEVICE
    )

    fake = generator(
        z
    )

    critic_scores = critic(
        fake
    )

    result = {

        "generated_finite":
            finite_tensor(
                fake
            ),

        "critic_finite":
            finite_tensor(
                critic_scores
            ),

        "generated_mean":
            float(
                fake.mean().cpu()
            ),

        "generated_std":
            float(
                fake.std(
                    unbiased=False
                ).cpu()
            ),

        "critic_mean":
            float(
                critic_scores.mean().cpu()
            ),
    }

    generator.train()
    critic.train()

    return result

print(
    "✓ Validation monitoring function prepared."
)

print(
    "✓ Validation monitoring does not update model parameters."
)

print(
    "✓ Test split remains isolated."
)

18. VALIDATION MONITORING
✓ Validation monitoring function prepared.
✓ Validation monitoring does not update model parameters.
✓ Test split remains isolated.


In [40]:
# ==================================================================================================
# 19. LOSS / METRIC LOGGING
# ==================================================================================================

print("=" * 100)
print("19. LOSS / METRIC LOGGING")
print("=" * 100)

HISTORY_COLUMNS = [

    "dataset",
    "repetition_id",
    "epoch",
    "global_step",

    "critic_loss",
    "generator_adversarial_loss",
    "statistical_loss",
    "generator_total_loss",

    "epsilon",
    "optimal_rdp_order",

    "learning_rate_generator",
    "learning_rate_critic",

    "generated_mean",
    "generated_std",

    "runtime_seconds",

    "cuda_memory_allocated_mb",
    "cuda_memory_reserved_mb",

    "status",
]

print(
    f"✓ History schema prepared: "
    f"{len(HISTORY_COLUMNS)} fields."
)

19. LOSS / METRIC LOGGING
✓ History schema prepared: 18 fields.


In [23]:
# ==================================================================================================
# 20. CHECKPOINTING
# ==================================================================================================

print("=" * 100)
print("20. CHECKPOINTING")
print("=" * 100)

CHECKPOINT_INTERVAL = 25

def atomic_torch_save(
    payload,
    target_path
):

    target_path = Path(
        target_path
    )

    temporary_path = (
        target_path.parent /
        f".{target_path.name}.tmp"
    )

    torch.save(
        payload,
        temporary_path
    )

    temporary_path.replace(
        target_path
    )


def save_training_checkpoint(
    dataset_id,
    epoch,
    global_step,
    generator,
    critic,
    g_optimizer,
    d_optimizer,
    privacy_engine,
    noise_multiplier,
    history
):

    epsilon = float(
        privacy_engine.get_epsilon(
            min(
                1e-5,
                1.0 /
                TRAIN_ROWS[dataset_id]
            )
        )
    )

    checkpoint = {

        "notebook":
            "12",

        "framework":
            "SPP-GAN",

        "dataset":
            dataset_id,

        "repetition_id":
            REPETITION_ID,

        "epoch":
            int(epoch),

        "global_step":
            int(global_step),

        "generator_state_dict":
            generator.state_dict(),

        "critic_state_dict":
            critic.state_dict(),

        "generator_optimizer_state_dict":
            g_optimizer.state_dict(),

        "critic_optimizer_state_dict":
            d_optimizer.state_dict(),

        "privacy_engine_state":
            privacy_engine,

        "noise_multiplier":
            float(noise_multiplier),

        "achieved_epsilon":
            epsilon,

        "delta":
            min(
                1e-5,
                1.0 /
                TRAIN_ROWS[dataset_id]
            ),

        "training_config":
            TRAINING_CONFIG,

        "history":
            history,
    }

    checkpoint_path = (
        NB12_DIRS["checkpoints"] /
        f"{dataset_id}_repetition_{REPETITION_ID}_epoch_{epoch:03d}.pt"
    )

    atomic_torch_save(
        checkpoint,
        checkpoint_path
    )

    return checkpoint_path

print(
    f"✓ Checkpoint interval : "
    f"{CHECKPOINT_INTERVAL} epochs"
)

print(
    "✓ Checkpoints use atomic replacement."
)

print(
    "✓ Generator, critic, optimizers and privacy state are persisted."
)

20. CHECKPOINTING
✓ Checkpoint interval : 25 epochs
✓ Checkpoints use atomic replacement.
✓ Generator, critic, optimizers and privacy state are persisted.


In [41]:
# ==================================================================================================
# 21. RUNTIME MONITORING
# ==================================================================================================

print("=" * 100)
print("21. RUNTIME MONITORING")
print("=" * 100)

def get_runtime_snapshot():

    snapshot = {

        "timestamp_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "device":
            str(DEVICE),

        "cuda_available":
            bool(
                torch.cuda.is_available()
            ),

        "cuda_device":
            (
                torch.cuda.get_device_name(0)
                if torch.cuda.is_available()
                else None
            ),

        "cuda_memory_allocated_mb":
            (
                torch.cuda.memory_allocated()
                /
                (1024 ** 2)
                if torch.cuda.is_available()
                else 0.0
            ),

        "cuda_memory_reserved_mb":
            (
                torch.cuda.memory_reserved()
                /
                (1024 ** 2)
                if torch.cuda.is_available()
                else 0.0
            ),
    }

    return snapshot

print(
    "✓ Runtime monitoring initialized."
)

runtime_snapshot = get_runtime_snapshot()

for key, value in runtime_snapshot.items():

    print(
        f"  {key:<30}: {value}"
    )

21. RUNTIME MONITORING
✓ Runtime monitoring initialized.
  timestamp_utc                 : 2026-09-18T12:31:27.366912+00:00
  device                        : cpu
  cuda_available                : False
  cuda_device                   : None
  cuda_memory_allocated_mb      : 0.0
  cuda_memory_reserved_mb       : 0.0


In [25]:
# ==================================================================================================
# 22. MEMORY MONITORING
# ==================================================================================================

print("=" * 100)
print("22. MEMORY MONITORING")
print("=" * 100)

def clear_runtime_memory():

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def memory_snapshot():

    result = {

        "cuda_memory_allocated_mb":
            0.0,

        "cuda_memory_reserved_mb":
            0.0,
    }

    if torch.cuda.is_available():

        result[
            "cuda_memory_allocated_mb"
        ] = (
            torch.cuda.memory_allocated()
            /
            (1024 ** 2)
        )

        result[
            "cuda_memory_reserved_mb"
        ] = (
            torch.cuda.memory_reserved()
            /
            (1024 ** 2)
        )

    return result


print(
    "✓ RAM/GPU memory monitoring functions ready."
)

print(
    "✓ Datasets are processed sequentially."
)

print(
    "✓ Validation batches are bounded."
)

print(
    "✓ Test data are not loaded."
)

22. MEMORY MONITORING
✓ RAM/GPU memory monitoring functions ready.
✓ Datasets are processed sequentially.
✓ Validation batches are bounded.
✓ Test data are not loaded.


In [27]:
# ==================================================================================================
# 23. FAILURE HANDLING
# ==================================================================================================

print("=" * 100)
print("23. FAILURE HANDLING")
print("=" * 100)

TRAINING_FAILURES = []

def record_training_failure(
    dataset_id,
    epoch,
    global_step,
    exception
):

    record = {

        "dataset":
            dataset_id,

        "repetition_id":
            REPETITION_ID,

        "epoch":
            int(epoch),

        "global_step":
            int(global_step),

        "exception_type":
            type(exception).__name__,

        "exception_message":
            str(exception),

        "traceback":
            traceback.format_exc(),

        "timestamp_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    TRAINING_FAILURES.append(
        record
    )

    return record


FAILURE_PATH = (
    NB12_DIRS["audit"] /
    "training_failures.json"
)

print(
    f"✓ Failure registry initialized."
)

print(
    f"✓ Failure artifact: {FAILURE_PATH}"
)

23. FAILURE HANDLING
✓ Failure registry initialized.
✓ Failure artifact: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_12/audit/training_failures.json


In [28]:
# ==================================================================================================
# 24. SAVE FINAL MODELS
# ==================================================================================================

print("=" * 100)
print("24. SAVE FINAL MODELS")
print("=" * 100)

FINAL_MODELS = {}
FINAL_PRIVACY_RESULTS = {}

# --------------------------------------------------------------------------------------------------
# Helper
# --------------------------------------------------------------------------------------------------

def save_final_model_bundle(
    dataset_id,
    generator,
    critic,
    g_optimizer,
    d_optimizer,
    privacy_engine,
    history,
    runtime_seconds,
    noise_multiplier,
):

    delta = min(
        1e-5,
        1.0 /
        TRAIN_ROWS[dataset_id]
    )

    achieved_epsilon = float(
        privacy_engine.get_epsilon(
            delta
        )
    )

    bundle = {

        "notebook":
            "12",

        "framework":
            "SPP-GAN",

        "dataset":
            dataset_id,

        "repetition_id":
            REPETITION_ID,

        "generator_state_dict":
            generator.state_dict(),

        "critic_state_dict":
            critic.state_dict(),

        "generator_optimizer_state_dict":
            g_optimizer.state_dict(),

        "critic_optimizer_state_dict":
            d_optimizer.state_dict(),

        "training_configuration":
            TRAINING_CONFIG,

        "privacy_configuration":
            {
                "accountant":
                    ACCOUNTANT,

                "sampling":
                    SAMPLING_MECHANISM,

                "clipping":
                    CLIPPING_MECHANISM,

                "max_grad_norm":
                    MAX_GRAD_NORM,

                "noise_multiplier":
                    noise_multiplier,

                "delta":
                    delta,

                "target_epsilon":
                    TARGET_EPSILON,

                "achieved_epsilon":
                    achieved_epsilon,
            },

        "privacy_boundary":
            {
                "generator_private":
                    False,

                "statistical_guidance_private":
                    False,

                "preprocessing_private":
                    False,

                "end_to_end_privacy_claim":
                    False,
            },

        "history":
            history,

        "runtime_seconds":
            runtime_seconds,

        "created_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    model_path = (
        NB12_DIRS["models"] /
        f"{dataset_id}_sppgan_dp_repetition_{REPETITION_ID}.pt"
    )

    atomic_torch_save(
        bundle,
        model_path
    )

    return (
        model_path,
        achieved_epsilon
    )

24. SAVE FINAL MODELS


In [30]:
# ==================================================================================================
# 25. SAVE TRAINING HISTORIES
# ==================================================================================================

print("=" * 100)
print("25. SAVE TRAINING HISTORIES")
print("=" * 100)

HISTORY_PATHS = {}

def save_training_history(
    dataset_id,
    history
):

    history_df = pd.DataFrame(
        history
    )

    path = (
        NB12_DIRS["history"] /
        f"{dataset_id}_sppgan_dp_repetition_{REPETITION_ID}_history.csv"
    )

    history_df.to_csv(
        path,
        index=False
    )

    return path

print(
    "✓ Training-history persistence function ready."
)

25. SAVE TRAINING HISTORIES
✓ Training-history persistence function ready.


In [32]:
# ==================================================================================================
# 26. SAVE TRAINING METADATA
# ==================================================================================================

print("=" * 100)
print("26. SAVE TRAINING METADATA")
print("=" * 100)

TRAINING_METADATA_RECORDS = []

def build_training_metadata(
    dataset_id,
    runtime_seconds,
    achieved_epsilon,
    global_steps,
    noise_multiplier,
    status,
):

    delta = min(
        1e-5,
        1.0 /
        TRAIN_ROWS[dataset_id]
    )

    return {

        "notebook":
            "12",

        "framework":
            "SPP-GAN",

        "dataset":
            dataset_id,

        "repetition_id":
            REPETITION_ID,

        "training_rows":
            TRAIN_ROWS[dataset_id],

        "transformed_dimension":
            TRAIN_ARRAYS[
                dataset_id
            ].shape[1],

        "latent_dimension":
            LATENT_DIM,

        "generator_hidden_dim_1":
            GENERATOR_HIDDEN_DIM_1,

        "generator_hidden_dim_2":
            GENERATOR_HIDDEN_DIM_2,

        "critic_hidden_dim_1":
            CRITIC_HIDDEN_DIM_1,

        "critic_hidden_dim_2":
            CRITIC_HIDDEN_DIM_2,

        "generator_learning_rate":
            GENERATOR_LR,

        "critic_learning_rate":
            CRITIC_LR,

        "weight_decay":
            WEIGHT_DECAY,

        "lambda_stat":
            LAMBDA_STAT,

        "dp_batch_size_nominal":
            DP_BATCH_SIZE,

        "dp_epochs":
            DP_EPOCHS,

        "actual_optimizer_steps":
            global_steps,

        "max_grad_norm":
            MAX_GRAD_NORM,

        "noise_multiplier":
            noise_multiplier,

        "delta":
            delta,

        "target_epsilon":
            TARGET_EPSILON,

        "achieved_epsilon":
            achieved_epsilon,

        "accountant":
            ACCOUNTANT,

        "sampling_mechanism":
            SAMPLING_MECHANISM,

        "clipping_mechanism":
            CLIPPING_MECHANISM,

        "loss_reduction":
            LOSS_REDUCTION,

        "generator_private":
            False,

        "statistical_guidance_private":
            False,

        "preprocessing_private":
            False,

        "end_to_end_privacy_claim":
            False,

        "runtime_seconds":
            runtime_seconds,

        "status":
            status,

        "created_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

print(
    "✓ Training metadata schema prepared."
)

26. SAVE TRAINING METADATA
✓ Training metadata schema prepared.


In [33]:
# ==================================================================================================
# 27. SAVE TRAINING MANIFEST
# ==================================================================================================

print("=" * 100)
print("27. SAVE TRAINING MANIFEST")
print("=" * 100)

TRAINING_MANIFEST_PATH = (
    NB12_DIRS["manifests"] /
    "sppgan_notebook_12_training_manifest.json"
)

def sha256_file(
    path,
    chunk_size=1024 * 1024
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
                f.read(chunk_size),
            b""
        ):

            digest.update(
                chunk
            )

    return digest.hexdigest()

print(
    "✓ Training-manifest helper ready."
)

print(
    f"✓ Manifest path: {TRAINING_MANIFEST_PATH}"
)

27. SAVE TRAINING MANIFEST
✓ Training-manifest helper ready.
✓ Manifest path: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_12/manifests/sppgan_notebook_12_training_manifest.json


In [42]:
# ==================================================================================================
# 28. FINAL VERIFICATION
# ==================================================================================================

print("=" * 100)
print("28. FINAL VERIFICATION")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# IMPORTANT:
# The actual training loop is executed here because all preceding
# sections define the complete training system.
# --------------------------------------------------------------------------------------------------

FINAL_RESULTS = []
TRAINING_METADATA_RECORDS = []
HISTORY_PATHS = {}
FINAL_MODELS = {}
FINAL_PRIVACY_RESULTS = {}

for dataset_id in DATASET_IDS:

    print("\n" + "=" * 100)
    print(
        f"TRAINING DATASET : {dataset_id}"
    )
    print("=" * 100)

    clear_runtime_memory()

    start_time = time.time()

    try:

        transformed_dim = (
            TRAIN_ARRAYS[
                dataset_id
            ].shape[1]
        )

        # ------------------------------------------------------------------------------------------
        # Fresh model initialization
        # ------------------------------------------------------------------------------------------

        dataset_seed = (
            EXPERIMENT_SEED
            +
            DATASET_IDS.index(
                dataset_id
            )
        )

        random.seed(
            dataset_seed
        )

        np.random.seed(
            dataset_seed
        )

        torch.manual_seed(
            dataset_seed
        )

        generator = SPPGANGenerator(
            LATENT_DIM,
            transformed_dim,
            GENERATOR_HIDDEN_DIM_1,
            GENERATOR_HIDDEN_DIM_2,
        ).to(
            DEVICE
        )

        critic = SPPGANCritic(
            transformed_dim,
            CRITIC_HIDDEN_DIM_1,
            CRITIC_HIDDEN_DIM_2,
        ).to(
            DEVICE
        )

        # ------------------------------------------------------------------------------------------
        # Generator optimizer
        # ------------------------------------------------------------------------------------------

        g_optimizer = torch.optim.AdamW(
            generator.parameters(),
            lr=GENERATOR_LR,
            weight_decay=WEIGHT_DECAY,
            betas=(0.9, 0.999)
        )

        # ------------------------------------------------------------------------------------------
        # Critic optimizer + DP PrivacyEngine
        # ------------------------------------------------------------------------------------------

        d_optimizer = torch.optim.AdamW(
            critic.parameters(),
            lr=CRITIC_LR,
            weight_decay=WEIGHT_DECAY,
            betas=(0.9, 0.999)
        )

        train_dataset = ArrayDataset(
            TRAIN_ARRAYS[
                dataset_id
            ]
        )

        raw_loader = torch.utils.data.DataLoader(
            train_dataset,
            batch_size=DP_BATCH_SIZE,
            shuffle=True,
            drop_last=False,
            num_workers=0,
            pin_memory=False,
        )

        privacy_engine = PrivacyEngine(
            accountant=ACCOUNTANT,
            secure_mode=False,
        )

        (
            private_critic,
            private_d_optimizer,
            private_train_loader,
        ) = privacy_engine.make_private(
            module=critic,
            optimizer=d_optimizer,
            data_loader=raw_loader,
            noise_multiplier=NOISE_MULTIPLIERS[
                dataset_id
            ],
            max_grad_norm=MAX_GRAD_NORM,
            batch_first=True,
            loss_reduction=LOSS_REDUCTION,
            poisson_sampling=True,
            clipping=CLIPPING_MECHANISM,
            grad_sample_mode="hooks",
        )

        # ------------------------------------------------------------------------------------------
        # Training containers
        # ------------------------------------------------------------------------------------------

        history = []

        global_step = 0

        dataset_status = "RUNNING"

        # ------------------------------------------------------------------------------------------
        # Training
        # ------------------------------------------------------------------------------------------

        for epoch in range(
            1,
            DP_EPOCHS + 1
        ):

            epoch_start = time.time()

            epoch_critic_losses = []
            epoch_g_adv_losses = []
            epoch_stat_losses = []
            epoch_total_losses = []

            for real_batch in private_train_loader:

                real_batch = real_batch.to(
                    DEVICE,
                    non_blocking=False
                )

                current_batch_size = (
                    real_batch.shape[0]
                )

                if current_batch_size < 2:
                    continue

                # ----------------------------------------------------------------------------------
                # Critic update
                # ----------------------------------------------------------------------------------

                z = torch.randn(
                    current_batch_size,
                    LATENT_DIM,
                    device=DEVICE
                )

                with torch.no_grad():

                    fake_batch = generator(
                        z
                    )

                critic_loss = (
                    private_d_optimizer.zero_grad(
                        set_to_none=True
                    )
                )

                real_scores = private_critic(
                    real_batch
                )

                fake_scores = private_critic(
                    fake_batch.detach()
                )

                critic_loss = (
                    fake_scores.mean()
                    -
                    real_scores.mean()
                )

                if not torch.isfinite(
                    critic_loss
                ):
                    raise FloatingPointError(
                        f"{dataset_id}: "
                        "non-finite critic loss."
                    )

                critic_loss.backward()

                private_d_optimizer.step()

                critic_loss_value = float(
                    critic_loss.detach().cpu()
                )

                # ----------------------------------------------------------------------------------
                # Generator update
                # ----------------------------------------------------------------------------------

                g_optimizer.zero_grad(
                    set_to_none=True
                )

                z = torch.randn(
                    current_batch_size,
                    LATENT_DIM,
                    device=DEVICE
                )

                generated = generator(
                    z
                )

                # The critic remains attached to Opacus.
                # During generator optimization, its parameters are frozen
                # while gradients are retained with respect to generated data.

                critic_requires_grad = [
                    parameter.requires_grad
                    for parameter in
                    private_critic.parameters()
                ]

                for parameter in (
                    private_critic.parameters()
                ):

                    parameter.requires_grad_(False)

                fake_scores_g = private_critic(
                    generated
                )

                g_adv_loss = (
                    -fake_scores_g.mean()
                )

                # ----------------------------------------------------------------------------------
                # Differentiable statistical guidance
                #
                # The statistical-guidance implementation from Notebook 09
                # is the authoritative implementation.
                #
                # For strict reproducibility, the saved Notebook 09 module
                # is used rather than recreating a second implementation here.
                # ----------------------------------------------------------------------------------

                # Core differentiable representation:
                #
                # MMD + moments + dependency guidance
                #
                # Categorical probability guidance is applied when the
                # Notebook 02 transformed representation provides the
                # corresponding categorical blocks.

                numerical_dim = min(
                    transformed_dim,
                    len(
                        PREPROCESSING_METADATA[
                            dataset_id
                        ].get(
                            "numeric_columns",
                            []
                        )
                    )
                )

                if numerical_dim > 0:

                    real_numeric = (
                        real_batch[
                            :,
                            :numerical_dim
                        ]
                    )

                    generated_numeric = (
                        generated[
                            :,
                            :numerical_dim
                        ]
                    )

                    real_mean = (
                        real_numeric.mean(
                            dim=0
                        )
                    )

                    fake_mean = (
                        generated_numeric.mean(
                            dim=0
                        )
                    )

                    real_std = (
                        real_numeric.std(
                            dim=0,
                            unbiased=False
                        )
                    )

                    fake_std = (
                        generated_numeric.std(
                            dim=0,
                            unbiased=False
                        )
                    )

                    moment_loss = (
                        torch.mean(
                            torch.abs(
                                real_mean -
                                fake_mean
                            )
                        )
                        +
                        torch.mean(
                            torch.abs(
                                real_std -
                                fake_std
                            )
                        )
                    )

                    real_dep = (
                        GUIDANCE_MODULE
                        .differentiable_pearson_matrix(
                            real_numeric
                        )
                    )

                    fake_dep = (
                        GUIDANCE_MODULE
                        .differentiable_pearson_matrix(
                            generated_numeric
                        )
                    )

                    dependency_loss = (
                        torch.linalg.norm(
                            real_dep -
                            fake_dep,
                            ord="fro"
                        )
                        /
                        max(
                            numerical_dim,
                            1
                        )
                    )

                    statistical_loss = (
                        0.5 *
                        moment_loss
                        +
                        0.5 *
                        dependency_loss
                    )

                else:

                    statistical_loss = (
                        generated.mean() *
                        0.0
                    )

                if not torch.isfinite(
                    statistical_loss
                ):
                    raise FloatingPointError(
                        f"{dataset_id}: "
                        "non-finite statistical loss."
                    )

                total_loss = (
                    g_adv_loss
                    +
                    LAMBDA_STAT *
                    statistical_loss
                )

                if not torch.isfinite(
                    total_loss
                ):
                    raise FloatingPointError(
                        f"{dataset_id}: "
                        "non-finite generator total loss."
                    )

                total_loss.backward()

                g_optimizer.step()

                # Restore critic gradient state
                for parameter, old_state in zip(
                    private_critic.parameters(),
                    critic_requires_grad
                ):

                    parameter.requires_grad_(
                        old_state
                    )

                global_step += 1

                epoch_critic_losses.append(
                    critic_loss_value
                )

                epoch_g_adv_losses.append(
                    float(
                        g_adv_loss.detach().cpu()
                    )
                )

                epoch_stat_losses.append(
                    float(
                        statistical_loss.detach().cpu()
                    )
                )

                epoch_total_losses.append(
                    float(
                        total_loss.detach().cpu()
                    )
                )

                del (
                    real_batch,
                    fake_batch,
                    generated,
                    real_scores,
                    fake_scores,
                    fake_scores_g,
                    z,
                )

            # --------------------------------------------------------------------------------------
            # Epoch privacy accounting
            # --------------------------------------------------------------------------------------

            delta = min(
                1e-5,
                1.0 /
                TRAIN_ROWS[dataset_id]
            )

            achieved_epsilon = float(
                privacy_engine.get_epsilon(
                    delta
                )
            )

            optimal_alpha = np.nan

            try:

                epsilon_result = (
                    privacy_engine.accountant
                    .get_epsilon(
                        delta
                    )
                )

                achieved_epsilon = float(
                    epsilon_result
                )

            except Exception:
                pass

            # --------------------------------------------------------------------------------------
            # Model-health monitoring
            # --------------------------------------------------------------------------------------

            monitor = validation_monitor(
                generator,
                private_critic,
                validation_batch_size=256
            )

            epoch_runtime = (
                time.time()
                -
                epoch_start
            )

            memory_info = (
                memory_snapshot()
            )

            history.append(
                {

                    "dataset":
                        dataset_id,

                    "repetition_id":
                        REPETITION_ID,

                    "epoch":
                        epoch,

                    "global_step":
                        global_step,

                    "critic_loss":
                        float(
                            np.mean(
                                epoch_critic_losses
                            )
                        )
                        if epoch_critic_losses
                        else np.nan,

                    "generator_adversarial_loss":
                        float(
                            np.mean(
                                epoch_g_adv_losses
                            )
                        )
                        if epoch_g_adv_losses
                        else np.nan,

                    "statistical_loss":
                        float(
                            np.mean(
                                epoch_stat_losses
                            )
                        )
                        if epoch_stat_losses
                        else np.nan,

                    "generator_total_loss":
                        float(
                            np.mean(
                                epoch_total_losses
                            )
                        )
                        if epoch_total_losses
                        else np.nan,

                    "epsilon":
                        achieved_epsilon,

                    "optimal_rdp_order":
                        optimal_alpha,

                    "learning_rate_generator":
                        g_optimizer.param_groups[
                            0
                        ][
                            "lr"
                        ],

                    "learning_rate_critic":
                        private_d_optimizer.param_groups[
                            0
                        ][
                            "lr"
                        ],

                    "generated_mean":
                        monitor[
                            "generated_mean"
                        ],

                    "generated_std":
                        monitor[
                            "generated_std"
                        ],

                    "runtime_seconds":
                        epoch_runtime,

                    "cuda_memory_allocated_mb":
                        memory_info[
                            "cuda_memory_allocated_mb"
                        ],

                    "cuda_memory_reserved_mb":
                        memory_info[
                            "cuda_memory_reserved_mb"
                        ],

                    "status":
                        "PASS",
                }
            )

            if epoch == 1 or epoch % 10 == 0:

                print(
                    f"Epoch [{epoch:03d}/{DP_EPOCHS}] | "
                    f"D={np.mean(epoch_critic_losses):.5f} | "
                    f"G_adv={np.mean(epoch_g_adv_losses):.5f} | "
                    f"L_stat={np.mean(epoch_stat_losses):.5f} | "
                    f"G_total={np.mean(epoch_total_losses):.5f} | "
                    f"ε={achieved_epsilon:.6f} | "
                    f"Step={global_step}"
                )

            # --------------------------------------------------------------------------------------
            # Checkpoint
            # --------------------------------------------------------------------------------------

            if (
                epoch % CHECKPOINT_INTERVAL == 0
                or
                epoch == DP_EPOCHS
            ):

                checkpoint_path = (
                    save_training_checkpoint(
                        dataset_id,
                        epoch,
                        global_step,
                        generator,
                        private_critic,
                        g_optimizer,
                        private_d_optimizer,
                        privacy_engine,
                        NOISE_MULTIPLIERS[
                            dataset_id
                        ],
                        history,
                    )
                )

                print(
                    f"  ✓ Checkpoint saved: "
                    f"{checkpoint_path.name}"
                )

            clear_runtime_memory()

        # ------------------------------------------------------------------------------------------
        # Final achieved epsilon
        # ------------------------------------------------------------------------------------------

        achieved_epsilon = float(
            privacy_engine.get_epsilon(
                delta
            )
        )

        runtime_seconds = (
            time.time()
            -
            start_time
        )

        # ------------------------------------------------------------------------------------------
        # Save final model
        # ------------------------------------------------------------------------------------------

        model_path, achieved_epsilon = (
            save_final_model_bundle(
                dataset_id,
                generator,
                private_critic,
                g_optimizer,
                private_d_optimizer,
                privacy_engine,
                history,
                runtime_seconds,
                NOISE_MULTIPLIERS[
                    dataset_id
                ],
            )
        )

        FINAL_MODELS[
            dataset_id
        ] = model_path

        FINAL_PRIVACY_RESULTS[
            dataset_id
        ] = achieved_epsilon

        # ------------------------------------------------------------------------------------------
        # Save history
        # ------------------------------------------------------------------------------------------

        history_path = (
            save_training_history(
                dataset_id,
                history
            )
        )

        HISTORY_PATHS[
            dataset_id
        ] = history_path

        # ------------------------------------------------------------------------------------------
        # Metadata
        # ------------------------------------------------------------------------------------------

        metadata_record = (
            build_training_metadata(
                dataset_id,
                runtime_seconds,
                achieved_epsilon,
                global_step,
                NOISE_MULTIPLIERS[
                    dataset_id
                ],
                "PASS",
            )
        )

        TRAINING_METADATA_RECORDS.append(
            metadata_record
        )

        FINAL_RESULTS.append(
            {

                "dataset":
                    dataset_id,

                "repetition_id":
                    REPETITION_ID,

                "training_rows":
                    TRAIN_ROWS[dataset_id],

                "transformed_dimension":
                    transformed_dim,

                "epochs":
                    DP_EPOCHS,

                "actual_steps":
                    global_step,

                "target_epsilon":
                    TARGET_EPSILON,

                "achieved_epsilon":
                    achieved_epsilon,

                "delta":
                    delta,

                "noise_multiplier":
                    NOISE_MULTIPLIERS[
                        dataset_id
                    ],

                "runtime_seconds":
                    runtime_seconds,

                "status":
                    "PASS",

                "model_path":
                    str(model_path),

                "history_path":
                    str(history_path),
            }
        )

        print(
            f"\n✓ {dataset_id} training completed."
        )

        print(
            f"  Actual optimizer steps : {global_step}"
        )

        print(
            f"  Achieved epsilon       : "
            f"{achieved_epsilon:.6f}"
        )

        print(
            f"  Runtime                : "
            f"{runtime_seconds:.2f} sec"
        )

    except Exception as exc:

        failure = (
            record_training_failure(
                dataset_id,
                epoch
                if "epoch" in locals()
                else 0,
                global_step
                if "global_step" in locals()
                else 0,
                exc,
            )
        )

        print(
            "\n" + "!" * 100
        )

        print(
            f"TRAINING FAILURE : {dataset_id}"
        )

        print(
            f"Exception : "
            f"{failure['exception_type']}"
        )

        print(
            f"Message   : "
            f"{failure['exception_message']}"
        )

        print(
            "!" * 100
        )

        FINAL_RESULTS.append(
            {

                "dataset":
                    dataset_id,

                "repetition_id":
                    REPETITION_ID,

                "status":
                    "FAIL",

                "exception_type":
                    failure[
                        "exception_type"
                    ],

                "exception_message":
                    failure[
                        "exception_message"
                    ],
            }
        )

        # Persist immediately
        with open(
            FAILURE_PATH,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                TRAINING_FAILURES,
                f,
                indent=2
            )

        clear_runtime_memory()

        raise

# --------------------------------------------------------------------------------------------------
# Persist metadata
# --------------------------------------------------------------------------------------------------

TRAINING_METADATA_DF = pd.DataFrame(
    TRAINING_METADATA_RECORDS
)

TRAINING_METADATA_PATH = (
    NB12_DIRS["metadata"] /
    "sppgan_training_metadata.csv"
)

TRAINING_METADATA_DF.to_csv(
    TRAINING_METADATA_PATH,
    index=False
)

# --------------------------------------------------------------------------------------------------
# Persist final results
# --------------------------------------------------------------------------------------------------

FINAL_RESULTS_DF = pd.DataFrame(
    FINAL_RESULTS
)

FINAL_RESULTS_PATH = (
    NB12_DIRS["metadata"] /
    "sppgan_training_results.csv"
)

FINAL_RESULTS_DF.to_csv(
    FINAL_RESULTS_PATH,
    index=False
)

# --------------------------------------------------------------------------------------------------
# Persist failures
# --------------------------------------------------------------------------------------------------

with open(
    FAILURE_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        TRAINING_FAILURES,
        f,
        indent=2
    )

print(
    "\n✓ Training execution completed."
)

print(
    f"✓ Results saved : {FINAL_RESULTS_PATH}"
)

print(
    f"✓ Metadata saved: {TRAINING_METADATA_PATH}"
)

print(
    f"✓ Failures saved: {FAILURE_PATH}"
)

28. FINAL VERIFICATION

TRAINING DATASET : adult_income

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
TRAINING FAILURE : adult_income
Exception : KeyError
Message   : 'adult_income'
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


KeyError: 'adult_income'

In [35]:
# ==================================================================================================
# 29. COMPLETION SUMMARY
# ==================================================================================================

print("=" * 100)
print("29. COMPLETION SUMMARY")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Final verification
# --------------------------------------------------------------------------------------------------

if FINAL_RESULTS_DF.empty:

    raise RuntimeError(
        "Final training results are empty."
    )

successful_results = FINAL_RESULTS_DF[
    FINAL_RESULTS_DF[
        "status"
    ].eq(
        "PASS"
    )
]

failed_results = FINAL_RESULTS_DF[
    FINAL_RESULTS_DF[
        "status"
    ].eq(
        "FAIL"
    )
]

if len(
    successful_results
) != len(
    DATASET_IDS
):

    raise RuntimeError(
        "Not all canonical datasets completed successfully."
    )

if len(
    failed_results
) != 0:

    raise RuntimeError(
        "One or more SPP-GAN training experiments failed."
    )

# --------------------------------------------------------------------------------------------------
# Achieved epsilon validation
# --------------------------------------------------------------------------------------------------

if not np.isfinite(
    successful_results[
        "achieved_epsilon"
    ].to_numpy(
        dtype=float
    )
).all():

    raise RuntimeError(
        "Achieved epsilon contains non-finite values."
    )

if not (
    successful_results[
        "achieved_epsilon"
    ]
    > 0
).all():

    raise RuntimeError(
        "Achieved epsilon must be positive."
    )

# --------------------------------------------------------------------------------------------------
# Metadata validation
# --------------------------------------------------------------------------------------------------

if not TRAINING_METADATA_PATH.exists():
    raise RuntimeError(
        "Training metadata was not persisted."
    )

if not FINAL_RESULTS_PATH.exists():
    raise RuntimeError(
        "Training results were not persisted."
    )

if not FAILURE_PATH.exists():
    raise RuntimeError(
        "Failure registry was not persisted."
    )

# --------------------------------------------------------------------------------------------------
# Build artifact registry
# --------------------------------------------------------------------------------------------------

ARTIFACT_REGISTRY = []

for dataset_id in DATASET_IDS:

    model_path = FINAL_MODELS[
        dataset_id
    ]

    history_path = HISTORY_PATHS[
        dataset_id
    ]

    ARTIFACT_REGISTRY.extend(
        [

            {
                "dataset":
                    dataset_id,

                "artifact_type":
                    "final_model",

                "path":
                    str(model_path),

                "sha256":
                    sha256_file(
                        model_path
                    ),

                "status":
                    "PASS",
            },

            {
                "dataset":
                    dataset_id,

                "artifact_type":
                    "training_history",

                "path":
                    str(history_path),

                "sha256":
                    sha256_file(
                        history_path
                    ),

                "status":
                    "PASS",
            },
        ]
    )

ARTIFACT_REGISTRY.extend(
    [

        {
            "dataset":
                "ALL",

            "artifact_type":
                "training_metadata",

            "path":
                str(TRAINING_METADATA_PATH),

            "sha256":
                sha256_file(
                    TRAINING_METADATA_PATH
                ),

            "status":
                "PASS",
        },

        {
            "dataset":
                "ALL",

            "artifact_type":
                "training_results",

            "path":
                str(FINAL_RESULTS_PATH),

            "sha256":
                sha256_file(
                    FINAL_RESULTS_PATH
                ),

            "status":
                "PASS",
        },

        {
            "dataset":
                "ALL",

            "artifact_type":
                "training_failures",

            "path":
                str(FAILURE_PATH),

            "sha256":
                sha256_file(
                    FAILURE_PATH
                ),

            "status":
                "PASS",
        },

        {
            "dataset":
                "ALL",

            "artifact_type":
                "training_configuration",

            "path":
                str(TRAINING_CONFIG_PATH),

            "sha256":
                sha256_file(
                    TRAINING_CONFIG_PATH
                ),

            "status":
                "PASS",
        },
    ]
)

ARTIFACT_REGISTRY_DF = pd.DataFrame(
    ARTIFACT_REGISTRY
)

ARTIFACT_REGISTRY_PATH = (
    NB12_DIRS["metadata"] /
    "sppgan_training_artifact_registry.csv"
)

ARTIFACT_REGISTRY_DF.to_csv(
    ARTIFACT_REGISTRY_PATH,
    index=False
)

# --------------------------------------------------------------------------------------------------
# Final manifest
# --------------------------------------------------------------------------------------------------

FINAL_ACHIEVED_EPSILON = {
    row["dataset"]:
        float(
            row["achieved_epsilon"]
        )
    for _, row in
    successful_results.iterrows()
}

FINAL_NOISE_MULTIPLIERS = {
    dataset_id:
        float(
            NOISE_MULTIPLIERS[
                dataset_id
            ]
        )
    for dataset_id in DATASET_IDS
}

TRAINING_MANIFEST = {

    "notebook":
        "12",

    "name":
        "SPP-GAN Training",

    "framework":
        "SPP-GAN",

    "status":
        "PASS",

    "project_root":
        str(PROJECT_ROOT),

    "datasets":
        list(DATASET_IDS),

    "repetition_id":
        REPETITION_ID,

    "experiment_seed":
        EXPERIMENT_SEED,

    "training":

        {
            "epochs":
                DP_EPOCHS,

            "generator_learning_rate":
                GENERATOR_LR,

            "critic_learning_rate":
                CRITIC_LR,

            "weight_decay":
                WEIGHT_DECAY,

            "latent_dimension":
                LATENT_DIM,

            "lambda_stat":
                LAMBDA_STAT,
        },

    "privacy":

        {
            "mechanism":
                "DP-SGD",

            "protected_component":
                "SPP-GAN discriminator",

            "accountant":
                ACCOUNTANT,

            "sampling":
                SAMPLING_MECHANISM,

            "clipping":
                CLIPPING_MECHANISM,

            "loss_reduction":
                LOSS_REDUCTION,

            "target_epsilon":
                TARGET_EPSILON,

            "achieved_epsilon":
                FINAL_ACHIEVED_EPSILON,

            "noise_multiplier":
                FINAL_NOISE_MULTIPLIERS,

            "max_grad_norm":
                MAX_GRAD_NORM,

            "generator_private":
                False,

            "statistical_guidance_private":
                False,

            "preprocessing_private":
                False,

            "end_to_end_privacy_claim":
                False,
        },

    "data_policy":

        {
            "training":
                "Notebook 02 TRAIN only",

            "validation":
                "monitoring only",

            "test":
                "isolated",

            "preprocessing_fit":
                "train_only",

            "statistical_reference":
                "Notebook 03 TRAIN-only statistical guidance",
        },

    "results":

        {
            "successful_datasets":
                len(successful_results),

            "failed_datasets":
                len(failed_results),

            "actual_epsilon_recorded":
                True,

            "synthetic_generation_performed":
                False,
        },

    "upstream":

        {
            "notebook_02":
                str(NB02_ROOT),

            "notebook_03":
                str(NB03_ROOT),

            "notebook_08":
                str(NB08_ROOT),

            "notebook_09":
                str(NB09_ROOT),

            "notebook_10":
                str(NB10_ROOT),

            "notebook_11":
                str(NB11_ROOT),
        },

    "downstream":

        {
            "notebook_13":
                "SPP-GAN Synthetic Generation",
        },

    "artifacts":

        {
            "training_configuration":
                str(TRAINING_CONFIG_PATH),

            "training_results":
                str(FINAL_RESULTS_PATH),

            "training_metadata":
                str(TRAINING_METADATA_PATH),

            "artifact_registry":
                str(ARTIFACT_REGISTRY_PATH),

            "training_failures":
                str(FAILURE_PATH),

            "models":
                {
                    dataset_id:
                        str(
                            FINAL_MODELS[
                                dataset_id
                            ]
                        )
                    for dataset_id in DATASET_IDS
                },

            "histories":
                {
                    dataset_id:
                        str(
                            HISTORY_PATHS[
                                dataset_id
                            ]
                        )
                    for dataset_id in DATASET_IDS
                },
        },

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

with open(
    TRAINING_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2
    )

# --------------------------------------------------------------------------------------------------
# Reload manifest
# --------------------------------------------------------------------------------------------------

with open(
    TRAINING_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    RELOADED_MANIFEST = json.load(
        f
    )

if RELOADED_MANIFEST[
    "status"
] != "PASS":

    raise RuntimeError(
        "Persisted training manifest is not PASS."
    )

if not RELOADED_MANIFEST[
    "privacy"
][
    "actual_epsilon_recorded"
    if "actual_epsilon_recorded"
    in RELOADED_MANIFEST["privacy"]
    else "target_epsilon"
]:

    pass

# --------------------------------------------------------------------------------------------------
# Final display
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 12 — FINAL STATUS")
print("=" * 100)

print(
    f"Framework                       : SPP-GAN"
)

print(
    f"Datasets                        : {len(DATASET_IDS)}"
)

print(
    f"Repetition                      : {REPETITION_ID}"
)

print(
    f"Experiment seed                 : {EXPERIMENT_SEED}"
)

print(
    f"Device                          : {DEVICE}"
)

print(
    f"Epochs                          : {DP_EPOCHS}"
)

print(
    f"Generator learning rate        : {GENERATOR_LR}"
)

print(
    f"Critic learning rate            : {CRITIC_LR}"
)

print(
    f"Lambda statistical              : {LAMBDA_STAT}"
)

print(
    f"Privacy mechanism               : DP-SGD"
)

print(
    f"Protected component             : SPP-GAN discriminator"
)

print(
    f"Sampling                        : Poisson"
)

print(
    f"Clipping                        : flat L2"
)

print(
    f"Accountant                      : RDP"
)

print(
    f"Target epsilon                  : {TARGET_EPSILON:.6f}"
)

print(
    "\nAchieved training epsilon:"
)

for dataset_id in DATASET_IDS:

    print(
        f"  {dataset_id:<20} : "
        f"{FINAL_ACHIEVED_EPSILON[dataset_id]:.6f}"
    )

print(
    "\nTraining                         : COMPLETED"
)

print(
    "Actual accountant epsilon        : RECORDED"
)

print(
    "Statistical guidance             : ENABLED"
)

print(
    "Generator private                : FALSE"
)

print(
    "Statistical guidance private     : FALSE"
)

print(
    "Preprocessing private            : FALSE"
)

print(
    "End-to-end privacy claim         : NOT ESTABLISHED"
)

print(
    "Synthetic generation             : NOT PERFORMED"
)

print(
    f"Successful datasets              : "
    f"{len(successful_results)}/{len(DATASET_IDS)}"
)

print(
    f"Failed datasets                  : "
    f"{len(failed_results)}"
)

print(
    "\nArtifacts:"
)

print(
    f"  Models                         : "
    f"{NB12_DIRS['models']}"
)

print(
    f"  Histories                      : "
    f"{NB12_DIRS['history']}"
)

print(
    f"  Metadata                       : "
    f"{TRAINING_METADATA_PATH}"
)

print(
    f"  Results                        : "
    f"{FINAL_RESULTS_PATH}"
)

print(
    f"  Registry                       : "
    f"{ARTIFACT_REGISTRY_PATH}"
)

print(
    f"  Manifest                       : "
    f"{TRAINING_MANIFEST_PATH}"
)

print(
    "\nNext:"
)

print(
    "  Notebook 13 — SPP-GAN Synthetic Generation"
)

print("=" * 100)

print(
    "\n✓ NOTEBOOK 12 TRAINING COMPLETED SUCCESSFULLY."
)

29. COMPLETION SUMMARY


NameError: name 'FINAL_RESULTS_DF' is not defined